# E3: toda la secuencia en una máquina

Selecciona GPU y ejecuta todo. No cambies de notebook. Guarda checkpoints en Drive y reanuda al repetir. Entradas: paquete_colab_e2_v2.zip, compactas_a0498_e3.zip y la carpeta histórica del benchmark en MyDrive. No evalúa test. Si ya estás entrenando en otra celda, espera que termine antes de iniciar.


In [ ]:
import os
os.environ['E3_CARRIL'] = 'todo'
# Una máquina: todo. Dos: original_y_compactas y rivales. Al acabar ambas: cierre.
# Sin entrenar el original: sin_original (rivales y recepción, sin test).
# También: original, compactas, rivales_a, rivales_b, rivales_c


In [ ]:
"""Una sesión de Colab, etapas reanudables. Cargado por el lanzador autocontenido."""
import hashlib
import json
import os
import time
import traceback
import ast
import contextlib
import subprocess
import sys
import threading
import uuid
import zipfile
from pathlib import Path

LANES = {'todo':['01','02','03','08','05','06','07','04','09','10'],
         'original':['01','02','03','04'], 'compactas':['05'],
         'rivales_a':['06'], 'rivales_b':['07'], 'rivales_c':['08'], 'cierre':['09','10'],
         'original_y_compactas':['01','02','03','04','05'], 'rivales':['08','06','07'],
         'sin_original':['08','05','06','07','09']}
ROOT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')


def source_digest(project):
    """La misma selección y serialización de source_signature() en Colab (rutas POSIX)."""
    paths=list((project/'scripts').glob('*.py'))+list((project/'third_party/e2').rglob('*.py'))
    paths += [project/'third_party/e2/sources.lock.json',project/'requirements-e2.txt']
    values={p.relative_to(project).as_posix():sha(p) for p in sorted(paths)}
    return hashlib.sha256(json.dumps(values,sort_keys=True).encode()).hexdigest()


def supplemental_sources(notebooks):
    """Lee los bytes textuales congelados; nunca ejecuta el entrenamiento de la etapa 05."""
    nb=next(b for n,b in notebooks.items() if n.startswith('05_'))
    for cell in nb['cells']:
        if cell['cell_type']!='code':continue
        for node in ast.parse(''.join(cell['source'])).body:
            if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='FUENTES_E2' for t in node.targets):
                return ast.literal_eval(node.value)
    raise RuntimeError('Falta FUENTES_E2 histórico; no reconstruir desde archivos actuales')


def prepare_sources(notebooks, scope, stage):
    """Restaura el perfil de fuentes histórico antes de preflight, con 05 completa o no."""
    project=Path(scope['PROJECT']);output=Path(scope['OUTPUT']);extra=supplemental_sources(notebooks)
    expected=set()
    candidates=list(output.glob('*/config.json'))
    preflight=output/'preflight.json'
    if preflight.exists():candidates.append(preflight)
    for path in candidates:
        record=json.loads(path.read_text())
        value=record.get('identity',{}).get('source')
        if value:expected.add(value)
    if len(expected)>1:raise RuntimeError('La etapa contiene varias firmas históricas; no se modifica ni reanuda')
    # Validar las dos variantes a partir del ZIP íntegro. Solo se restauran fuentes locales.
    archive=Path(scope['ZIP']);base=Path(scope['BASE'])
    with zipfile.ZipFile(archive) as z:
        names=set(z.namelist())
        signed=list((project/'scripts').glob('*.py'))+list((project/'third_party/e2').rglob('*.py'))
        signed += [project/'third_party/e2/sources.lock.json',project/'requirements-e2.txt']
        for path in signed:
            rel=path.relative_to(project).as_posix()
            if rel in extra:continue
            member=path.relative_to(base).as_posix()
            if member not in names or path.read_bytes()!=z.read(member):
                raise RuntimeError('Fuente ajena o modificada respecto al ZIP histórico: '+rel)
        for rel in extra:
            target=(project/rel).resolve()
            if not target.is_relative_to(project.resolve()):raise ValueError('Fuente fuera de PROJECT')
            member=(project/rel).relative_to(base).as_posix()
            if member in names:
                target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(member))
            elif target.exists():
                # Nunca borrar un archivo desconocido: solo el efecto exacto de FUENTES_E2.
                if target.read_bytes()!=extra[rel].encode('utf-8'):
                    raise RuntimeError('Fuente local desconocida: '+rel)
                target.unlink()
        base_signature=source_digest(project)
        for rel,text in extra.items():
            target=project/rel;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(text.encode('utf-8'))
        augmented_signature=source_digest(project)
        wanted=next(iter(expected),augmented_signature if stage in ['05','06','07','10'] else base_signature)
        if wanted==base_signature:
            for rel in extra:
                target=project/rel;member=target.relative_to(base).as_posix()
                if member in names:target.write_bytes(z.read(member))
                elif target.exists():target.unlink()
        elif wanted!=augmented_signature:
            raise RuntimeError(f'No se pudo recuperar la firma histórica {wanted}; base={base_signature}, ampliada={augmented_signature}. No se altera ningún checkpoint.')
    actual=source_digest(project)
    if actual!=wanted:raise RuntimeError('Firma de fuentes distinta después de reconstruir')
    print(f'Fuentes preparadas etapa {stage}: {actual} (histórica={bool(expected)})',flush=True)
    return {'source':actual,'base':base_signature,'ampliada':augmented_signature,'historica':bool(expected)}


class Tee:
    def __init__(self, stream, logfile):
        self.stream=stream;self.logfile=logfile;self.lock=threading.Lock()
    def write(self,text):
        with self.lock:
            self.stream.write(text);self.stream.flush();self.logfile.write(text);self.logfile.flush()
        return len(text)
    def flush(self):
        self.stream.flush();self.logfile.flush()


def stream_run(args, *, check=False, capture_output=False, text=False, env=None, timeout=None, **kwargs):
    """Muestra ambas tuberías mientras corren; conserva la semántica de captura y errores."""
    if 'stdout' in kwargs or 'stderr' in kwargs or 'input' in kwargs:
        raise ValueError('Redirección no soportada; usar capture_output para conservar la salida')
    process=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                             env=dict(os.environ if env is None else env,PYTHONUNBUFFERED='1'),**kwargs)
    buffers=[bytearray(),bytearray()]
    def pump(pipe,index,display):
        import codecs
        decoder=codecs.getincrementaldecoder('utf-8')(errors='replace')
        while True:
            chunk=pipe.read1(4096)
            if not chunk:break
            if capture_output:buffers[index].extend(chunk)
            display.write(decoder.decode(chunk));display.flush()
        display.write(decoder.decode(b'',final=True));display.flush();pipe.close()
    threads=[threading.Thread(target=pump,args=(pipe,i,display),daemon=True)
             for i,(pipe,display) in enumerate([(process.stdout,sys.stdout),(process.stderr,sys.stderr)])]
    for thread in threads:thread.start()
    try:process.wait(timeout=timeout)
    except BaseException:
        process.kill();process.wait()
        raise
    finally:
        for thread in threads:thread.join()
    values=[bytes(b).decode('utf-8',errors='replace') if text else bytes(b) for b in buffers]
    result=subprocess.CompletedProcess(args,process.returncode,*(values if capture_output else [None,None]))
    if check:result.check_returncode()
    return result


def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()


def targets(stage):
    if stage in ['01','02','03']:
        seeds={'01':[11],'02':[23,37,53,71],'03':[89,97,101,103,107]}[stage]
        return [(f'unetat_{a}',s,ROOT/'original'/f'unetat_{a}_seed{s}')
                for a in ['dbnet','dbnet_at'] for s in seeds]
    if stage=='04':
        return [('unetat_dbnet_at',11,ROOT/'hpo'/f'hpo_original_t{i}'/'unetat_dbnet_at_seed11') for i in range(4)]
    if stage=='05':
        return [(m,s,ROOT/'compactas/fase2'/f'{m}_a0498_s{s}')
                for m in ['fcn','unet','segnet'] for s in [89,97,101,103,107]]
    if stage in ['06','07','08']:
        models={'06':['transunet_2d','pidnet_s'],'07':['vmunet','segnext_t','deeplabv3plus','unetplusplus'],
                '08':['segformer_b0']}[stage]
        seeds=[89,97,101,103,107] if stage!='08' else [11,23,37,53,71,89,97,101,103,107]
        session={'06':'A','07':'B','08':'C'}[stage]
        return [(m,s,ROOT/f'rivales/sesion_{session}'/f'{m}_seed{s}') for m in models for s in seeds]
    return []


def completed(stage):
    expected=targets(stage)
    if not expected:return False
    for model,seed,folder in expected:
        if not (folder/'result.json').is_file():return False
        r=json.loads((folder/'result.json').read_text())
        c=json.loads((folder/'config.json').read_text())
        assert r['completed'] and r.get('seed',r.get('semilla'))==seed
        assert r.get('model',r.get('familia'))==model and r['fingerprint']==c['fingerprint']
        best=r['best_file']; assert Path(best).name==best
        assert sha(folder/best)==r['best_sha256'], f'Checkpoint alterado: {folder}'
        if model.startswith('unetat_'):assert c['identity']['opencv']=='4.11.0'
        if stage=='05':assert c['tarea']['arq']=='a0498' and c['tarea']['w']==[64,64,64]
        if stage=='08':assert abs(c['config']['models'][model]['lr']-.0005573638936676344)<1e-14
    return True


def transformed(source, stage):
    # Entornos separados en la misma VM: instalar Mamba no cambia la identidad del original.
    if stage in ['06','07','08','10']:
        source=source.replace("ENV=Path('/content/imo-e2-env')", "ENV=Path('/content/imo-e3-rivales-env')")
        # Fuera de las fuentes que participan en los fingerprints; antes de importar Mamba.
        source=source.replace("pip_install('torch==2.5.1'", "pip_install('setuptools==84.2.1')\npip_install('torch==2.5.1'")
    source='\n'.join(line for line in source.splitlines() if not line.strip().startswith('files.download('))+'\n'
    class Calls(ast.NodeTransformer):
        def visit_Call(self,node):
            self.generic_visit(node)
            if isinstance(node.func,ast.Attribute) and isinstance(node.func.value,ast.Name) and node.func.value.id=='subprocess' and node.func.attr=='run':
                if stage in ['06','07','08','10'] and node.args and isinstance(node.args[0],ast.List):
                    for argument in node.args[0].elts:
                        if isinstance(argument,ast.Constant) and isinstance(argument.value,str) and argument.value.startswith('import torch, torchvision, mamba_ssm;'):
                            argument.value="import importlib.metadata; assert importlib.metadata.version('setuptools')=='84.2.1'; "+argument.value
                node.func=ast.copy_location(ast.Name(id='_e3_run',ctx=ast.Load()),node.func)
            return node
    # Transformar llamadas, no cadenas CONTENIDO/FUENTES_E2 que fijan hashes históricos.
    return ast.unparse(ast.fix_missing_locations(Calls().visit(ast.parse(source))))+'\n'


def run(notebooks, lane='todo'):
    if lane not in LANES:raise ValueError(f'Carril desconocido: {lane}; opciones {list(LANES)}')
    from google.colab import drive, files
    drive.mount('/content/drive')
    ROOT.mkdir(parents=True,exist_ok=True)
    logdir=ROOT/'logs'/f"{time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())}_{lane}_{uuid.uuid4().hex[:8]}"
    logdir.mkdir(parents=True,exist_ok=True)
    status_path=ROOT/f'progreso_{lane}.json'
    status={'carril':lane,'etapas':{},'nota':'Solo entrenamiento y validación; no test ni despliegue'}

    def save():
        temp=status_path.with_suffix('.tmp')
        temp.write_text(json.dumps(status,indent=2,ensure_ascii=False),encoding='utf-8')
        os.replace(temp,status_path)

    for stage in LANES[lane]:
        stamp={'inicio_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'estado':'verificando'}
        status['etapas'][stage]=stamp; save()
        try:
            if completed(stage):
                print(f'Etapa {stage}: ya completa, checkpoints verificados. Se conserva.',flush=True)
                stamp['estado']='completa_previa_verificada';save();continue
            name,nb=next((n,b) for n,b in notebooks.items() if n.startswith(stage+'_'))
            print(f'\nEJECUTANDO {name} — misma máquina, checkpoints en Drive',flush=True)
            stamp['estado']='ejecutando';save()
            scope={'__name__':'__main__','_e3_run':stream_run}
            started=time.perf_counter()
            prepared=False
            stamp['logs']={'stdout':str(logdir/f'{stage}.stdout.log'),'stderr':str(logdir/f'{stage}.stderr.log')};save()
            with (logdir/f'{stage}.stdout.log').open('a',encoding='utf-8') as out, (logdir/f'{stage}.stderr.log').open('a',encoding='utf-8') as err:
                with contextlib.redirect_stdout(Tee(sys.stdout,out)),contextlib.redirect_stderr(Tee(sys.stderr,err)):
                    try:
                        for index,cell in enumerate(nb['cells']):
                            if cell['cell_type']!='code':continue
                            code=transformed(''.join(cell['source']),stage)
                            exec(compile(code,f'{name}:celda{index}','exec'),scope)
                            if not prepared and 'PROJECT' in scope:
                                stamp['fuentes']=prepare_sources(notebooks,scope,stage);prepared=True;save()
                    except BaseException:
                        traceback.print_exc();raise
            if targets(stage):assert completed(stage), f'Etapa {stage} no dejó todos los resultados'
            stamp.update(estado='completa',segundos=time.perf_counter()-started);save()
        except BaseException:
            stamp.update(estado='interrumpida_o_fallida',error=traceback.format_exc()[-5000:]);save()
            print(f'Se detuvo en {stage}. Los checkpoints escritos permanecen en Drive. '
                  'Corrige el error o reconecta y ejecuta el mismo lanzador; no borres resultados.',flush=True)
            raise
    print('Carril terminado. Progreso:',status_path,flush=True)
    if lane in ['todo','cierre','sin_original']:
        import zipfile
        archive=ROOT/'resultados_para_revision.zip'
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            for p in ROOT.rglob('*'):
                if p.is_file() and p.suffix in ['.json','.csv','.txt']:
                    z.write(p,str(p.relative_to(ROOT)))
        print('Resultados también guardados en Drive:',archive,flush=True)
        files.download(str(archive))


NOTEBOOKS = {'01_Original_semilla11.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 01 · Sanidad y semilla 11 del original\n', '\n', 'Reinicia ambos brazos con OpenCV 4.11. Primero exige que pase la batería de pruebas. Una prueba de equivalencia con el E2 histórico se omite porque falta su fuente; queda declarada.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ["env = dict(os.environ, OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')\n", "subprocess.run([PY, '-m', 'pytest', str(E3 / 'tests/test_original_model.py'), '-q', '-rs'], env=env, check=True)\n", 'entrenar([11])\n', 'verificar_original([11])\n', "print('Etapa 01 completa: semilla 11 verificada para ambos brazos')\n"], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '02_Original_semillas.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 02 · Ampliar original\n', '\n', 'Continúa con semillas [23, 37, 53, 71]. Mantiene la receta fijada, sin ajustarla por resultados.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['verificar_original([11])\n', 'entrenar([23, 37, 53, 71])\n', 'verificar_original([11, 23, 37, 53, 71])\n'], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '03_Original_semillas.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 03 · Ampliar original\n', '\n', 'Continúa con semillas [89, 97, 101, 103, 107]. Mantiene la receta fijada, sin ajustarla por resultados.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/original')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['verificar_original([11, 23, 37, 53, 71])\n', 'entrenar([89, 97, 101, 103, 107])\n', 'verificar_original([11, 23, 37, 53, 71, 89, 97, 101, 103, 107])\n'], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '04_HPO_original.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 04 · Búsqueda exploratoria original\n', '\n', 'Cuatro recetas, diez épocas como máximo y semilla 11. Exploratoria: no sustituye los resultados de la receta principal.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/hpo`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/hpo')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['verificar_original([11,23,37,53,71,89,97,101,103,107])\n', "for cfg in sorted((E3 / 'config').glob('hpo_original_t*.json')):\n", "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT / cfg.stem),\n", "        '--variante', 'dbnet_at', '--seeds', '11', '--config', str(cfg),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '05_Compactas_a0498.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 05 · Cinco semillas nuevas de compactas a0498\n', '\n', 'FCN, U-Net y SegNet: 89,97,101,103,107. Usa las recetas de los archivos originales recuperados; no repite las primeras cinco semillas.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/compactas`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/compactas')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['FUENTES_E2 = {\'scripts/busqueda_arquitectura_e2.py\': \'"""Búsqueda de arquitectura en dos fases para las tres redes compactas (sección 14 de la guía).\\n\\nLas siete redes nuevas del benchmark tienen arquitectura publicada; FCN, U-Net y SegNet compactas\\nson diseño propio y por eso recorren el espacio obligatorio: 3 bloques, anchos {16, 32, 64, 128,\\n256} en todas las combinaciones y dropout 0,2 en todas las posiciones = 1000 por familia.\\nProtocolo: config/busqueda_arquitectura_e2.json y docs/PROTOCOLO_BUSQUEDA_ARQUITECTURA_E2.md.\\n\\nBloques:\\n  listar      escribe config/arquitecturas_compactas_e2.csv (id, w, b, parámetros y MACs por familia)\\n  piloto      una época de 5 anchos x 3 familias a 256 y a 128 px, con la caché llena, para medir tiempo\\n  estimar     horas de GPU de la búsqueda a partir del piloto (marcadas como estimadas)\\n  fidelidad   fija la fidelidad de la fase 1 con la regla del protocolo y el presupuesto de horas\\n  ajuste      ajuste previo de la tasa con la referencia (64,64,64)/(0,1,0), por familia\\n  fase1       las 1000 arquitecturas x 3 familias, semilla 97, 6 épocas; y la repetición de ruido\\n  finalistas  ruido por semilla, k y lista de finalistas (solo validación)\\n  fase2       finalistas + referencia + heredada, 5 semillas, presupuesto completo\\n\\nSolo train y validación: el test no se carga. Reutiliza de complemento_entreno_e2 la caché en RAM,\\nla identidad de datos y fuentes y el orden de lotes por semilla del benchmark.\\n"""\\nimport argparse\\nimport itertools\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nimport complemento_entreno_e2 as C\\nfrom dataset_ready import metrics\\nfrom modelos_base import block\\n\\nE = C.E\\nPROJECT = E.PROJECT\\nCONFIG = PROJECT/\\\'config/busqueda_arquitectura_e2.json\\\'\\nLISTADO = PROJECT/\\\'config/arquitecturas_compactas_e2.csv\\\'\\nFAMILIAS = (\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\')\\n\\n\\n# ---------------------------------------------------------------- espacio\\ndef espacio(cfg):\\n    """Las 1000 arquitecturas en el orden del producto cartesiano, con su identificador."""\\n    W = cfg[\\\'espacio\\\'][\\\'anchos\\\']\\n    out = []\\n    for k, (w1, w2, w3, b1, b2, b3) in enumerate(itertools.product(W, W, W, (0, 1), (0, 1), (0, 1))):\\n        out.append({\\\'arq\\\': f\\\'a{k:04d}\\\', \\\'w\\\': (w1, w2, w3), \\\'b\\\': (b1, b2, b3)})\\n    if len(out) != cfg[\\\'espacio\\\'][\\\'cardinalidad\\\']:\\n        raise RuntimeError(\\\'El espacio no tiene la cardinalidad declarada\\\')\\n    return out\\n\\n\\ndef buscar(arqs, w, b):\\n    for a in arqs:\\n        if a[\\\'w\\\'] == tuple(w) and a[\\\'b\\\'] == tuple(b):\\n            return a\\n    raise KeyError((w, b))\\n\\n\\ndef orden_fase1(cfg, arqs):\\n    perm = np.random.default_rng(cfg[\\\'fase1\\\'][\\\'semilla_orden\\\']).permutation(len(arqs))\\n    return [arqs[i] for i in perm]\\n\\n\\n# ---------------------------------------------------------------- familias parametrizadas\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\\n\\nclass EntradaReducida(nn.Module):\\n    """Fidelidad reducida de la fase 1: la red ve la imagen a px x px (promedio por área) y su salida\\n    vuelve a 256 por interpolación bilineal; pérdida y Dice se miden a 256 contra la máscara original."""\\n\\n    def __init__(self, red, px):\\n        super().__init__()\\n        self.red, self.px = red, px\\n\\n    def forward(self, x):\\n        z = self.red(F.interpolate(x, size=(self.px, self.px), mode=\\\'area\\\'))\\n        return F.interpolate(z, size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\n\\ndef subconjunto_train(train, fraccion, semilla):\\n    """Subconjunto fijo de train por grupos de similitud (no parte grupos), el mismo para todas."""\\n    if fraccion >= 1:\\n        return train\\n    grupos = sorted({r[\\\'group_id\\\'] for r in train})\\n    orden = np.random.default_rng(semilla).permutation(len(grupos))\\n    elegidos, n = set(), 0\\n    por_grupo = defaultdict(int)\\n    for r in train:\\n        por_grupo[r[\\\'group_id\\\']] += 1\\n    for i in orden:\\n        if n >= fraccion*len(train):\\n            break\\n        elegidos.add(grupos[i]); n += por_grupo[grupos[i]]\\n    return [r for r in train if r[\\\'group_id\\\'] in elegidos]\\n\\n\\ndef parametros(model):\\n    return sum(q.numel() for q in model.parameters())\\n\\n\\ndef macs(model, size=256, size_calculo=32):\\n    """Multiplicaciones-acumulaciones de las convoluciones por imagen (sin pooling ni interpolación).\\n\\n    Se calcula a 32 px y se escala por (size/32)^2: todas las capas son convoluciones con relleno\\n    que conservan el tamaño y el pooling divide exacto, así que el conteo es proporcional al área.\\n    """\\n    total = [0]\\n\\n    def gancho(mod, inp, out):\\n        total[0] += out.numel() // out.shape[0] * (mod.in_channels // mod.groups) * mod.kernel_size[0] * mod.kernel_size[1]\\n    hs = [m.register_forward_hook(gancho) for m in model.modules() if isinstance(m, nn.Conv2d)]\\n    was = model.training\\n    model.eval()\\n    with torch.no_grad():\\n        model(torch.zeros(1, 3, size_calculo, size_calculo))\\n    model.train(was)\\n    for h in hs:\\n        h.remove()\\n    return int(total[0] * (size // size_calculo)**2)\\n\\n\\ndef listar(cfg, destino=LISTADO):\\n    arqs = espacio(cfg); p = cfg[\\\'espacio\\\'][\\\'dropout_p\\\']\\n    lineas = [\\\'arq,w1,w2,w3,b1,b2,b3,\\\' + \\\',\\\'.join(f\\\'{f}_parametros,{f}_macs\\\' for f in FAMILIAS)]\\n    for a in arqs:\\n        celdas = [a[\\\'arq\\\'], *map(str, a[\\\'w\\\']), *map(str, a[\\\'b\\\'])]\\n        for f in FAMILIAS:\\n            m = construir_red(f, a[\\\'w\\\'], a[\\\'b\\\'], p)\\n            celdas += [str(parametros(m)), str(macs(m))]\\n        lineas.append(\\\',\\\'.join(celdas))\\n    Path(destino).write_text(\\\'\\\\n\\\'.join(lineas) + \\\'\\\\n\\\', encoding=\\\'utf-8\\\')\\n    print(f\\\'{len(arqs)} arquitecturas escritas en {destino} (sha256 {E.sha(destino)[:12]})\\\')\\n    return destino\\n\\n\\n# ---------------------------------------------------------------- identidad\\nDEPENDE_DE = [\\\'scripts/busqueda_arquitectura_e2.py\\\', \\\'scripts/complemento_entreno_e2.py\\\',\\n              \\\'scripts/experimento_e2_colab.py\\\', \\\'scripts/modelos_e2.py\\\', \\\'scripts/modelos_base.py\\\',\\n              \\\'scripts/dataset_ready.py\\\', \\\'config/busqueda_arquitectura_e2.json\\\',\\n              \\\'config/arquitecturas_compactas_e2.csv\\\', \\\'config/experimento_e2.json\\\', \\\'requirements-e2.txt\\\']\\n\\n\\ndef identidad(args, train, val):\\n    """Como C.identidad, pero la firma de fuentes cubre solo los archivos de los que depende la búsqueda:\\n    otro script nuevo en scripts/ no invalida las corridas hechas."""\\n    ident = C.identidad(args, train, val)\\n    ident[\\\'source\\\'] = E.digest({rel: E.sha(PROJECT/rel) for rel in DEPENDE_DE})\\n    return ident\\n\\n\\n# ---------------------------------------------------------------- evaluación\\n@torch.no_grad()\\ndef evaluar(model, rows, device, proto, con_detalle=False):\\n    """Dice por grupo (como E.evaluate_validation) y pérdida media por imagen, en modo evaluación."""\\n    model.eval(); grupos = defaultdict(list); detalle = []; perdidas = []\\n    devices = [torch.cuda.current_device()] if device.type == \\\'cuda\\\' else []\\n    with torch.random.fork_rng(devices=devices):\\n        torch.manual_seed(20261118)\\n        for row in rows:\\n            x, y, v = E.batch_tensors([row], device)\\n            z = model(x)\\n            perdidas.append(float(E.compute_loss(z, y, v, proto)))\\n            prob = z.sigmoid()[0, 0].cpu().numpy()\\n            s = metrics(prob, y[0, 0].cpu().numpy(), v[0, 0].cpu().numpy())\\n            grupos[row[\\\'group_id\\\']].append(s[\\\'Dice\\\'])\\n            if con_detalle:\\n                detalle.append({\\\'id\\\': row[\\\'id\\\'], \\\'group_id\\\': row[\\\'group_id\\\'], **s})\\n    return float(np.mean([np.mean(g) for g in grupos.values()])), float(np.mean(perdidas)), detalle\\n\\n\\n# ---------------------------------------------------------------- entrenamiento\\ndef entrenar(tarea, salida, proto, ident, device, train, val, detener_tras=None):\\n    """Entrena una arquitectura. Fase 1 y ajuste: sin pesos, se reanuda por tarea. Fase 2: last.pt por época."""\\n    dest = Path(salida)/tarea[\\\'bloque\\\']/tarea[\\\'id\\\']\\n    dest.mkdir(parents=True, exist_ok=True)\\n    train = subconjunto_train(train, tarea[\\\'fraccion_train\\\'], tarea[\\\'semilla_subconjunto\\\'])\\n    spec = {\\\'tarea\\\': tarea, \\\'protocolo\\\': {k: proto[k] for k in (\\\'batch_size\\\', \\\'size\\\', \\\'threshold\\\')},\\n            \\\'identidad\\\': ident, \\\'entorno\\\': C.entorno(), \\\'filas\\\': {\\\'train\\\': len(train), \\\'val\\\': len(val)}}\\n    huella = E.digest(spec)\\n    res_file = dest/\\\'result.json\\\'\\n    if res_file.exists():\\n        result = json.loads(res_file.read_text(encoding=\\\'utf-8\\\'))\\n        if result.get(\\\'fingerprint\\\') != huella or not result.get(\\\'completed\\\'):\\n            raise RuntimeError(f\\\'Resultado incompatible: {dest}. Usa una carpeta de salida nueva.\\\')\\n        return result\\n    cfg_file = dest/\\\'config.json\\\'\\n    if cfg_file.exists() and json.loads(cfg_file.read_text(encoding=\\\'utf-8\\\'))[\\\'fingerprint\\\'] != huella:\\n        raise RuntimeError(f\\\'Corrida incompatible: {dest}\\\')\\n    E.atomic_json(cfg_file, {\\\'fingerprint\\\': huella, **spec})\\n    E.setup_seed(tarea[\\\'semilla\\\'])\\n    model = construir_red(tarea[\\\'familia\\\'], tarea[\\\'w\\\'], tarea[\\\'b\\\'], tarea[\\\'dropout_p\\\'])\\n    n_param = parametros(model)\\n    if tarea[\\\'px\\\'] != proto[\\\'size\\\']:\\n        model = EntradaReducida(model, tarea[\\\'px\\\'])\\n    model = model.to(device)\\n    optimizer = E.make_optimizer(model, tarea[\\\'optim\\\'])\\n    guardar = tarea[\\\'conservar\\\'] == \\\'best\\\'\\n    last = dest/\\\'last.pt\\\'\\n    start, best, stale, history, best_state = 0, -1., 0, [], None\\n    if guardar and last.exists():\\n        saved = torch.load(last, map_location=\\\'cpu\\\', weights_only=False)  # solo archivos propios\\n        if saved[\\\'fingerprint\\\'] != huella:\\n            raise RuntimeError(\\\'Checkpoint incompatible\\\')\\n        model.load_state_dict(saved[\\\'model\\\']); optimizer.load_state_dict(saved[\\\'optimizer\\\'])\\n        start, best, stale, history = saved[\\\'epoch\\\'] + 1, saved[\\\'best\\\'], saved[\\\'stale\\\'], saved[\\\'history\\\']\\n        best_state = saved[\\\'best_state\\\']\\n        E.restore_rng(saved[\\\'rng\\\'])\\n    t0 = time.perf_counter()\\n    for epoch in range(start, tarea[\\\'epocas\\\']):\\n        if stale >= tarea[\\\'paciencia\\\']:\\n            break\\n        model.train(); started = time.perf_counter(); loss_sum = 0.; seen = 0\\n        for rows in E.training_batches(train, tarea[\\\'semilla\\\'], epoch, proto[\\\'batch_size\\\']):\\n            x, y, v = C.lote(rows, device)\\n            optimizer.zero_grad(set_to_none=True)\\n            loss = E.compute_loss(model(x), y, v, proto)\\n            if not torch.isfinite(loss):\\n                raise RuntimeError(f\\\'Pérdida no finita en {tarea["id"]}, época {epoch+1}\\\')\\n            loss.backward(); optimizer.step()\\n            loss_sum += loss.item()*len(rows); seen += len(rows)\\n        entreno_s = time.perf_counter() - started\\n        score, val_loss, detalle = evaluar(model, val, device, proto, con_detalle=guardar)\\n        if not np.isfinite(score):\\n            raise RuntimeError(\\\'Validación no finita\\\')\\n        if score > best:\\n            best, stale = score, 0\\n            if guardar:\\n                best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}\\n                E.atomic_json(dest/\\\'best_val_predictions.json\\\', {\\\'epoch\\\': epoch+1, \\\'validation\\\': detalle})\\n        else:\\n            stale += 1\\n        history.append({\\\'epoch\\\': epoch+1, \\\'train_loss\\\': loss_sum/seen, \\\'val_loss\\\': val_loss,\\n                        \\\'val_group_Dice\\\': score, \\\'seconds\\\': time.perf_counter()-started,\\n                        \\\'train_seconds\\\': entreno_s})\\n        if guardar:\\n            E.atomic_torch(last, {\\\'fingerprint\\\': huella, \\\'epoch\\\': epoch, \\\'model\\\': model.state_dict(),\\n                                  \\\'optimizer\\\': optimizer.state_dict(), \\\'best\\\': best, \\\'stale\\\': stale,\\n                                  \\\'history\\\': history, \\\'best_state\\\': best_state, \\\'rng\\\': E.rng_state()})\\n        print(f\\\'{tarea["id"]} epoch={epoch+1} loss={loss_sum/seen:.4f} val={score:.5f} \\\'\\n              f\\\'{history[-1]["seconds"]:.1f}s\\\', flush=True)\\n        if detener_tras is not None and epoch+1 >= detener_tras:\\n            return {\\\'completed\\\': False, \\\'id\\\': tarea[\\\'id\\\'], \\\'epochs_done\\\': epoch+1, \\\'history\\\': history}\\n    mejor = max(history, key=lambda h: h[\\\'val_group_Dice\\\'])\\n    result = {\\\'completed\\\': True, \\\'fingerprint\\\': huella, **{k: tarea[k] for k in\\n              (\\\'id\\\', \\\'bloque\\\', \\\'familia\\\', \\\'arq\\\', \\\'w\\\', \\\'b\\\', \\\'semilla\\\', \\\'optim\\\', \\\'epocas\\\', \\\'paciencia\\\',\\n               \\\'px\\\', \\\'fraccion_train\\\')},\\n              \\\'n_train\\\': len(train),\\n              \\\'best_val_group_Dice\\\': best, \\\'best_epoch\\\': mejor[\\\'epoch\\\'],\\n              \\\'val_loss_best_epoch\\\': mejor[\\\'val_loss\\\'], \\\'train_loss_best_epoch\\\': mejor[\\\'train_loss\\\'],\\n              \\\'epochs_completed\\\': len(history), \\\'history\\\': history,\\n              \\\'parameters\\\': n_param, \\\'macs\\\': tarea[\\\'macs\\\'], \\\'macs_efectivos\\\': tarea[\\\'macs_efectivos\\\'],\\n              \\\'seconds_this_session\\\': time.perf_counter()-t0,\\n              \\\'seconds_total\\\': sum(h[\\\'seconds\\\'] for h in history),\\n              \\\'device\\\': str(device), \\\'test_evaluated\\\': False}\\n    if guardar:\\n        model.load_state_dict(best_state)\\n        tr, tr_loss, _ = evaluar(model, train, device, proto)\\n        result.update({\\\'best_train_group_Dice\\\': tr, \\\'best_train_loss_eval\\\': tr_loss,\\n                       \\\'brecha_train_val\\\': tr - best, \\\'best_file\\\': \\\'best.pt\\\'})\\n        E.atomic_torch(dest/\\\'best.pt\\\', {\\\'model\\\': best_state, \\\'epoch\\\': mejor[\\\'epoch\\\'], \\\'fingerprint\\\': huella})\\n        result[\\\'best_sha256\\\'] = E.sha(dest/\\\'best.pt\\\')\\n    E.atomic_json(res_file, result)\\n    if last.exists():\\n        last.unlink()\\n    return result\\n\\n\\n# ---------------------------------------------------------------- tareas\\ndef receta(cfg, familia, lr=None):\\n    r = cfg[\\\'receta\\\']\\n    return {\\\'optimizer\\\': r[\\\'optimizer\\\'], \\\'lr\\\': float(lr if lr is not None else r[\\\'lr\\\']),\\n            \\\'weight_decay\\\': r[\\\'weight_decay\\\']}\\n\\n\\ndef tarea(cfg, bloque, familia, arq, semilla, lr, epocas, paciencia, conservar, args, macs_tabla,\\n          px=256, fraccion=1.0):\\n    sufijo = (f\\\'_px{px}\\\' if px != 256 else \\\'\\\') + (f\\\'_f{round(100*fraccion)}\\\' if fraccion < 1 else \\\'\\\')\\n    t = {\\\'id\\\': f\\\'{familia}_{arq["arq"]}_s{semilla}\\\' + (f\\\'_lr{lr:g}\\\' if bloque == \\\'ajuste\\\' else \\\'\\\') + sufijo,\\n         \\\'bloque\\\': bloque, \\\'familia\\\': familia, \\\'arq\\\': arq[\\\'arq\\\'], \\\'w\\\': list(arq[\\\'w\\\']), \\\'b\\\': list(arq[\\\'b\\\']),\\n         \\\'dropout_p\\\': cfg[\\\'espacio\\\'][\\\'dropout_p\\\'], \\\'semilla\\\': semilla, \\\'optim\\\': receta(cfg, familia, lr),\\n         \\\'epocas\\\': epocas, \\\'paciencia\\\': paciencia, \\\'conservar\\\': conservar,\\n         \\\'macs\\\': macs_tabla[(familia, arq[\\\'arq\\\'])],\\n         \\\'macs_efectivos\\\': int(macs_tabla[(familia, arq[\\\'arq\\\'])]*(px/256)**2),\\n         \\\'px\\\': px, \\\'fraccion_train\\\': fraccion, \\\'semilla_subconjunto\\\': cfg[\\\'fidelidad_fase1\\\'][\\\'semilla_subconjunto\\\']}\\n    if args.epocas_max:\\n        t[\\\'epocas\\\'] = min(t[\\\'epocas\\\'], args.epocas_max); t[\\\'prueba_local\\\'] = True\\n    if args.limite_train or args.limite_val:\\n        t[\\\'prueba_local\\\'] = True\\n    return t\\n\\n\\ndef tabla_macs(cfg, arqs, familias):\\n    """MACs por (familia, arq) desde el listado versionado; si falta, se calcula y se comprueba."""\\n    tabla = {}\\n    if LISTADO.exists():\\n        lineas = LISTADO.read_text(encoding=\\\'utf-8\\\').splitlines()\\n        cab = lineas[0].split(\\\',\\\')\\n        for linea in lineas[1:]:\\n            c = dict(zip(cab, linea.split(\\\',\\\')))\\n            for f in FAMILIAS:\\n                tabla[(f, c[\\\'arq\\\'])] = int(c[f\\\'{f}_macs\\\'])\\n    if len(tabla) != len(arqs)*len(FAMILIAS):\\n        p = cfg[\\\'espacio\\\'][\\\'dropout_p\\\']\\n        for a in arqs:\\n            for f in familias:\\n                tabla[(f, a[\\\'arq\\\'])] = macs(construir_red(f, a[\\\'w\\\'], a[\\\'b\\\'], p))\\n    return tabla\\n\\n\\ndef fidelidad(salida):\\n    archivo = Path(salida)/\\\'fidelidad.json\\\'\\n    if not archivo.exists():\\n        raise RuntimeError(\\\'Falta fidelidad.json: corre antes el bloque fidelidad\\\')\\n    f = json.loads(archivo.read_text(encoding=\\\'utf-8\\\'))[\\\'elegida\\\']\\n    return f[\\\'px\\\'], f[\\\'fraccion_train\\\']\\n\\n\\ndef lr_elegida(salida, familia):\\n    archivo = Path(salida)/\\\'ajuste_previo.json\\\'\\n    if not archivo.exists():\\n        raise RuntimeError(\\\'Falta ajuste_previo.json: corre antes el bloque ajuste\\\')\\n    return json.loads(archivo.read_text(encoding=\\\'utf-8\\\'))[\\\'lr_elegida\\\'][familia]\\n\\n\\ndef tareas_ajuste(cfg, arqs, familias, salida, args, mt):\\n    ref = buscar(arqs, cfg[\\\'referencia\\\'][\\\'w\\\'], cfg[\\\'referencia\\\'][\\\'b\\\']); f1 = cfg[\\\'fase1\\\']\\n    px, fr = fidelidad(salida)\\n    return [tarea(cfg, \\\'ajuste\\\', f, ref, f1[\\\'semilla\\\'], lr, f1[\\\'epocas\\\'], f1[\\\'paciencia\\\'], \\\'ninguno\\\', args, mt,\\n                  px, fr)\\n            for f in familias for lr in cfg[\\\'ajuste_previo\\\'][\\\'lr\\\']]\\n\\n\\ndef tareas_fase1(cfg, arqs, familias, salida, args, mt, n_arq=None):\\n    f1 = cfg[\\\'fase1\\\']; orden = orden_fase1(cfg, arqs)[:n_arq]\\n    n_ruido = cfg[\\\'ruido\\\'][\\\'n_arquitecturas\\\'] if n_arq is None else min(n_arq, 2)\\n    lrs = {f: lr_elegida(salida, f) for f in familias}\\n    px, fr = fidelidad(salida)\\n    out = []\\n    for i, a in enumerate(orden):\\n        for f in familias:\\n            out.append(tarea(cfg, \\\'fase1\\\', f, a, f1[\\\'semilla\\\'], lrs[f], f1[\\\'epocas\\\'], f1[\\\'paciencia\\\'],\\n                             \\\'ninguno\\\', args, mt, px, fr))\\n        if i < n_ruido:  # repetición para el ruido por semilla, intercalada al principio\\n            for f in familias:\\n                out.append(tarea(cfg, \\\'fase1\\\', f, a, cfg[\\\'ruido\\\'][\\\'semilla_extra\\\'], lrs[f], f1[\\\'epocas\\\'],\\n                                 f1[\\\'paciencia\\\'], \\\'ninguno\\\', args, mt, px, fr))\\n    return out\\n\\n\\ndef tareas_fase2(cfg, arqs, familias, salida, args, mt):\\n    fin = json.loads((Path(salida)/\\\'finalistas.json\\\').read_text(encoding=\\\'utf-8\\\'))\\n    f2 = cfg[\\\'fase2\\\']; por_id = {a[\\\'arq\\\']: a for a in arqs}; out = []\\n    for s in f2[\\\'semillas\\\']:\\n        for f in familias:\\n            for arq in fin[\\\'familias\\\'][f][\\\'fase2\\\']:\\n                out.append(tarea(cfg, \\\'fase2\\\', f, por_id[arq], s, lr_elegida(salida, f), f2[\\\'epocas\\\'],\\n                                 f2[\\\'paciencia\\\'], \\\'best\\\', args, mt))\\n    return out\\n\\n\\n# ---------------------------------------------------------------- decisiones entre bloques\\ndef resultados(salida, bloque):\\n    out = []\\n    for p in sorted((Path(salida)/bloque).glob(\\\'*/result.json\\\')):\\n        r = json.loads(p.read_text(encoding=\\\'utf-8\\\'))\\n        if r.get(\\\'completed\\\'):\\n            out.append(r)\\n    return out\\n\\n\\ndef cerrar_ajuste(cfg, salida, familias):\\n    """Regla de 14.3, fijada antes: mayor Dice; si gana a la receta por menos de 0,005, la receta."""\\n    res = resultados(salida, \\\'ajuste\\\'); ap = cfg[\\\'ajuste_previo\\\']; base = cfg[\\\'receta\\\'][\\\'lr\\\']\\n    elegida, tabla = {}, {}\\n    for f in familias:\\n        por_lr = {r[\\\'optim\\\'][\\\'lr\\\']: r[\\\'best_val_group_Dice\\\'] for r in res if r[\\\'familia\\\'] == f}\\n        if sorted(por_lr) != sorted(ap[\\\'lr\\\']):\\n            raise RuntimeError(f\\\'Ajuste previo incompleto en {f}: {sorted(por_lr)}\\\')\\n        mejor = max(por_lr, key=por_lr.get)\\n        elegida[f] = mejor if por_lr[mejor] - por_lr[base] >= 0.005 else base\\n        tabla[f] = {f\\\'{k:g}\\\': v for k, v in por_lr.items()}\\n    out = {\\\'regla\\\': ap[\\\'regla\\\'], \\\'dice_validacion\\\': tabla, \\\'lr_elegida\\\': elegida,\\n           \\\'arquitectura\\\': {\\\'w\\\': cfg[\\\'referencia\\\'][\\\'w\\\'], \\\'b\\\': cfg[\\\'referencia\\\'][\\\'b\\\']}}\\n    E.atomic_json(Path(salida)/\\\'ajuste_previo.json\\\', out)\\n    print(json.dumps(out, indent=2, ensure_ascii=False), flush=True)\\n    return out\\n\\n\\ndef cerrar_finalistas(cfg, salida, familias, arqs, permitir_incompleta=False):\\n    res = resultados(salida, \\\'fase1\\\'); f1 = cfg[\\\'fase1\\\']\\n    por_id = {a[\\\'arq\\\']: a for a in arqs}\\n    ref = buscar(arqs, cfg[\\\'referencia\\\'][\\\'w\\\'], cfg[\\\'referencia\\\'][\\\'b\\\'])[\\\'arq\\\']\\n    her = buscar(arqs, cfg[\\\'heredada\\\'][\\\'w\\\'], cfg[\\\'heredada\\\'][\\\'b\\\'])[\\\'arq\\\']\\n    info, rs = {}, {}\\n    for f in familias:\\n        principal = {r[\\\'arq\\\']: r for r in res if r[\\\'familia\\\'] == f and r[\\\'semilla\\\'] == f1[\\\'semilla\\\']}\\n        extra = {r[\\\'arq\\\']: r for r in res if r[\\\'familia\\\'] == f and r[\\\'semilla\\\'] == cfg[\\\'ruido\\\'][\\\'semilla_extra\\\']}\\n        if len(principal) < len(arqs) and not permitir_incompleta:\\n            raise RuntimeError(f\\\'Fase 1 incompleta en {f}: {len(principal)} de {len(arqs)}\\\')\\n        y = np.array([r[\\\'best_val_group_Dice\\\'] for r in principal.values()])\\n        pares = [(principal[a][\\\'best_val_group_Dice\\\'], extra[a][\\\'best_val_group_Dice\\\']) for a in extra if a in principal]\\n        d = np.array([p - q for p, q in pares])\\n        s_sem = float(np.sqrt(np.mean(d**2)/2)) if len(d) else float(\\\'nan\\\')\\n        s_tot = float(np.std(y, ddof=1))\\n        s_arq = float(np.sqrt(max(s_tot**2 - s_sem**2, 0.)))\\n        rs[f] = s_sem/s_arq if s_arq > 0 else float(\\\'inf\\\')\\n        ranking = sorted(principal.values(), key=lambda r: (-r[\\\'best_val_group_Dice\\\'], r[\\\'parameters\\\']))\\n        info[f] = {\\\'n_fase1\\\': len(principal), \\\'pares_ruido\\\': len(d), \\\'sigma_semilla\\\': s_sem,\\n                   \\\'sigma_total\\\': s_tot, \\\'sigma_arquitectura\\\': s_arq, \\\'r\\\': rs[f],\\n                   \\\'ranking\\\': [r[\\\'arq\\\'] for r in ranking],\\n                   \\\'rango_referencia\\\': [r[\\\'arq\\\'] for r in ranking].index(ref) + 1 if ref in principal else None,\\n                   \\\'rango_heredada\\\': [r[\\\'arq\\\'] for r in ranking].index(her) + 1 if her in principal else None}\\n    k = 10 if all(r <= 1 for r in rs.values()) else 20\\n    out = {\\\'regla_k\\\': cfg[\\\'ruido\\\'][\\\'regla_k\\\'], \\\'k\\\': k, \\\'referencia\\\': ref, \\\'heredada\\\': her,\\n           \\\'fase1_completa\\\': all(info[f][\\\'n_fase1\\\'] == len(arqs) for f in familias), \\\'familias\\\': {}}\\n    for f in familias:\\n        top = info[f][\\\'ranking\\\'][:k]\\n        fase2 = top + [a for a in (ref, her) if a not in top]\\n        out[\\\'familias\\\'][f] = {**{x: y for x, y in info[f].items() if x != \\\'ranking\\\'},\\n                              \\\'finalistas\\\': top, \\\'fase2\\\': fase2,\\n                              \\\'detalle\\\': [{\\\'arq\\\': a, \\\'w\\\': por_id[a][\\\'w\\\'], \\\'b\\\': por_id[a][\\\'b\\\']} for a in fase2]}\\n    E.atomic_json(Path(salida)/\\\'finalistas.json\\\', out)\\n    print(json.dumps({f: {\\\'k\\\': k, \\\'r\\\': rs[f], \\\'fase2\\\': out[\\\'familias\\\'][f][\\\'fase2\\\']} for f in familias},\\n                     indent=2), flush=True)\\n    return out\\n\\n\\n# ---------------------------------------------------------------- piloto, estimación y fidelidad\\nPILOTO = [((16, 16, 16), (0, 0, 0)), ((16, 32, 64), (0, 0, 0)), ((64, 64, 64), (0, 1, 0)),\\n          ((128, 128, 128), (0, 0, 0)), ((256, 256, 256), (1, 1, 1))]\\n\\n\\ndef tareas_piloto(cfg, arqs, args, mt):\\n    """Cinco anchos que cubren el rango de costo, en las tres familias, a 256 y a 128 px, una época."""\\n    out = []\\n    for px in sorted({o[\\\'px\\\'] for o in cfg[\\\'fidelidad_fase1\\\'][\\\'opciones\\\']}, reverse=True):\\n        for w, b in PILOTO:\\n            for f in FAMILIAS:\\n                out.append(tarea(cfg, \\\'piloto\\\', f, buscar(arqs, w, b), cfg[\\\'fase1\\\'][\\\'semilla\\\'], None, 1, 5,\\n                                 \\\'ninguno\\\', args, mt, px, 1.0))\\n    return out\\n\\n\\ndef estimar(cfg, salida, arqs, mt):\\n    """t_epoca = a + c * MACs efectivos por familia y resolución, con el piloto. Es una ESTIMACIÓN."""\\n    salida = Path(salida); filas = []\\n    for p in (salida/\\\'piloto\\\').glob(\\\'*/result.json\\\'):\\n        r = json.loads(p.read_text(encoding=\\\'utf-8\\\')); h = r[\\\'history\\\'][0]\\n        filas.append({\\\'familia\\\': r[\\\'familia\\\'], \\\'px\\\': r[\\\'px\\\'], \\\'macs\\\': r[\\\'macs_efectivos\\\'],\\n                      \\\'train_s\\\': h[\\\'train_seconds\\\'], \\\'val_s\\\': h[\\\'seconds\\\'] - h[\\\'train_seconds\\\']})\\n    modelo = {}\\n    for f in FAMILIAS:\\n        for px in sorted({x[\\\'px\\\'] for x in filas}):\\n            sel = [x for x in filas if x[\\\'familia\\\'] == f and x[\\\'px\\\'] == px]\\n            if len(sel) < 2:\\n                raise RuntimeError(f\\\'Piloto insuficiente en {f} a {px} px\\\')\\n            m = np.array([x[\\\'macs\\\'] for x in sel], float)\\n            ct, at = np.polyfit(m, [x[\\\'train_s\\\'] for x in sel], 1)\\n            cv, av = np.polyfit(m, [x[\\\'val_s\\\'] for x in sel], 1)\\n            modelo[f\\\'{f}@{px}\\\'] = {\\\'train_a\\\': float(at), \\\'train_c\\\': float(ct), \\\'val_a\\\': float(av), \\\'val_c\\\': float(cv),\\n                                   \\\'n\\\': len(sel)}\\n\\n    def epoca(f, arq, px, fr=1.0):\\n        mo = modelo[f\\\'{f}@{px}\\\']; m = mt[(f, arq)]*(px/256)**2\\n        return max(mo[\\\'train_a\\\'] + mo[\\\'train_c\\\']*m, .5)*fr + max(mo[\\\'val_a\\\'] + mo[\\\'val_c\\\']*m, .2)\\n\\n    f1, f2 = cfg[\\\'fase1\\\'], cfg[\\\'fase2\\\']; n_r = cfg[\\\'ruido\\\'][\\\'n_arquitecturas\\\']\\n    orden = orden_fase1(cfg, arqs)\\n    ref = buscar(arqs, cfg[\\\'referencia\\\'][\\\'w\\\'], cfg[\\\'referencia\\\'][\\\'b\\\'])[\\\'arq\\\']\\n    opciones = []\\n    for o in cfg[\\\'fidelidad_fase1\\\'][\\\'opciones\\\']:\\n        px, fr = o[\\\'px\\\'], o[\\\'fraccion_train\\\']\\n        h1 = sum(epoca(f, a[\\\'arq\\\'], px, fr) for f in FAMILIAS for a in arqs)*f1[\\\'epocas\\\']/3600\\n        hr = sum(epoca(f, a[\\\'arq\\\'], px, fr) for f in FAMILIAS for a in orden[:n_r])*f1[\\\'epocas\\\']/3600\\n        ha = sum(epoca(f, ref, px, fr) for f in FAMILIAS)*f1[\\\'epocas\\\']*len(cfg[\\\'ajuste_previo\\\'][\\\'lr\\\'])/3600\\n        opciones.append({**o, \\\'horas_fase1\\\': round(h1, 1), \\\'horas_ruido\\\': round(hr, 1), \\\'horas_ajuste\\\': round(ha, 2),\\n                         \\\'horas_total_antes_de_fase2\\\': round(h1 + hr + ha, 1)})\\n    fase2 = {}\\n    for k in (10, 20):\\n        n = k + 2\\n        por_f = {f: sorted(epoca(f, a[\\\'arq\\\'], 256) for a in arqs) for f in FAMILIAS}\\n        fase2[f\\\'k{k}\\\'] = {\\n            \\\'mediana_del_espacio\\\': round(sum(float(np.median(v))*n for v in por_f.values())*5*f2[\\\'epocas\\\']/3600, 1),\\n            \\\'los_mas_caros\\\': round(sum(sum(v[-n:]) for v in por_f.values())*5*f2[\\\'epocas\\\']/3600, 1)}\\n    estim = {\\\'estimado\\\': True, \\\'nota\\\': \\\'Piloto de una época con la caché llena, un proceso. No es una medición de la búsqueda.\\\',\\n             \\\'modelo_tiempo\\\': modelo, \\\'opciones_fase1\\\': opciones, \\\'horas_fase2_cota_30_epocas\\\': fase2,\\n             \\\'nota_fase2\\\': \\\'Cota con 30 épocas; con paciencia 5 las compactas del benchmark promediaron 19,6 a 29,2. \\\'\\n                           \\\'Depende de qué finalistas salgan: los anchos cuestan más.\\\',\\n             \\\'nota_paralelo\\\': \\\'Horas de un proceso. Varios procesos en la misma GPU bajan el reloj mientras la GPU \\\'\\n                              \\\'no esté saturada.\\\'}\\n    E.atomic_json(salida/\\\'estimacion_computo.json\\\', estim)\\n    print(json.dumps(estim, indent=2, ensure_ascii=False), flush=True)\\n    return estim\\n\\n\\ndef fijar_fidelidad(cfg, salida, horas):\\n    """Regla del protocolo: la primera opción (de mayor a menor fidelidad) cuya fase 1 estimada cabe."""\\n    archivo = Path(salida)/\\\'fidelidad.json\\\'\\n    if archivo.exists():\\n        previa = json.loads(archivo.read_text(encoding=\\\'utf-8\\\'))\\n        if previa[\\\'presupuesto_horas_fase1\\\'] != horas:\\n            raise RuntimeError(\\\'La fidelidad ya se fijó con otro presupuesto; no se cambia después\\\')\\n        return previa\\n    if any((Path(salida)/b).exists() for b in (\\\'ajuste\\\', \\\'fase1\\\')):\\n        raise RuntimeError(\\\'Ya hay corridas de ajuste o fase 1: la fidelidad no se puede fijar después\\\')\\n    est = json.loads((Path(salida)/\\\'estimacion_computo.json\\\').read_text(encoding=\\\'utf-8\\\'))\\n    caben = [o for o in est[\\\'opciones_fase1\\\'] if o[\\\'horas_total_antes_de_fase2\\\'] <= horas]\\n    elegida = caben[0] if caben else est[\\\'opciones_fase1\\\'][-1]\\n    out = {\\\'regla\\\': cfg[\\\'fidelidad_fase1\\\'][\\\'regla\\\'], \\\'presupuesto_horas_fase1\\\': horas,\\n           \\\'opciones_estimadas\\\': est[\\\'opciones_fase1\\\'], \\\'elegida\\\': elegida,\\n           \\\'cabe_en_el_presupuesto\\\': bool(caben),\\n           \\\'desviacion\\\': None if elegida[\\\'px\\\'] == 256 and elegida[\\\'fraccion_train\\\'] == 1 else\\n           \\\'Fidelidad de la fase 1 reducida respecto del 20 % de épocas a resolución completa; se declara y se \\\'\\n           \\\'mide su efecto con la correlación de rangos fase 1 - fase 2.\\\'}\\n    E.atomic_json(archivo, out)\\n    print(json.dumps(out, indent=2, ensure_ascii=False), flush=True)\\n    return out\\n\\n\\n# ---------------------------------------------------------------- principal\\ndef main(argv=None):\\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    p.add_argument(\\\'bloque\\\', choices=[\\\'listar\\\', \\\'piloto\\\', \\\'estimar\\\', \\\'fidelidad\\\', \\\'ajuste\\\', \\\'cerrar_ajuste\\\',\\n                                      \\\'fase1\\\', \\\'finalistas\\\', \\\'fase2\\\'])\\n    p.add_argument(\\\'--output\\\')\\n    p.add_argument(\\\'--config\\\', default=str(CONFIG))\\n    p.add_argument(\\\'--familias\\\', nargs=\\\'+\\\', choices=FAMILIAS)\\n    p.add_argument(\\\'--fragmento\\\', help=\\\'i/n: este proceso corre las tareas j con j mod n = i\\\')\\n    p.add_argument(\\\'--horas-fase1\\\', type=float, help=\\\'Presupuesto de horas de GPU para ajuste + fase 1 + ruido\\\')\\n    p.add_argument(\\\'--permitir-incompleta\\\', action=\\\'store_true\\\',\\n                   help=\\\'finalistas con la fase 1 cortada; queda registrado como desviación\\\')\\n    # Solo pruebas locales en CPU:\\n    p.add_argument(\\\'--local\\\', action=\\\'store_true\\\')\\n    p.add_argument(\\\'--raiz-imagenes\\\'); p.add_argument(\\\'--raiz-mascaras\\\')\\n    p.add_argument(\\\'--limite-train\\\', type=int); p.add_argument(\\\'--limite-val\\\', type=int)\\n    p.add_argument(\\\'--epocas-max\\\', type=int)\\n    p.add_argument(\\\'--n-arq\\\', type=int, help=\\\'Solo pruebas locales: primeras n arquitecturas del orden\\\')\\n    p.add_argument(\\\'--n-semillas\\\', type=int, help=\\\'Solo pruebas locales: primeras n semillas de la fase 2\\\')\\n    args = p.parse_args(argv)\\n\\n    cfg = json.loads(Path(args.config).read_text(encoding=\\\'utf-8\\\'))\\n    if args.bloque == \\\'listar\\\':\\n        listar(cfg); return\\n    if not args.output:\\n        raise ValueError(\\\'--output es obligatorio\\\')\\n    proto = json.loads(C.PROTOCOLO.read_text(encoding=\\\'utf-8\\\'))\\n    for f in FAMILIAS:  # la receta declarada es la del benchmark\\n        r = proto[\\\'models\\\'][f]\\n        assert (r[\\\'optimizer\\\'], r[\\\'lr\\\'], r[\\\'weight_decay\\\']) == (cfg[\\\'receta\\\'][\\\'optimizer\\\'], cfg[\\\'receta\\\'][\\\'lr\\\'],\\n                                                                cfg[\\\'receta\\\'][\\\'weight_decay\\\']), f\\n    assert proto[\\\'size\\\'] == 256 and proto[\\\'batch_size\\\'] == cfg[\\\'receta\\\'][\\\'batch_size\\\']\\n    salida = Path(args.output).resolve(); salida.mkdir(parents=True, exist_ok=True)\\n    familias = tuple(args.familias or FAMILIAS)\\n    arqs = espacio(cfg)\\n    if (args.n_arq or args.n_semillas) and not args.local:\\n        raise RuntimeError(\\\'--n-arq y --n-semillas solo en pruebas locales\\\')\\n    if args.n_semillas:\\n        cfg[\\\'fase2\\\'][\\\'semillas\\\'] = cfg[\\\'fase2\\\'][\\\'semillas\\\'][:args.n_semillas]\\n    if not args.local:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'La búsqueda requiere GPU; las pruebas en CPU van con --local\\\')\\n        if any([args.limite_train, args.limite_val, args.epocas_max, args.raiz_imagenes, args.raiz_mascaras]):\\n            raise RuntimeError(\\\'Los recortes de prueba solo se admiten con --local\\\')\\n    mt = tabla_macs(cfg, arqs, familias)\\n    if args.bloque == \\\'estimar\\\':\\n        estimar(cfg, salida, arqs, mt); return\\n    if args.bloque == \\\'fidelidad\\\':\\n        if args.horas_fase1 is None:\\n            raise ValueError(\\\'--horas-fase1 es obligatorio: es el presupuesto que decide la fidelidad\\\')\\n        fijar_fidelidad(cfg, salida, args.horas_fase1); return\\n    if args.bloque == \\\'cerrar_ajuste\\\':\\n        cerrar_ajuste(cfg, salida, familias); return\\n    if args.bloque == \\\'finalistas\\\':\\n        cerrar_finalistas(cfg, salida, familias, arqs, args.permitir_incompleta); return\\n    device = torch.device(\\\'cuda\\\' if torch.cuda.is_available() and not args.local else \\\'cpu\\\')\\n    train, val = C.filas(args)\\n    ident = identidad(args, train, val)\\n    if args.bloque == \\\'piloto\\\':\\n        tareas = tareas_piloto(cfg, arqs, args, mt)\\n        for row in train + val:  # caché llena antes de medir: así corre la búsqueda en un proceso largo\\n            C.ejemplo(row)\\n    elif args.bloque == \\\'ajuste\\\':\\n        tareas = tareas_ajuste(cfg, arqs, familias, salida, args, mt)\\n    elif args.bloque == \\\'fase1\\\':\\n        tareas = tareas_fase1(cfg, arqs, familias, salida, args, mt, args.n_arq)\\n    else:\\n        tareas = tareas_fase2(cfg, arqs, familias, salida, args, mt)\\n    hechos = []\\n    for t in C.fragmento(tareas, args.fragmento):\\n        r = entrenar(t, salida, proto, ident, device, train, val)\\n        hechos.append({k: r.get(k) for k in (\\\'id\\\', \\\'best_val_group_Dice\\\', \\\'epochs_completed\\\', \\\'seconds_total\\\')})\\n        if device.type == \\\'cuda\\\':\\n            torch.cuda.empty_cache()\\n    sello = (args.fragmento or \\\'todo\\\').replace(\\\'/\\\', \\\'de\\\')\\n    E.atomic_json(salida/f\\\'ultima_llamada_{args.bloque}_{sello}.json\\\',\\n                  {\\\'argv\\\': sys.argv[1:] if argv is None else argv, \\\'tareas\\\': len(hechos), \\\'resultados\\\': hechos})\\n    if args.bloque == \\\'ajuste\\\' and not args.fragmento:\\n        cerrar_ajuste(cfg, salida, familias)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'scripts/complemento_entreno_e2.py\': \'"""Complemento de entrenamiento del E2: lo que exige volver a entrenar.\\n\\nBloques (config/complemento_entreno_e2.json):\\n  sobreajuste  sobreajustar un lote fijo de 4 imágenes, las 10 redes (sanidad de implementación)\\n  ablacion     TransUNet sin Transformer, sin conexiones de salto y sin término Dice, 5 semillas\\n  permutacion  máscaras de train barajadas entre imágenes; pipeline completo; p de permutación\\n  hpo          búsqueda aleatoria con la misma secuencia de intentos para las 10 redes\\n  piloto       primera(s) época(s) de tareas reales para medir tiempo; se reutilizan al reanudar\\n  estimar      horas de GPU por bloque a partir del piloto (marcadas como estimadas)\\n\\nReutiliza el protocolo de experimento_e2_colab.py: misma pérdida, mismo orden de lotes por\\nsemilla, misma validación por grupos, mismas semillas y checkpoints atómicos por época. Solo\\ntrain/val: el test no se carga. Todo lo que produce es exploratorio, porque el test del\\nbenchmark ya fue observado.\\n\\nDos diferencias con train_model, ambas sin efecto numérico buscado:\\n  * caché en RAM de los ejemplos ya preprocesados (mismos arreglos que load_example);\\n  * cada tarea declara qué conserva: \\\'best\\\' guarda el mejor checkpoint y borra last.pt al\\n    terminar; \\\'ninguno\\\' guarda solo JSON (permutaciones e HPO no necesitan pesos).\\n"""\\nimport argparse\\nimport copy\\nimport importlib\\nimport importlib.metadata\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nimport experimento_e2_colab as E\\nfrom dataset_ready import load_example as _load_example_original\\nfrom dataset_ready import metrics, partition\\nfrom modelos_e2 import MODELS_E2, SOURCES, main_logits, official_package, resize\\n\\nPROJECT = E.PROJECT\\nCONFIG = PROJECT/\\\'config/complemento_entreno_e2.json\\\'\\nPROTOCOLO = PROJECT/\\\'config/experimento_e2.json\\\'\\nTEST_CSV = \\\'test.csv\\\'\\n\\n\\n# ---------------------------------------------------------------- datos y caché\\n_CACHE = {}\\n_USAR_CACHE = True\\n\\n\\ndef ejemplo(row):\\n    """(x, y, valid) de load_example, memorizado por id. Los arreglos no se modifican."""\\n    if not _USAR_CACHE:\\n        return _load_example_original(row)\\n    key = row[\\\'id\\\']\\n    if key not in _CACHE:\\n        x, y, v = _load_example_original(row)\\n        for a in (x, y, v):\\n            a.setflags(write=False)\\n        _CACHE[key] = (x, y, v)\\n    return _CACHE[key]\\n\\n\\ndef _load_example_compatible(row, data_root=None):\\n    if data_root is not None:\\n        return _load_example_original(row, data_root)\\n    return ejemplo(row)\\n\\n\\n# evaluate_validation y batch_tensors de experimento_e2_colab leen con esta función.\\nE.load_example = _load_example_compatible\\n\\n\\ndef lote(rows, device, fuente=None):\\n    """Como E.batch_tensors; con \\\'fuente\\\' la máscara sale de otra imagen de train."""\\n    fuente = fuente or {}\\n    xs, ys, vs = [], [], []\\n    for row in rows:\\n        x, y, v = ejemplo(row)\\n        if row[\\\'id\\\'] in fuente:\\n            y = ejemplo(fuente[row[\\\'id\\\']])[1]\\n        xs.append(x); ys.append(y); vs.append(v)\\n    x = torch.from_numpy(np.stack(xs)).to(device)\\n    y = torch.from_numpy(np.stack(ys).astype(\\\'float32\\\'))[:, None].to(device)\\n    v = torch.from_numpy(np.stack(vs))[:, None].to(device)\\n    return x, y, v\\n\\n\\ndef reubicar(rows, raiz_imagenes=None, raiz_mascaras=None):\\n    """Solo para pruebas locales: rutas absolutas a imágenes y máscaras fuera del paquete."""\\n    out = []\\n    for row in rows:\\n        row = dict(row)\\n        if raiz_imagenes:\\n            row[\\\'image_path\\\'] = str((Path(raiz_imagenes)/row[\\\'image_path\\\']).resolve())\\n        if raiz_mascaras:\\n            row[\\\'binary_mask_path\\\'] = str((Path(raiz_mascaras)/Path(row[\\\'binary_mask_path\\\']).name).resolve())\\n        out.append(row)\\n    return out\\n\\n\\ndef verificar_filas(rows):\\n    for row in rows:\\n        for path, esperado in [(PROJECT.parent/row[\\\'image_path\\\'], row[\\\'image_sha256\\\']),\\n                               (PROJECT/row[\\\'binary_mask_path\\\'], row[\\\'binary_mask_sha256\\\'])]:\\n            if E.sha(path) != esperado:\\n                raise RuntimeError(f\\\'Datos modificados: {path}\\\')\\n\\n\\ndef filas(args):\\n    train, val = partition(\\\'train\\\'), partition(\\\'val\\\')\\n    if args.local:\\n        train = reubicar(train, args.raiz_imagenes, args.raiz_mascaras)\\n        val = reubicar(val, args.raiz_imagenes, args.raiz_mascaras)\\n    if args.limite_train:\\n        train = train[:args.limite_train]\\n    if args.limite_val:\\n        val = val[:args.limite_val]\\n    if len(train) < 2 or not val:\\n        raise ValueError(\\\'Se necesitan al menos 2 imágenes de train y 1 de validación\\\')\\n    return train, val\\n\\n\\n# ---------------------------------------------------------------- entorno e identidad\\ndef entorno():\\n    try:\\n        return E.environment()\\n    except importlib.metadata.PackageNotFoundError as error:\\n        # Solo en pruebas locales: el entorno de Colab tiene todos los paquetes.\\n        return {\\\'incompleto\\\': f\\\'falta {error}\\\', \\\'torch\\\': torch.__version__,\\n                \\\'device\\\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\\'cpu\\\'}\\n\\n\\ndef identidad(args, train, val):\\n    E.verify_sources()\\n    if args.local:\\n        verificar_filas(train + val)\\n        datos = \\\'local:\\\' + E.digest(sorted((r[\\\'id\\\'], r[\\\'image_sha256\\\'], r[\\\'binary_mask_sha256\\\'])\\n                                           for r in train + val))\\n    else:\\n        datos = E.verify_data()\\n    return {\\\'source\\\': E.source_signature(), \\\'data\\\': datos}\\n\\n\\n# ---------------------------------------------------------------- modelos de la ablación\\nclass TransUNetVariante(nn.Module):\\n    """Mismo constructor que TransUNetAdapter (modelos_e2) con dos perillas del código oficial."""\\n\\n    def __init__(self, num_layers=12, n_skip=3):\\n        super().__init__()\\n        pkg = official_package(\\\'e2_transunet\\\', SOURCES/\\\'transunet/networks\\\')\\n        module = importlib.import_module(pkg + \\\'.vit_seg_modeling\\\')\\n        config = copy.deepcopy(module.CONFIGS[\\\'R50-ViT-B_16\\\'])\\n        config.n_classes, config.n_skip = 1, n_skip\\n        config.patches.grid = (16, 16)\\n        config.transformer.num_layers = num_layers\\n        self.model = module.VisionTransformer(config, img_size=256, num_classes=1)\\n\\n    def forward(self, x):\\n        return resize(self.model(x), x.shape[-2:])\\n\\n\\ndef construir(tarea):\\n    variante = tarea.get(\\\'variante\\\')\\n    if variante in (None, \\\'completo\\\', \\\'sin_dice\\\'):\\n        return MODELS_E2[tarea[\\\'modelo\\\']]()\\n    if variante == \\\'sin_transformer\\\':\\n        return TransUNetVariante(num_layers=0)\\n    if variante == \\\'sin_saltos\\\':\\n        return TransUNetVariante(n_skip=0)\\n    raise ValueError(f\\\'Variante desconocida: {variante}\\\')\\n\\n\\ndef comprobar_variante(tarea, model):\\n    """La variante quita de verdad el componente; si no, se detiene."""\\n    variante = tarea.get(\\\'variante\\\')\\n    if variante is None:\\n        return\\n    core = model.model\\n    if variante == \\\'sin_transformer\\\':\\n        assert len(core.transformer.encoder.layer) == 0\\n        assert core.config.n_skip == 3\\n    elif variante == \\\'sin_saltos\\\':\\n        assert len(core.transformer.encoder.layer) == 12\\n        assert all(block.conv1[0].in_channels == ch for block, ch in\\n                   zip(core.decoder.blocks, [512, 256, 128, 64]))\\n    else:\\n        assert len(core.transformer.encoder.layer) == 12 and core.config.n_skip == 3\\n\\n\\n# ---------------------------------------------------------------- pérdida\\ndef perdida(salida, y, v, modo, proto):\\n    if modo == \\\'bce_dice\\\':\\n        return E.compute_loss(salida, y, v, proto)\\n    if modo == \\\'bce\\\':\\n        if isinstance(salida, dict):\\n            raise ValueError(\\\'BCE sola definida solo para salidas simples\\\')\\n        valores = []\\n        for z, t, m in zip(salida, y, v):\\n            if not m.any():\\n                raise ValueError(\\\'Imagen sin píxeles válidos\\\')\\n            valores.append(F.binary_cross_entropy_with_logits(z[m], t[m]))\\n        return torch.stack(valores).mean()\\n    raise ValueError(f\\\'Pérdida desconocida: {modo}\\\')\\n\\n\\n# ---------------------------------------------------------------- permutación\\ndef mapa_permutacion(train, spec):\\n    """Permutación de las máscaras de train; misma para cualquier sesión con el mismo (semilla, b)."""\\n    if not spec:\\n        return {}, None\\n    orden = np.random.default_rng([spec[\\\'semilla\\\'], spec[\\\'b\\\']]).permutation(len(train))\\n    fuente = {train[i][\\\'id\\\']: train[j] for i, j in enumerate(orden)}\\n    info = {\\\'puntos_fijos\\\': int((orden == np.arange(len(train))).sum()),\\n            \\\'sha256_orden\\\': E.digest([train[j][\\\'id\\\'] for j in orden])}\\n    return fuente, info\\n\\n\\n# ---------------------------------------------------------------- entrenamiento por épocas\\ndef entrenar(tarea, salida, proto, ident, device, train, val, detener_tras=None):\\n    """Copia de E.train_model generalizada a variante, pérdida, permutación y presupuesto."""\\n    E.setup_seed(tarea[\\\'semilla\\\'])\\n    dest = Path(salida)/tarea[\\\'id\\\']\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {\\\'tarea\\\': tarea, \\\'protocolo\\\': {k: proto[k] for k in\\n            (\\\'batch_size\\\', \\\'size\\\', \\\'threshold\\\', \\\'pidnet_aux_weight\\\', \\\'pidnet_boundary_weight\\\')},\\n            \\\'identidad\\\': ident, \\\'entorno\\\': entorno(),\\n            \\\'filas\\\': {\\\'train\\\': len(train), \\\'val\\\': len(val)}}\\n    huella = E.digest(spec)\\n    cfg_file = dest/\\\'config.json\\\'\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text(encoding=\\\'utf-8\\\'))[\\\'fingerprint\\\'] != huella:\\n            raise RuntimeError(f\\\'Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.\\\')\\n    else:\\n        E.atomic_json(cfg_file, {\\\'fingerprint\\\': huella, **spec})\\n    res_file = dest/\\\'result.json\\\'\\n    if res_file.exists():\\n        result = json.loads(res_file.read_text(encoding=\\\'utf-8\\\'))\\n        if result.get(\\\'fingerprint\\\') != huella or not result.get(\\\'completed\\\'):\\n            raise RuntimeError(f\\\'Resultado incompatible: {dest}\\\')\\n        if tarea[\\\'conservar\\\'] == \\\'best\\\' and E.sha(dest/result[\\\'best_file\\\']) != result[\\\'best_sha256\\\']:\\n            raise RuntimeError(f\\\'Checkpoint completado corrupto o modificado: {dest}\\\')\\n        return result\\n    model = construir(tarea).to(device)\\n    comprobar_variante(tarea, model)\\n    optimizer = E.make_optimizer(model, tarea[\\\'optim\\\'])\\n    fuente, info_perm = mapa_permutacion(train, tarea.get(\\\'permutacion\\\'))\\n    last = dest/\\\'last.pt\\\'\\n    start, best, stale, history, best_file = 0, -1., 0, [], None\\n    if last.exists():\\n        saved = torch.load(last, map_location=\\\'cpu\\\', weights_only=False)  # solo archivos propios\\n        if saved[\\\'fingerprint\\\'] != huella:\\n            raise RuntimeError(\\\'Checkpoint incompatible\\\')\\n        model.load_state_dict(saved[\\\'model\\\']); optimizer.load_state_dict(saved[\\\'optimizer\\\'])\\n        start, best, stale = saved[\\\'epoch\\\']+1, saved[\\\'best\\\'], saved[\\\'stale\\\']\\n        history, best_file = saved[\\\'history\\\'], saved[\\\'best_file\\\']\\n        E.restore_rng(saved[\\\'rng\\\'])\\n    t0 = time.perf_counter()\\n    for epoch in range(start, tarea[\\\'epocas\\\']):\\n        if stale >= tarea[\\\'paciencia\\\']:\\n            break\\n        model.train(); started = time.perf_counter(); loss_sum = 0.; seen = 0\\n        for rows in E.training_batches(train, tarea[\\\'semilla\\\'], epoch, proto[\\\'batch_size\\\']):\\n            x, y, v = lote(rows, device, fuente)\\n            optimizer.zero_grad(set_to_none=True)\\n            loss = perdida(model(x), y, v, tarea[\\\'perdida\\\'], proto)\\n            if not torch.isfinite(loss):\\n                raise RuntimeError(f\\\'Pérdida no finita en {tarea["id"]}, época {epoch+1}\\\')\\n            loss.backward(); optimizer.step()\\n            loss_sum += loss.item()*len(rows); seen += len(rows)\\n        entreno_s = time.perf_counter() - started\\n        score, details = E.evaluate_validation(model, val, device)\\n        if not np.isfinite(score):\\n            raise RuntimeError(\\\'Validación no finita\\\')\\n        if score > best:\\n            best, stale = score, 0\\n            E.atomic_json(dest/\\\'best_val_predictions.json\\\', {\\\'epoch\\\': epoch+1, \\\'validation\\\': details})\\n            if tarea[\\\'conservar\\\'] == \\\'best\\\':\\n                best_file = f\\\'best_epoch_{epoch+1:03d}.pt\\\'\\n                E.atomic_torch(dest/best_file, {\\\'model\\\': model.state_dict(), \\\'epoch\\\': epoch,\\n                                                \\\'fingerprint\\\': huella, \\\'validation\\\': details})\\n            else:\\n                best_file = None\\n        else:\\n            stale += 1\\n        history.append({\\\'epoch\\\': epoch+1, \\\'train_loss\\\': loss_sum/seen, \\\'val_group_Dice\\\': score,\\n                        \\\'seconds\\\': time.perf_counter()-started, \\\'train_seconds\\\': entreno_s})\\n        E.atomic_torch(last, {\\\'fingerprint\\\': huella, \\\'epoch\\\': epoch, \\\'model\\\': model.state_dict(),\\n                              \\\'optimizer\\\': optimizer.state_dict(), \\\'best\\\': best, \\\'stale\\\': stale,\\n                              \\\'history\\\': history, \\\'best_file\\\': best_file, \\\'rng\\\': E.rng_state()})\\n        E.atomic_json(dest/\\\'history.json\\\', history)\\n        for old in dest.glob(\\\'best_epoch_*.pt\\\'):\\n            if old.name != best_file:\\n                old.unlink()\\n        print(f\\\'{tarea["id"]} epoch={epoch+1} loss={loss_sum/seen:.4f} val={score:.5f} \\\'\\n              f\\\'{history[-1]["seconds"]:.1f}s\\\', flush=True)\\n        if detener_tras is not None and epoch+1 >= detener_tras:\\n            return {\\\'completed\\\': False, \\\'id\\\': tarea[\\\'id\\\'], \\\'epochs_done\\\': epoch+1}\\n    result = {\\\'completed\\\': True, \\\'fingerprint\\\': huella, \\\'id\\\': tarea[\\\'id\\\'], \\\'bloque\\\': tarea[\\\'bloque\\\'],\\n              \\\'modelo\\\': tarea[\\\'modelo\\\'], \\\'variante\\\': tarea.get(\\\'variante\\\'), \\\'semilla\\\': tarea[\\\'semilla\\\'],\\n              \\\'optim\\\': tarea[\\\'optim\\\'], \\\'perdida\\\': tarea[\\\'perdida\\\'],\\n              \\\'permutacion\\\': tarea.get(\\\'permutacion\\\'), \\\'permutacion_info\\\': info_perm,\\n              \\\'intento\\\': tarea.get(\\\'intento\\\'),\\n              \\\'best_val_group_Dice\\\': best, \\\'best_epoch\\\': max(history, key=lambda h: h[\\\'val_group_Dice\\\'])[\\\'epoch\\\'],\\n              \\\'epochs_completed\\\': len(history), \\\'best_file\\\': best_file,\\n              \\\'parameters\\\': sum(p.numel() for p in model.parameters()),\\n              \\\'seconds_this_session\\\': time.perf_counter()-t0,\\n              \\\'seconds_total\\\': sum(h[\\\'seconds\\\'] for h in history),\\n              \\\'device\\\': str(device), \\\'test_evaluated\\\': False}\\n    if tarea[\\\'conservar\\\'] == \\\'best\\\':\\n        result[\\\'best_sha256\\\'] = E.sha(dest/best_file)\\n    E.atomic_json(res_file, result)\\n    last.unlink()  # el resultado queda cerrado; last.pt solo servía para reanudar\\n    return result\\n\\n\\n# ---------------------------------------------------------------- listas de tareas\\ndef receta(proto, modelo):\\n    return dict(proto[\\\'models\\\'][modelo])\\n\\n\\ndef tareas_ablacion(cfg, proto, variantes, semillas, args):\\n    modelo = cfg[\\\'ablacion\\\'][\\\'modelo\\\']\\n    perdidas = {k: v[\\\'perdida\\\'] for k, v in cfg[\\\'ablacion\\\'][\\\'variantes\\\'].items()}\\n    perdidas[\\\'completo\\\'] = \\\'bce_dice\\\'\\n    return [ajustar({\\\'id\\\': f\\\'abl_{var}_s{s}\\\', \\\'bloque\\\': \\\'ablacion\\\', \\\'modelo\\\': modelo, \\\'variante\\\': var,\\n                     \\\'semilla\\\': s, \\\'optim\\\': receta(proto, modelo), \\\'epocas\\\': proto[\\\'epochs\\\'],\\n                     \\\'paciencia\\\': proto[\\\'patience\\\'], \\\'perdida\\\': perdidas[var],\\n                     \\\'conservar\\\': cfg[\\\'ablacion\\\'][\\\'conservar\\\']}, args)\\n            for var in variantes for s in semillas]\\n\\n\\ndef tareas_permutacion(cfg, proto, bloque, B, args):\\n    pc = cfg[\\\'permutacion\\\']; spec = pc[\\\'bloques\\\'][bloque]; modelo = spec[\\\'modelo\\\']\\n    return [ajustar({\\\'id\\\': f\\\'perm_{modelo}_b{b:03d}\\\', \\\'bloque\\\': f\\\'permutacion:{bloque}\\\', \\\'modelo\\\': modelo,\\n                     \\\'semilla\\\': pc[\\\'semilla_entrenamiento\\\'], \\\'optim\\\': receta(proto, modelo),\\n                     \\\'epocas\\\': proto[\\\'epochs\\\'], \\\'paciencia\\\': proto[\\\'patience\\\'], \\\'perdida\\\': \\\'bce_dice\\\',\\n                     \\\'permutacion\\\': {\\\'semilla\\\': pc[\\\'semilla_permutaciones\\\'], \\\'b\\\': b},\\n                     \\\'conservar\\\': pc[\\\'conservar\\\']}, args)\\n            for b in range(1, B+1)]\\n\\n\\ndef secuencia_hpo(cfg):\\n    """Secuencia común de intentos; se genera completa para que todo prefijo sea estable."""\\n    h = cfg[\\\'hpo\\\']; rng = np.random.default_rng(h[\\\'semilla_secuencia\\\'])\\n    intentos = [{\\\'t\\\': 0, \\\'exponente_lr\\\': 0.0, \\\'weight_decay\\\': None}]\\n    for t in range(1, h[\\\'T_disenado\\\']):\\n        intentos.append({\\\'t\\\': t, \\\'exponente_lr\\\': float(rng.uniform(-1, 1)),\\n                         \\\'weight_decay\\\': float(rng.choice([0.0, 1e-4, 1e-2]))})\\n    return intentos\\n\\n\\ndef tareas_hpo(cfg, proto, T, modelos, args):\\n    h = cfg[\\\'hpo\\\']; presupuesto = h[\\\'presupuesto_por_intento\\\']; out = []\\n    for intento in secuencia_hpo(cfg)[:T]:\\n        for modelo in modelos:\\n            optim = receta(proto, modelo)\\n            optim[\\\'lr\\\'] = float(optim[\\\'lr\\\'] * 10**intento[\\\'exponente_lr\\\'])\\n            if intento[\\\'weight_decay\\\'] is not None:\\n                optim[\\\'weight_decay\\\'] = intento[\\\'weight_decay\\\']\\n            out.append(ajustar({\\\'id\\\': f\\\'hpo_{modelo}_t{intento["t"]:02d}\\\', \\\'bloque\\\': \\\'hpo\\\', \\\'modelo\\\': modelo,\\n                                \\\'semilla\\\': presupuesto[\\\'semilla\\\'], \\\'optim\\\': optim, \\\'intento\\\': intento,\\n                                \\\'epocas\\\': presupuesto[\\\'epocas\\\'], \\\'paciencia\\\': presupuesto[\\\'paciencia\\\'],\\n                                \\\'perdida\\\': \\\'bce_dice\\\', \\\'conservar\\\': h[\\\'conservar\\\']}, args))\\n    return out\\n\\n\\ndef ajustar(tarea, args):\\n    """Solo pruebas locales: recortar épocas y marcar la tarea para que no se confunda."""\\n    if args.epocas_max:\\n        tarea[\\\'epocas\\\'] = min(tarea[\\\'epocas\\\'], args.epocas_max)\\n        tarea[\\\'prueba_local\\\'] = True\\n    if args.limite_train or args.limite_val:\\n        tarea[\\\'prueba_local\\\'] = True\\n    return tarea\\n\\n\\ndef fragmento(tareas, texto):\\n    if not texto:\\n        return tareas\\n    i, n = map(int, texto.split(\\\'/\\\'))\\n    if not 0 <= i < n:\\n        raise ValueError(\\\'--fragmento debe ser i/n con 0 <= i < n\\\')\\n    return [t for k, t in enumerate(tareas) if k % n == i]\\n\\n\\n# ---------------------------------------------------------------- sobreajuste a un lote\\ndef sobreajuste(modelo, salida, cfg, proto, ident, device, train, pasos):\\n    sc = cfg[\\\'sobreajuste\\\']\\n    elegidas = sorted(np.random.default_rng(sc[\\\'semilla_eleccion_imagenes\\\'])\\n                      .choice(len(train), size=min(sc[\\\'imagenes\\\'], len(train)), replace=False).tolist())\\n    rows = [train[i] for i in elegidas]\\n    spec = {\\\'modelo\\\': modelo, \\\'pasos\\\': pasos, \\\'imagenes\\\': [r[\\\'id\\\'] for r in rows],\\n            \\\'optim\\\': receta(proto, modelo), \\\'identidad\\\': ident, \\\'entorno\\\': entorno()}\\n    huella = E.digest(spec)\\n    dest = Path(salida)/\\\'sobreajuste\\\'; dest.mkdir(parents=True, exist_ok=True)\\n    archivo = dest/f\\\'{modelo}.json\\\'\\n    if archivo.exists():\\n        previo = json.loads(archivo.read_text(encoding=\\\'utf-8\\\'))\\n        if previo[\\\'fingerprint\\\'] == huella:\\n            return previo\\n        raise RuntimeError(f\\\'{archivo} es de otra configuración; usa otra carpeta\\\')\\n    E.setup_seed(sc[\\\'semilla_entrenamiento\\\'])\\n    model = MODELS_E2[modelo]().to(device)\\n    optimizer = E.make_optimizer(model, spec[\\\'optim\\\'])\\n    x, y, v = lote(rows, device)\\n    curva = []; started = time.perf_counter()\\n    for paso in range(1, pasos+1):\\n        model.train(); optimizer.zero_grad(set_to_none=True)\\n        loss = perdida(model(x), y, v, \\\'bce_dice\\\', proto)\\n        if not torch.isfinite(loss):\\n            raise RuntimeError(f\\\'Pérdida no finita: {modelo}, paso {paso}\\\')\\n        loss.backward(); optimizer.step()\\n        if paso == 1 or paso % sc[\\\'registrar_cada\\\'] == 0 or paso == pasos:\\n            curva.append({\\\'paso\\\': paso, \\\'perdida\\\': loss.item()})\\n    model.eval()\\n    devices = [torch.cuda.current_device()] if device.type == \\\'cuda\\\' else []\\n    with torch.no_grad(), torch.random.fork_rng(devices=devices):\\n        torch.manual_seed(20261118)\\n        prob = main_logits(model(x)).sigmoid().cpu().numpy()\\n    por_imagen = [metrics(prob[i, 0], y[i, 0].cpu().numpy(), v[i, 0].cpu().numpy())[\\\'Dice\\\']\\n                  for i in range(len(rows))]\\n    dice = float(np.mean(por_imagen))\\n    result = {\\\'fingerprint\\\': huella, **{k: spec[k] for k in (\\\'modelo\\\', \\\'pasos\\\', \\\'imagenes\\\', \\\'optim\\\')},\\n              \\\'dice_lote_modo_eval\\\': dice, \\\'dice_por_imagen\\\': por_imagen,\\n              \\\'perdida_inicial\\\': curva[0][\\\'perdida\\\'], \\\'perdida_final\\\': curva[-1][\\\'perdida\\\'],\\n              \\\'umbral\\\': sc[\\\'umbral_dice\\\'], \\\'pasa\\\': dice >= sc[\\\'umbral_dice\\\'], \\\'curva\\\': curva,\\n              \\\'segundos\\\': time.perf_counter()-started, \\\'device\\\': str(device)}\\n    E.atomic_json(archivo, result)\\n    print(f\\\'sobreajuste {modelo}: Dice={dice:.4f} pérdida {curva[0]["perdida"]:.4f}->{curva[-1]["perdida"]:.4f} \\\'\\n          f\\\'pasa={result["pasa"]}\\\', flush=True)\\n    return result\\n\\n\\n# ---------------------------------------------------------------- piloto y estimación\\nPILOTO = [(\\\'ablacion\\\', \\\'abl_sin_dice_s11\\\', 1), (\\\'ablacion\\\', \\\'abl_sin_transformer_s11\\\', 1),\\n          (\\\'ablacion\\\', \\\'abl_sin_saltos_s11\\\', 1), (\\\'permutacion\\\', \\\'perm_fcn_b001\\\', 2)]\\n\\n\\ndef estimar(salida, cfg, proto):\\n    """Horas de GPU por bloque. Todo lo que sale de aquí es una ESTIMACIÓN, no una medición."""\\n    ref = cfg[\\\'referencia_original\\\']; salida = Path(salida)\\n\\n    def historia(tid):\\n        p = salida/tid/\\\'history.json\\\'\\n        return json.loads(p.read_text(encoding=\\\'utf-8\\\')) if p.exists() else None\\n\\n    medidas = {tid: historia(tid) for _, tid, _ in PILOTO}\\n    faltan = [k for k, v in medidas.items() if not v]\\n    if faltan:\\n        raise RuntimeError(f\\\'Falta el piloto de: {faltan}\\\')\\n    s_full = medidas[\\\'abl_sin_dice_s11\\\'][0][\\\'seconds\\\']\\n    factor = s_full / ref[\\\'segundos_por_epoca_l4\\\'][\\\'transunet_2d\\\']\\n    s_fcn = medidas[\\\'perm_fcn_b001\\\'][-1][\\\'seconds\\\']  # segunda época: caché ya llena\\n    e_trans = ref[\\\'epocas_medias\\\'][\\\'transunet_2d\\\']\\n\\n    def h(seg):\\n        return round(seg/3600, 2)\\n\\n    s_var = {v: medidas[f\\\'abl_{v}_s11\\\'][0][\\\'seconds\\\'] for v in (\\\'sin_dice\\\', \\\'sin_transformer\\\', \\\'sin_saltos\\\')}\\n    abl = {\\\'central\\\': h(sum(s_var.values())*e_trans*5), \\\'cota_superior_30_epocas\\\': h(sum(s_var.values())*30*5),\\n           \\\'supuesto\\\': f\\\'{e_trans} épocas medias por corrida, como TransUNet en el benchmark\\\'}\\n    perm_epocas = []\\n    for p in salida.glob(\\\'perm_fcn_b*/result.json\\\'):\\n        perm_epocas.append(json.loads(p.read_text(encoding=\\\'utf-8\\\'))[\\\'epochs_completed\\\'])\\n    e_perm = float(np.mean(perm_epocas)) if perm_epocas else None\\n    bloques = cfg[\\\'permutacion\\\'][\\\'bloques\\\']\\n\\n    def perm(seg_epoca, B):\\n        d = {\\\'minimo_6_epocas\\\': h(seg_epoca*6*B), \\\'cota_superior_30_epocas\\\': h(seg_epoca*30*B)}\\n        if e_perm:\\n            d[\\\'con_epocas_medias_observadas\\\'] = h(seg_epoca*e_perm*B)\\n        return d\\n\\n    hpo_epoca = sum(ref[\\\'segundos_por_epoca_l4\\\'][m] for m in cfg[\\\'sobreajuste\\\'][\\\'modelos\\\'])*factor\\n    hp = cfg[\\\'hpo\\\']\\n    estim = {\\n        \\\'estimado\\\': True,\\n        \\\'nota\\\': \\\'Estimación con las épocas del piloto y los tiempos del benchmark; no es una medición del bloque.\\\',\\n        \\\'factor_gpu_frente_al_benchmark\\\': round(factor, 3),\\n        \\\'segundos_por_epoca_medidos\\\': {**s_var, \\\'fcn_segunda_epoca_con_cache\\\': s_fcn},\\n        \\\'horas_l4_estimadas\\\': {\\n            \\\'sobreajuste_10_redes\\\': h(sum(ref[\\\'segundos_por_epoca_l4\\\'].values())*factor\\n                                      * cfg[\\\'sobreajuste\\\'][\\\'pasos\\\']/600),\\n            \\\'ablacion_15_corridas\\\': abl,\\n            \\\'control_completo_5_corridas\\\': h(s_full*e_trans*5),\\n            \\\'permutacion_pipeline_fcn\\\': {\\\'B\\\': bloques[\\\'pipeline_fcn\\\'][\\\'B\\\'],\\n                                         **perm(s_fcn, bloques[\\\'pipeline_fcn\\\'][\\\'B\\\'])},\\n            \\\'permutacion_transunet\\\': {\\\'B\\\': bloques[\\\'transunet\\\'][\\\'B\\\'],\\n                                      **perm(s_full, bloques[\\\'transunet\\\'][\\\'B\\\'])},\\n            \\\'hpo_por_intento_10_redes\\\': {\\\'minimo_4_epocas\\\': h(hpo_epoca*4),\\n                                         \\\'cota_superior_10_epocas\\\': h(hpo_epoca*10)},\\n            \\\'hpo_T_recomendado\\\': {\\\'T\\\': hp[\\\'T_recomendado_antes_del_domingo\\\'],\\n                                  \\\'cota_superior\\\': h(hpo_epoca*10*hp[\\\'T_recomendado_antes_del_domingo\\\'])},\\n            \\\'hpo_T_disenado\\\': {\\\'T\\\': hp[\\\'T_disenado\\\'], \\\'cota_superior\\\': h(hpo_epoca*10*hp[\\\'T_disenado\\\'])},\\n        },\\n        \\\'epocas_medias_permutacion_fcn_observadas\\\': e_perm,\\n        \\\'permutaciones_fcn_completas\\\': len(perm_epocas),\\n    }\\n    E.atomic_json(salida/\\\'estimacion_computo.json\\\', estim)\\n    print(json.dumps(estim, indent=2, ensure_ascii=False), flush=True)\\n    return estim\\n\\n\\n# ---------------------------------------------------------------- principal\\ndef main(argv=None):\\n    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    p.add_argument(\\\'bloque\\\', choices=[\\\'sobreajuste\\\', \\\'ablacion\\\', \\\'permutacion\\\', \\\'hpo\\\', \\\'piloto\\\', \\\'estimar\\\'])\\n    p.add_argument(\\\'--output\\\', required=True)\\n    p.add_argument(\\\'--config\\\', default=str(CONFIG))\\n    p.add_argument(\\\'--modelos\\\', nargs=\\\'+\\\')\\n    p.add_argument(\\\'--variantes\\\', nargs=\\\'+\\\', choices=[\\\'sin_transformer\\\', \\\'sin_saltos\\\', \\\'sin_dice\\\', \\\'completo\\\'])\\n    p.add_argument(\\\'--semillas\\\', nargs=\\\'+\\\', type=int)\\n    p.add_argument(\\\'--permutacion\\\', choices=[\\\'pipeline_fcn\\\', \\\'transunet\\\'], default=\\\'pipeline_fcn\\\')\\n    p.add_argument(\\\'--B\\\', type=int, help=\\\'Número de permutaciones; por defecto el de la configuración\\\')\\n    p.add_argument(\\\'--T\\\', type=int, help=\\\'Intentos de HPO; por defecto T_recomendado_antes_del_domingo\\\')\\n    p.add_argument(\\\'--pasos\\\', type=int, help=\\\'Pasos de sobreajuste; por defecto los de la configuración\\\')\\n    p.add_argument(\\\'--fragmento\\\', help=\\\'i/n: esta sesión corre las tareas k con k mod n = i\\\')\\n    p.add_argument(\\\'--sin-cache\\\', action=\\\'store_true\\\')\\n    # Solo pruebas locales en CPU:\\n    p.add_argument(\\\'--local\\\', action=\\\'store_true\\\', help=\\\'Permite CPU y rutas locales; no es una corrida válida\\\')\\n    p.add_argument(\\\'--raiz-imagenes\\\'); p.add_argument(\\\'--raiz-mascaras\\\')\\n    p.add_argument(\\\'--limite-train\\\', type=int); p.add_argument(\\\'--limite-val\\\', type=int)\\n    p.add_argument(\\\'--epocas-max\\\', type=int)\\n    args = p.parse_args(argv)\\n\\n    global _USAR_CACHE\\n    _USAR_CACHE = not args.sin_cache\\n    cfg = json.loads(Path(args.config).read_text(encoding=\\\'utf-8\\\'))\\n    proto = json.loads(PROTOCOLO.read_text(encoding=\\\'utf-8\\\'))\\n    salida = Path(args.output).resolve(); salida.mkdir(parents=True, exist_ok=True)\\n    if args.bloque == \\\'estimar\\\':\\n        estimar(salida, cfg, proto); return\\n    if not args.local:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'El complemento requiere GPU; las pruebas en CPU van con --local\\\')\\n        if any([args.limite_train, args.limite_val, args.epocas_max, args.raiz_imagenes, args.raiz_mascaras]):\\n            raise RuntimeError(\\\'Los recortes de prueba solo se admiten con --local\\\')\\n    device = torch.device(\\\'cuda\\\' if torch.cuda.is_available() and not args.local else \\\'cpu\\\')\\n    train, val = filas(args)\\n    ident = identidad(args, train, val)\\n    semillas = args.semillas or cfg[\\\'comun\\\'][\\\'semillas\\\']\\n    resultados = []\\n    if args.bloque == \\\'sobreajuste\\\':\\n        for modelo in args.modelos or cfg[\\\'sobreajuste\\\'][\\\'modelos\\\']:\\n            resultados.append(sobreajuste(modelo, salida, cfg, proto, ident, device, train,\\n                                          args.pasos or cfg[\\\'sobreajuste\\\'][\\\'pasos\\\']))\\n            if device.type == \\\'cuda\\\':\\n                torch.cuda.empty_cache()\\n    else:\\n        if args.bloque == \\\'ablacion\\\':\\n            tareas = tareas_ablacion(cfg, proto, args.variantes or list(cfg[\\\'ablacion\\\'][\\\'variantes\\\']), semillas, args)\\n        elif args.bloque == \\\'permutacion\\\':\\n            B = args.B or cfg[\\\'permutacion\\\'][\\\'bloques\\\'][args.permutacion][\\\'B\\\']\\n            tareas = tareas_permutacion(cfg, proto, args.permutacion, B, args)\\n        elif args.bloque == \\\'hpo\\\':\\n            T = args.T or cfg[\\\'hpo\\\'][\\\'T_recomendado_antes_del_domingo\\\']\\n            if T > cfg[\\\'hpo\\\'][\\\'T_disenado\\\']:\\n                raise ValueError(\\\'T mayor que la secuencia diseñada\\\')\\n            tareas = tareas_hpo(cfg, proto, T, args.modelos or cfg[\\\'sobreajuste\\\'][\\\'modelos\\\'], args)\\n        else:  # piloto\\n            todas = {t[\\\'id\\\']: t for t in tareas_ablacion(cfg, proto, [\\\'sin_dice\\\', \\\'sin_transformer\\\', \\\'sin_saltos\\\'],\\n                                                         [11], args)\\n                     + tareas_permutacion(cfg, proto, \\\'pipeline_fcn\\\', 1, args)}\\n            for _, tid, epocas in PILOTO:\\n                resultados.append(entrenar(todas[tid], salida, proto, ident, device, train, val,\\n                                           detener_tras=epocas))\\n            if not args.local:\\n                estimar(salida, cfg, proto)\\n            tareas = []\\n        for tarea in fragmento(tareas, args.fragmento):\\n            resultados.append(entrenar(tarea, salida, proto, ident, device, train, val))\\n            if device.type == \\\'cuda\\\':\\n                torch.cuda.empty_cache()\\n    E.atomic_json(salida/f\\\'ultima_llamada_{args.bloque}.json\\\',\\n                  {\\\'argv\\\': sys.argv[1:] if argv is None else argv, \\\'resultados\\\': resultados})\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'config/busqueda_arquitectura_e2.json\': \'{\\n  "schema": 1,\\n  "experimento": "e2_busqueda_arquitectura_compactas_v1",\\n  "fecha_protocolo": "2026-11-22",\\n  "estado": "Protocolo fijado antes de correr. Ningún número de este archivo es un resultado de la búsqueda.",\\n  "documento": "docs/PROTOCOLO_BUSQUEDA_ARQUITECTURA_E2.md",\\n  "control_guia": "Sección 14 (14.1 a 14.7) y sección 4.1: todo modelo sin arquitectura definida en su artículo recorre el espacio obligatorio de 1000 arquitecturas en dos fases. Checklist 38 y B14.",\\n  "exposicion_test": "El test del benchmark ya fue observado. Esta búsqueda no carga el test: el ajuste previo, el cribado, los finalistas y la selección usan solo validación, y todo se declara exploratorio. El paso A6 de la guía (una sola pasada por test) no se hace en esta búsqueda.",\\n  "protocolo_base": "config/experimento_e2.json",\\n  "familias": {\\n    "fcn": "Codificador de 3 bloques convolucionales (conv 3x3 + ReLU, dos veces) con max pooling entre bloques; cabezal 1x1 sobre el bloque 3 e interpolación bilineal a la resolución de entrada. Sin decodificador.",\\n    "unet": "Codificador de 3 niveles; decodificador espejo de 2 bloques que concatena el mapa del codificador del mismo nivel (d2: w3+w2 -> w2; d1: w2+w1 -> w1); cabezal 1x1.",\\n    "segnet": "Codificador de 3 niveles con índices de max pooling; reducción 1x1 w3 -> w2; decodificador espejo con max unpooling (d2: w2 -> w1; d1: w1 -> w1); cabezal 1x1."\\n  },\\n  "espacio": {\\n    "profundidad": 3,\\n    "anchos": [16, 32, 64, 128, 256],\\n    "que_es_w_i": "Canales de los dos conv 3x3 del bloque del nivel i del codificador; el decodificador espeja los anchos (sección 14.2, fila de codificador-decodificador).",\\n    "dropout_p": 0.2,\\n    "capa_dropout": "torch.nn.Dropout (elemento a elemento), la misma en las tres familias",\\n    "posiciones_dropout": {\\n      "fcn": "b_i = 1: dropout tras el bloque i del codificador (el del bloque 3 queda antes del cabezal 1x1).",\\n      "unet": "b_1: tras e1 y tras su espejo d1; b_2: tras e2 y tras su espejo d2; b_3: tras e3, que es el cuello y no tiene espejo.",\\n      "segnet": "b_1: tras e1 y tras su espejo d1; b_2: tras e2 y tras su espejo d2; b_3: tras e3 (antes de la reducción 1x1), cuello sin espejo."\\n    },\\n    "cardinalidad": 1000,\\n    "orden_identificadores": "Producto cartesiano itertools.product(W, W, W, {0,1}, {0,1}, {0,1}); el identificador es a0000 a a0999 en ese orden.",\\n    "listado": "config/arquitecturas_compactas_e2.csv"\\n  },\\n  "referencia": {"w": [64, 64, 64], "b": [0, 1, 0], "papel": "Arquitectura de referencia de la guía: ajuste previo y control en la fase 2."},\\n  "heredada": {"w": [16, 32, 64], "b": [0, 0, 0], "papel": "La compacta del benchmark. Está dentro del espacio y es numéricamente idéntica a modelos_base.py con la misma semilla; entra en la fase 2 como segundo control para la comparación."},\\n  "receta": {\\n    "fuente": "Receta de fcn, unet y segnet en config/experimento_e2.json",\\n    "optimizer": "Adam", "lr": 0.001, "weight_decay": 0.0,\\n    "batch_size": 2, "size": 256, "perdida": "0,5 BCE + 0,5 Dice suave por imagen, solo en la zona válida",\\n    "umbral": 0.5, "precision": "float32 con TF32 de cuDNN por defecto y algoritmos deterministas, como el benchmark",\\n    "criterio_validacion": "Dice medio por grupo en validación (media de imágenes dentro del grupo y luego entre grupos)",\\n    "inicializacion": "La de PyTorch por defecto, tras setup_seed(semilla), igual en todas las arquitecturas"\\n  },\\n  "ajuste_previo": {\\n    "control_guia": "14.3: los hiperparámetros se fijan antes de la búsqueda con la arquitectura de referencia (64,64,64) con dropout (0,1,0), usando solo validación.",\\n    "que_se_ajusta": "Solo la tasa de aprendizaje de Adam, por familia; el resto de la receta queda fijo.",\\n    "lr": [0.0003, 0.001, 0.003],\\n    "presupuesto": "El de la fase 1 (semilla, épocas y paciencia)",\\n    "regla": "Se queda la tasa con mayor Dice de validación. Si su ventaja sobre la receta (0,001) es menor que 0,005, se queda la receta. La tasa elegida queda fija para las 1000 arquitecturas y para la fase 2 de esa familia."\\n  },\\n  "fase1": {\\n    "semilla": 97,\\n    "epocas": 6,\\n    "paciencia": 5,\\n    "fraccion_de_epocas": "6 de 30 = 20 %, dentro del 20-30 % que da la guía como ejemplo",\\n    "semilla_orden": 20261122,\\n    "orden": "Las 1000 arquitecturas en una permutación fija (semilla_orden); para cada una, las tres familias seguidas. Cualquier prefijo cortado es una muestra aleatoria del espacio, igual para las tres familias.",\\n    "metrica": "Mejor Dice de validación por grupo entre las épocas corridas",\\n    "registro": "Por época: pérdida de entrenamiento, pérdida de validación y Dice de validación por grupo; parámetros, MACs y segundos. No guarda pesos.",\\n    "exhaustiva": "Las 1000 de cada familia, sin poda previa"\\n  },\\n  "fidelidad_fase1": {\\n    "control_guia": "14.4: si el cómputo no alcanza, no se recorta en silencio: se declara la desviación, se justifica y se reporta su efecto. Las 1000 arquitecturas, la semilla y las 6 épocas no cambian en ninguna opción.",\\n    "opciones": [\\n      {"nombre": "A", "px": 256, "fraccion_train": 1.0, "descripcion": "La del protocolo: imagen completa a 256 px, todo train"},\\n      {"nombre": "B", "px": 128, "fraccion_train": 1.0, "descripcion": "La red ve la imagen reducida a 128 px por promedio de área; su salida vuelve a 256 por interpolación bilineal y la pérdida y el Dice se miden a 256 contra la máscara original. Todo train. Cómputo de la red / 4"},\\n      {"nombre": "C", "px": 128, "fraccion_train": 0.5, "descripcion": "Como B, entrenando con la mitad de train elegida por grupos de similitud (semilla_subconjunto), la misma para todas. Validación completa"}\\n    ],\\n    "semilla_subconjunto": 20261123,\\n    "regla": "Con el piloto en L4 se estiman las horas de ajuste previo + fase 1 + repetición de cada opción. Se elige la primera opción, en el orden A, B, C, cuya estimación no pase del presupuesto de horas que se fija antes de ver el piloto. Si ninguna cabe, se usa C y se registra que excede el presupuesto. La elección queda en fidelidad.json y no se cambia después de empezar el ajuste previo. La fase 2 es siempre a fidelidad completa.",\\n    "efecto": "Se reporta la correlación de rangos entre la fase 1 y la fase 2 en los finalistas: si es baja, el cribado reducido fue poco informativo y se declara."\\n  },\\n  "ruido": {\\n    "control_guia": "14.4: estimar el ruido por semilla con un piloto de arquitecturas repetidas y elegir k en consecuencia; 14.6: repetir la fase 1 con al menos dos semillas en una muestra representativa.",\\n    "arquitecturas": "Las primeras 40 del orden de la fase 1 (muestra aleatoria del espacio), en las tres familias",\\n    "n_arquitecturas": 40,\\n    "semilla_extra": 98,\\n    "estimador": "sigma_semilla^2 = media de (d^2)/2 con d la diferencia entre las semillas 97 y 98 de la misma arquitectura; sigma_arq^2 = varianza de las 1000 de la fase 1 menos sigma_semilla^2 (truncada en 0); r = sigma_semilla / sigma_arq, por familia",\\n    "regla_k": "k = 10 si r <= 1 en las tres familias; k = 20 si r > 1 en alguna. El mismo k para las tres (14.4: mismo |A|, k y R)."\\n  },\\n  "finalistas": {\\n    "orden": "Dice de la fase 1 descendente; empate exacto, menos parámetros",\\n    "controles": "Se añaden la referencia (64,64,64)/(0,1,0) y la heredada (16,32,64)/(0,0,0) si no están entre los k",\\n    "nota": "La fase 1 es un cribado: el ganador es el mejor entre los finalistas, no necesariamente el mejor de las 1000."\\n  },\\n  "fase2": {\\n    "semillas": [11, 23, 37, 53, 71],\\n    "nota_semillas": "Las del benchmark, distintas de las de la fase 1 (97, 98). Con la misma semilla el orden de lotes es el mismo que en el benchmark, así que la heredada se compara pareada por semilla.",\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "orden": "Por semilla y luego por familia y arquitectura: un corte deja el mismo número de semillas en todos los finalistas",\\n    "registro": "Por corrida: Dice de validación por grupo en la mejor época, predicciones por imagen de validación, Dice de entrenamiento por grupo con los pesos de la mejor época (brecha train-validación), parámetros, MACs y segundos. Guarda el mejor checkpoint."\\n  },\\n  "seleccion": {\\n    "control_guia": "14.5",\\n    "delta": 0.02,\\n    "nota_delta": "δ = 0,02 de Dice, heredado del preregistro",\\n    "regla": "(i) Mayor media de Dice de validación en las 5 semillas de la fase 2. (ii) Empate técnico con el mejor: diferencia de medias menor que δ o intervalos t al 95 % de las medias solapados. Entre los empatados (incluido el mejor), la de menos parámetros; si empatan en parámetros, la de mayor media. Los controles (referencia y heredada) compiten como cualquier finalista.",\\n    "indistinguibles": "TOST pareado por semilla con margen ±δ frente al mejor (dos t unilaterales al 5 %, 4 gl); se reporta el tamaño del conjunto equivalente",\\n    "prohibido": "Mirar el test para elegir o reabrir la búsqueda después de verlo"\\n  },\\n  "comparacion_heredada": {\\n    "pregunta": "¿La arquitectura ganadora de cada familia supera a la compacta heredada del benchmark, en validación?",\\n    "contraste": "Ganadora frente a heredada, ambas de la fase 2 (misma receta, mismas 5 semillas): diferencia media pareada por semilla con IC95 t; bootstrap por grupo de validación (2000 remuestreos) del Dice medio en las 5 semillas; TOST con ±δ.",\\n    "contraste_secundario": "Ganadora frente a las 5 corridas originales del benchmark (resultados/validacion_por_imagen_e2/ o auditoría), solo descriptivo: la receta puede diferir si el ajuste previo cambia la tasa.",\\n    "lectura": "Si la heredada cae en el conjunto equivalente al ganador, la herencia queda justificada por la búsqueda. Si el ganador la supera por más de δ, se declara que el benchmark subestimó esa familia y se reporta la diferencia, sin cambiar las conclusiones del test ya cerrado."\\n  },\\n  "analisis_espacio": {\\n    "control_guia": "14.6",\\n    "superficie": "Mapa de calor del Dice de la fase 1 sobre (w1, w2) por w3, uno por patrón de dropout (figuras en local)",\\n    "modelo_lineal": "y = b0 + sum_i sum_w beta_iw 1[w_i = w] + sum_i gamma_i b_i + sum_{i<j} lambda_ij b_i b_j + e, por familia, con IC95 y proporción de varianza (suma de cuadrados tipo II)",\\n    "pruebas_factores": "F parcial por factor (anchos de cada capa, dropout de cada posición, interacciones de dropout) con Holm por el número de factores; tendencia sobre log2 w por capa",\\n    "dropout": "Wilcoxon pareado entre (0,0,0) y la media de los 7 patrones con dropout, con las 125 combinaciones de anchos como bloques; tamaño de efecto r = Z/sqrt(n)",\\n    "pareto": "Dice de validación frente a parámetros y frente a segundos; arquitecturas no dominadas",\\n    "sensibilidad": "Dispersión entre arquitecturas (rango, DE) frente a la DE por semilla (fase 2 y repetición). Si son parecidas, la arquitectura es irrelevante para este problema y se concluye así.",\\n    "sobreajuste": "Brecha train-validación de la fase 2 por finalista; en la fase 1, pérdida de validación menos pérdida de entrenamiento en la última época, por ancho total y patrón de dropout",\\n    "borde": "Se informa si el ganador cae en el borde del espacio (algún w_i = 256 o = 16)",\\n    "correlacion": "Spearman entre el Dice de la fase 1 y la media de la fase 2 en los finalistas"\\n  },\\n  "computo": {\\n    "formula": "C_total = m (|A| t1 + (k + 2) R t2) + ajuste previo + repetición; m = 3, |A| = 1000, R = 5",\\n    "estimacion": "resultados/busqueda_arquitectura_e2/estimacion_costo_local.json (local) y estimacion_computo.json del piloto en L4"\\n  }\\n}\\n\', \'config/arquitecturas_compactas_e2.csv\': \'arq,w1,w2,w3,b1,b2,b3,fcn_parametros,fcn_macs,unet_parametros,unet_macs,segnet_parametros,segnet_macs\\na0000,16,16,16,0,0,0,12065,273743872,25953,840957952,21617,653262848\\na0001,16,16,16,0,0,1,12065,273743872,25953,840957952,21617,653262848\\na0002,16,16,16,0,1,0,12065,273743872,25953,840957952,21617,653262848\\na0003,16,16,16,0,1,1,12065,273743872,25953,840957952,21617,653262848\\na0004,16,16,16,1,0,0,12065,273743872,25953,840957952,21617,653262848\\na0005,16,16,16,1,0,1,12065,273743872,25953,840957952,21617,653262848\\na0006,16,16,16,1,1,0,12065,273743872,25953,840957952,21617,653262848\\na0007,16,16,16,1,1,1,12065,273743872,25953,840957952,21617,653262848\\na0008,16,16,32,0,0,0,21329,311558144,37505,916455424,31121,692060160\\na0009,16,16,32,0,0,1,21329,311558144,37505,916455424,31121,692060160\\na0010,16,16,32,0,1,0,21329,311558144,37505,916455424,31121,692060160\\na0011,16,16,32,0,1,1,21329,311558144,37505,916455424,31121,692060160\\na0012,16,16,32,1,0,0,21329,311558144,37505,916455424,31121,692060160\\na0013,16,16,32,1,0,1,21329,311558144,37505,916455424,31121,692060160\\na0014,16,16,32,1,1,0,21329,311558144,37505,916455424,31121,692060160\\na0015,16,16,32,1,1,1,21329,311558144,37505,916455424,31121,692060160\\na0016,16,16,64,0,0,0,53681,443809792,74433,1124073472,63953,826277888\\na0017,16,16,64,0,0,1,53681,443809792,74433,1124073472,63953,826277888\\na0018,16,16,64,0,1,0,53681,443809792,74433,1124073472,63953,826277888\\na0019,16,16,64,0,1,1,53681,443809792,74433,1124073472,63953,826277888\\na0020,16,16,64,1,0,0,53681,443809792,74433,1124073472,63953,826277888\\na0021,16,16,64,1,0,1,53681,443809792,74433,1124073472,63953,826277888\\na0022,16,16,64,1,1,0,53681,443809792,74433,1124073472,63953,826277888\\na0023,16,16,64,1,1,1,53681,443809792,74433,1124073472,63953,826277888\\na0024,16,16,128,0,0,0,173681,934805504,203585,1765801984,184913,1321205760\\na0025,16,16,128,0,0,1,173681,934805504,203585,1765801984,184913,1321205760\\na0026,16,16,128,0,1,0,173681,934805504,203585,1765801984,184913,1321205760\\na0027,16,16,128,0,1,1,173681,934805504,203585,1765801984,184913,1321205760\\na0028,16,16,128,1,0,0,173681,934805504,203585,1765801984,184913,1321205760\\na0029,16,16,128,1,0,1,173681,934805504,203585,1765801984,184913,1321205760\\na0030,16,16,128,1,1,0,173681,934805504,203585,1765801984,184913,1321205760\\na0031,16,16,128,1,1,1,173681,934805504,203585,1765801984,184913,1321205760\\na0032,16,16,256,0,0,0,634865,2822766592,683073,3955228672,648017,3217031168\\na0033,16,16,256,0,0,1,634865,2822766592,683073,3955228672,648017,3217031168\\na0034,16,16,256,0,1,0,634865,2822766592,683073,3955228672,648017,3217031168\\na0035,16,16,256,0,1,1,634865,2822766592,683073,3955228672,648017,3217031168\\na0036,16,16,256,1,0,0,634865,2822766592,683073,3955228672,648017,3217031168\\na0037,16,16,256,1,0,1,634865,2822766592,683073,3955228672,648017,3217031168\\na0038,16,16,256,1,1,0,634865,2822766592,683073,3955228672,648017,3217031168\\na0039,16,16,256,1,1,1,634865,2822766592,683073,3955228672,648017,3217031168\\na0040,16,32,16,0,0,0,23617,434176000,55969,1416626176,35745,852492288\\na0041,16,32,16,0,0,1,23617,434176000,55969,1416626176,35745,852492288\\na0042,16,32,16,0,1,0,23617,434176000,55969,1416626176,35745,852492288\\na0043,16,32,16,0,1,1,23617,434176000,55969,1416626176,35745,852492288\\na0044,16,32,16,1,0,0,23617,434176000,55969,1416626176,35745,852492288\\na0045,16,32,16,1,0,1,23617,434176000,55969,1416626176,35745,852492288\\na0046,16,32,16,1,1,0,23617,434176000,55969,1416626176,35745,852492288\\na0047,16,32,16,1,1,1,23617,434176000,55969,1416626176,35745,852492288\\na0048,16,32,32,0,0,0,35185,481427456,72129,1539309568,47809,901775360\\na0049,16,32,32,0,0,1,35185,481427456,72129,1539309568,47809,901775360\\na0050,16,32,32,0,1,0,35185,481427456,72129,1539309568,47809,901775360\\na0051,16,32,32,0,1,1,35185,481427456,72129,1539309568,47809,901775360\\na0052,16,32,32,1,0,0,35185,481427456,72129,1539309568,47809,901775360\\na0053,16,32,32,1,0,1,35185,481427456,72129,1539309568,47809,901775360\\na0054,16,32,32,1,1,0,35185,481427456,72129,1539309568,47809,901775360\\na0055,16,32,32,1,1,1,35185,481427456,72129,1539309568,47809,901775360\\na0056,16,32,64,0,0,0,72145,632553472,118273,1841299456,85761,1056964608\\na0057,16,32,64,0,0,1,72145,632553472,118273,1841299456,85761,1056964608\\na0058,16,32,64,0,1,0,72145,632553472,118273,1841299456,85761,1056964608\\na0059,16,32,64,0,1,1,72145,632553472,118273,1841299456,85761,1056964608\\na0060,16,32,64,1,0,0,72145,632553472,118273,1841299456,85761,1056964608\\na0061,16,32,64,1,0,1,72145,632553472,118273,1841299456,85761,1056964608\\na0062,16,32,64,1,1,0,72145,632553472,118273,1841299456,85761,1056964608\\na0063,16,32,64,1,1,1,72145,632553472,118273,1841299456,85761,1056964608\\na0064,16,32,128,0,0,0,201361,1161297920,265857,2671771648,216961,1593835520\\na0065,16,32,128,0,0,1,201361,1161297920,265857,2671771648,216961,1593835520\\na0066,16,32,128,0,1,0,201361,1161297920,265857,2671771648,216961,1593835520\\na0067,16,32,128,0,1,1,201361,1161297920,265857,2671771648,216961,1593835520\\na0068,16,32,128,1,0,0,201361,1161297920,265857,2671771648,216961,1593835520\\na0069,16,32,128,1,0,1,201361,1161297920,265857,2671771648,216961,1593835520\\na0070,16,32,128,1,1,0,201361,1161297920,265857,2671771648,216961,1593835520\\na0071,16,32,128,1,1,1,201361,1161297920,265857,2671771648,216961,1593835520\\na0072,16,32,256,0,0,0,680977,3124756480,782209,5238685696,700545,3573547008\\na0073,16,32,256,0,0,1,680977,3124756480,782209,5238685696,700545,3573547008\\na0074,16,32,256,0,1,0,680977,3124756480,782209,5238685696,700545,3573547008\\na0075,16,32,256,0,1,1,680977,3124756480,782209,5238685696,700545,3573547008\\na0076,16,32,256,1,0,0,680977,3124756480,782209,5238685696,700545,3573547008\\na0077,16,32,256,1,0,1,680977,3124756480,782209,5238685696,700545,3573547008\\na0078,16,32,256,1,1,0,680977,3124756480,782209,5238685696,700545,3573547008\\na0079,16,32,256,1,1,1,680977,3124756480,782209,5238685696,700545,3573547008\\na0080,16,64,16,0,0,0,60545,981532672,157473,3247439872,77825,1477443584\\na0081,16,64,16,0,0,1,60545,981532672,157473,3247439872,77825,1477443584\\na0082,16,64,16,0,1,0,60545,981532672,157473,3247439872,77825,1477443584\\na0083,16,64,16,0,1,1,60545,981532672,157473,3247439872,77825,1477443584\\na0084,16,64,16,1,0,0,60545,981532672,157473,3247439872,77825,1477443584\\na0085,16,64,16,1,0,1,60545,981532672,157473,3247439872,77825,1477443584\\na0086,16,64,16,1,1,0,60545,981532672,157473,3247439872,77825,1477443584\\na0087,16,64,16,1,1,1,60545,981532672,157473,3247439872,77825,1477443584\\na0088,16,64,32,0,0,0,76721,1047658496,182849,3464495104,95009,1547698176\\na0089,16,64,32,0,0,1,76721,1047658496,182849,3464495104,95009,1547698176\\na0090,16,64,32,0,1,0,76721,1047658496,182849,3464495104,95009,1547698176\\na0091,16,64,32,0,1,1,76721,1047658496,182849,3464495104,95009,1547698176\\na0092,16,64,32,1,0,0,76721,1047658496,182849,3464495104,95009,1547698176\\na0093,16,64,32,1,0,1,76721,1047658496,182849,3464495104,95009,1547698176\\na0094,16,64,32,1,1,0,76721,1047658496,182849,3464495104,95009,1547698176\\na0095,16,64,32,1,1,1,76721,1047658496,182849,3464495104,95009,1547698176\\na0096,16,64,64,0,0,0,122897,1236533248,247425,3955228672,143201,1744830464\\na0097,16,64,64,0,0,1,122897,1236533248,247425,3955228672,143201,1744830464\\na0098,16,64,64,0,1,0,122897,1236533248,247425,3955228672,143201,1744830464\\na0099,16,64,64,0,1,1,122897,1236533248,247425,3955228672,143201,1744830464\\na0100,16,64,64,1,0,0,122897,1236533248,247425,3955228672,143201,1744830464\\na0101,16,64,64,1,0,1,122897,1236533248,247425,3955228672,143201,1744830464\\na0102,16,64,64,1,1,0,122897,1236533248,247425,3955228672,143201,1744830464\\na0103,16,64,64,1,1,1,122897,1236533248,247425,3955228672,143201,1744830464\\na0104,16,64,128,0,0,0,270545,1840775168,431873,5163188224,294881,2365587456\\na0105,16,64,128,0,0,1,270545,1840775168,431873,5163188224,294881,2365587456\\na0106,16,64,128,0,1,0,270545,1840775168,431873,5163188224,294881,2365587456\\na0107,16,64,128,0,1,1,270545,1840775168,431873,5163188224,294881,2365587456\\na0108,16,64,128,1,0,0,270545,1840775168,431873,5163188224,294881,2365587456\\na0109,16,64,128,1,0,1,270545,1840775168,431873,5163188224,294881,2365587456\\na0110,16,64,128,1,1,0,270545,1840775168,431873,5163188224,294881,2365587456\\na0111,16,64,128,1,1,1,270545,1840775168,431873,5163188224,294881,2365587456\\na0112,16,64,256,0,0,0,787025,3955228672,1021953,8485076992,819425,4513071104\\na0113,16,64,256,0,0,1,787025,3955228672,1021953,8485076992,819425,4513071104\\na0114,16,64,256,0,1,0,787025,3955228672,1021953,8485076992,819425,4513071104\\na0115,16,64,256,0,1,1,787025,3955228672,1021953,8485076992,819425,4513071104\\na0116,16,64,256,1,0,0,787025,3955228672,1021953,8485076992,819425,4513071104\\na0117,16,64,256,1,0,1,787025,3955228672,1021953,8485076992,819425,4513071104\\na0118,16,64,256,1,1,0,787025,3955228672,1021953,8485076992,819425,4513071104\\na0119,16,64,256,1,1,1,787025,3955228672,1021953,8485076992,819425,4513071104\\na0120,16,128,16,0,0,0,189697,2982215680,526369,9626976256,217281,3633315840\\na0121,16,128,16,0,0,1,189697,2982215680,526369,9626976256,217281,3633315840\\na0122,16,128,16,0,1,0,189697,2982215680,526369,9626976256,217281,3633315840\\na0123,16,128,16,0,1,1,189697,2982215680,526369,9626976256,217281,3633315840\\na0124,16,128,16,1,0,0,189697,2982215680,526369,9626976256,217281,3633315840\\na0125,16,128,16,1,0,1,189697,2982215680,526369,9626976256,217281,3633315840\\na0126,16,128,16,1,1,0,189697,2982215680,526369,9626976256,217281,3633315840\\na0127,16,128,16,1,1,1,189697,2982215680,526369,9626976256,217281,3633315840\\na0128,16,128,32,0,0,0,215089,3086090240,570177,10032775168,244705,3745513472\\na0129,16,128,32,0,0,1,215089,3086090240,570177,10032775168,244705,3745513472\\na0130,16,128,32,0,1,0,215089,3086090240,570177,10032775168,244705,3745513472\\na0131,16,128,32,0,1,1,215089,3086090240,570177,10032775168,244705,3745513472\\na0132,16,128,32,1,0,0,215089,3086090240,570177,10032775168,244705,3745513472\\na0133,16,128,32,1,0,1,215089,3086090240,570177,10032775168,244705,3745513472\\na0134,16,128,32,1,1,0,215089,3086090240,570177,10032775168,244705,3745513472\\na0135,16,128,32,1,1,1,215089,3086090240,570177,10032775168,244705,3745513472\\na0136,16,128,64,0,0,0,279697,3350462464,671617,10900996096,313377,4026531840\\na0137,16,128,64,0,0,1,279697,3350462464,671617,10900996096,313377,4026531840\\na0138,16,128,64,0,1,0,279697,3350462464,671617,10900996096,313377,4026531840\\na0139,16,128,64,0,1,1,279697,3350462464,671617,10900996096,313377,4026531840\\na0140,16,128,64,1,0,0,279697,3350462464,671617,10900996096,313377,4026531840\\na0141,16,128,64,1,0,1,279697,3350462464,671617,10900996096,313377,4026531840\\na0142,16,128,64,1,1,0,279697,3350462464,671617,10900996096,313377,4026531840\\na0143,16,128,64,1,1,1,279697,3350462464,671617,10900996096,313377,4026531840\\na0144,16,128,128,0,0,0,464209,4105699328,929793,12863930368,506017,4815060992\\na0145,16,128,128,0,0,1,464209,4105699328,929793,12863930368,506017,4815060992\\na0146,16,128,128,0,1,0,464209,4105699328,929793,12863930368,506017,4815060992\\na0147,16,128,128,0,1,1,464209,4105699328,929793,12863930368,506017,4815060992\\na0148,16,128,128,1,0,0,464209,4105699328,929793,12863930368,506017,4815060992\\na0149,16,128,128,1,0,1,464209,4105699328,929793,12863930368,506017,4815060992\\na0150,16,128,128,1,1,0,464209,4105699328,929793,12863930368,506017,4815060992\\na0151,16,128,128,1,1,1,464209,4105699328,929793,12863930368,506017,4815060992\\na0152,16,128,256,0,0,0,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0153,16,128,256,0,0,1,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0154,16,128,256,0,1,0,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0155,16,128,256,0,1,1,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0156,16,128,256,1,0,0,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0157,16,128,256,1,0,1,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0158,16,128,256,1,1,0,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0159,16,128,256,1,1,1,1054417,6522142720,1667329,17695768576,1112481,7298088960\\na0160,16,256,16,0,0,0,669185,10607460352,1927713,33257684992,717377,11568939008\\na0161,16,256,16,0,0,1,669185,10607460352,1927713,33257684992,717377,11568939008\\na0162,16,256,16,0,1,0,669185,10607460352,1927713,33257684992,717377,11568939008\\na0163,16,256,16,0,1,1,669185,10607460352,1927713,33257684992,717377,11568939008\\na0164,16,256,16,1,0,0,669185,10607460352,1927713,33257684992,717377,11568939008\\na0165,16,256,16,1,0,1,669185,10607460352,1927713,33257684992,717377,11568939008\\na0166,16,256,16,1,1,0,669185,10607460352,1927713,33257684992,717377,11568939008\\na0167,16,256,16,1,1,1,669185,10607460352,1927713,33257684992,717377,11568939008\\na0168,16,256,32,0,0,0,713009,10786832384,2008385,34040971264,765281,11765022720\\na0169,16,256,32,0,0,1,713009,10786832384,2008385,34040971264,765281,11765022720\\na0170,16,256,32,0,1,0,713009,10786832384,2008385,34040971264,765281,11765022720\\na0171,16,256,32,0,1,1,713009,10786832384,2008385,34040971264,765281,11765022720\\na0172,16,256,32,1,0,0,713009,10786832384,2008385,34040971264,765281,11765022720\\na0173,16,256,32,1,0,1,713009,10786832384,2008385,34040971264,765281,11765022720\\na0174,16,256,32,1,1,0,713009,10786832384,2008385,34040971264,765281,11765022720\\na0175,16,256,32,1,1,1,713009,10786832384,2008385,34040971264,765281,11765022720\\na0176,16,256,64,0,0,0,814481,11202199552,2183553,35664166912,874913,12213813248\\na0177,16,256,64,0,0,1,814481,11202199552,2183553,35664166912,874913,12213813248\\na0178,16,256,64,0,1,0,814481,11202199552,2183553,35664166912,874913,12213813248\\na0179,16,256,64,0,1,1,814481,11202199552,2183553,35664166912,874913,12213813248\\na0180,16,256,64,1,0,0,814481,11202199552,2183553,35664166912,874913,12213813248\\na0181,16,256,64,1,0,1,814481,11202199552,2183553,35664166912,874913,12213813248\\na0182,16,256,64,1,1,0,814481,11202199552,2183553,35664166912,874913,12213813248\\na0183,16,256,64,1,1,1,814481,11202199552,2183553,35664166912,874913,12213813248\\na0184,16,256,128,0,0,0,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0185,16,256,128,0,0,1,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0186,16,256,128,0,1,0,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0187,16,256,128,0,1,1,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0188,16,256,128,1,0,0,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0189,16,256,128,1,0,1,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0190,16,256,128,1,1,0,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0191,16,256,128,1,1,1,1072721,12259426304,2589185,39137050624,1149473,13337886720\\na0192,16,256,256,0,0,0,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0193,16,256,256,0,0,1,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0194,16,256,256,0,1,0,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0195,16,256,256,0,1,1,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0196,16,256,256,1,0,0,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0197,16,256,256,1,0,1,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0198,16,256,256,1,1,0,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0199,16,256,256,1,1,1,1810385,15279849472,3621633,46988787712,1919777,16492003328\\na0200,32,16,16,0,0,0,21745,792788992,51809,2418016256,54417,2230321152\\na0201,32,16,16,0,0,1,21745,792788992,51809,2418016256,54417,2230321152\\na0202,32,16,16,0,1,0,21745,792788992,51809,2418016256,54417,2230321152\\na0203,32,16,16,0,1,1,21745,792788992,51809,2418016256,54417,2230321152\\na0204,32,16,16,1,0,0,21745,792788992,51809,2418016256,54417,2230321152\\na0205,32,16,16,1,0,1,21745,792788992,51809,2418016256,54417,2230321152\\na0206,32,16,16,1,1,0,21745,792788992,51809,2418016256,54417,2230321152\\na0207,32,16,16,1,1,1,21745,792788992,51809,2418016256,54417,2230321152\\na0208,32,16,32,0,0,0,31009,830603264,63361,2493513728,63921,2269118464\\na0209,32,16,32,0,0,1,31009,830603264,63361,2493513728,63921,2269118464\\na0210,32,16,32,0,1,0,31009,830603264,63361,2493513728,63921,2269118464\\na0211,32,16,32,0,1,1,31009,830603264,63361,2493513728,63921,2269118464\\na0212,32,16,32,1,0,0,31009,830603264,63361,2493513728,63921,2269118464\\na0213,32,16,32,1,0,1,31009,830603264,63361,2493513728,63921,2269118464\\na0214,32,16,32,1,1,0,31009,830603264,63361,2493513728,63921,2269118464\\na0215,32,16,32,1,1,1,31009,830603264,63361,2493513728,63921,2269118464\\na0216,32,16,64,0,0,0,63361,962854912,100289,2701131776,96753,2403336192\\na0217,32,16,64,0,0,1,63361,962854912,100289,2701131776,96753,2403336192\\na0218,32,16,64,0,1,0,63361,962854912,100289,2701131776,96753,2403336192\\na0219,32,16,64,0,1,1,63361,962854912,100289,2701131776,96753,2403336192\\na0220,32,16,64,1,0,0,63361,962854912,100289,2701131776,96753,2403336192\\na0221,32,16,64,1,0,1,63361,962854912,100289,2701131776,96753,2403336192\\na0222,32,16,64,1,1,0,63361,962854912,100289,2701131776,96753,2403336192\\na0223,32,16,64,1,1,1,63361,962854912,100289,2701131776,96753,2403336192\\na0224,32,16,128,0,0,0,183361,1453850624,229441,3342860288,217713,2898264064\\na0225,32,16,128,0,0,1,183361,1453850624,229441,3342860288,217713,2898264064\\na0226,32,16,128,0,1,0,183361,1453850624,229441,3342860288,217713,2898264064\\na0227,32,16,128,0,1,1,183361,1453850624,229441,3342860288,217713,2898264064\\na0228,32,16,128,1,0,0,183361,1453850624,229441,3342860288,217713,2898264064\\na0229,32,16,128,1,0,1,183361,1453850624,229441,3342860288,217713,2898264064\\na0230,32,16,128,1,1,0,183361,1453850624,229441,3342860288,217713,2898264064\\na0231,32,16,128,1,1,1,183361,1453850624,229441,3342860288,217713,2898264064\\na0232,32,16,256,0,0,0,644545,3341811712,708929,5532286976,680817,4794089472\\na0233,32,16,256,0,0,1,644545,3341811712,708929,5532286976,680817,4794089472\\na0234,32,16,256,0,1,0,644545,3341811712,708929,5532286976,680817,4794089472\\na0235,32,16,256,0,1,1,644545,3341811712,708929,5532286976,680817,4794089472\\na0236,32,16,256,1,0,0,644545,3341811712,708929,5532286976,680817,4794089472\\na0237,32,16,256,1,0,1,644545,3341811712,708929,5532286976,680817,4794089472\\na0238,32,16,256,1,1,0,644545,3341811712,708929,5532286976,680817,4794089472\\na0239,32,16,256,1,1,1,644545,3341811712,708929,5532286976,680817,4794089472\\na0240,32,32,16,0,0,0,35601,990969856,86433,3182428160,73153,2505048064\\na0241,32,32,16,0,0,1,35601,990969856,86433,3182428160,73153,2505048064\\na0242,32,32,16,0,1,0,35601,990969856,86433,3182428160,73153,2505048064\\na0243,32,32,16,0,1,1,35601,990969856,86433,3182428160,73153,2505048064\\na0244,32,32,16,1,0,0,35601,990969856,86433,3182428160,73153,2505048064\\na0245,32,32,16,1,0,1,35601,990969856,86433,3182428160,73153,2505048064\\na0246,32,32,16,1,1,0,35601,990969856,86433,3182428160,73153,2505048064\\na0247,32,32,16,1,1,1,35601,990969856,86433,3182428160,73153,2505048064\\na0248,32,32,32,0,0,0,47169,1038221312,102593,3305111552,85217,2554331136\\na0249,32,32,32,0,0,1,47169,1038221312,102593,3305111552,85217,2554331136\\na0250,32,32,32,0,1,0,47169,1038221312,102593,3305111552,85217,2554331136\\na0251,32,32,32,0,1,1,47169,1038221312,102593,3305111552,85217,2554331136\\na0252,32,32,32,1,0,0,47169,1038221312,102593,3305111552,85217,2554331136\\na0253,32,32,32,1,0,1,47169,1038221312,102593,3305111552,85217,2554331136\\na0254,32,32,32,1,1,0,47169,1038221312,102593,3305111552,85217,2554331136\\na0255,32,32,32,1,1,1,47169,1038221312,102593,3305111552,85217,2554331136\\na0256,32,32,64,0,0,0,84129,1189347328,148737,3607101440,123169,2709520384\\na0257,32,32,64,0,0,1,84129,1189347328,148737,3607101440,123169,2709520384\\na0258,32,32,64,0,1,0,84129,1189347328,148737,3607101440,123169,2709520384\\na0259,32,32,64,0,1,1,84129,1189347328,148737,3607101440,123169,2709520384\\na0260,32,32,64,1,0,0,84129,1189347328,148737,3607101440,123169,2709520384\\na0261,32,32,64,1,0,1,84129,1189347328,148737,3607101440,123169,2709520384\\na0262,32,32,64,1,1,0,84129,1189347328,148737,3607101440,123169,2709520384\\na0263,32,32,64,1,1,1,84129,1189347328,148737,3607101440,123169,2709520384\\na0264,32,32,128,0,0,0,213345,1718091776,296321,4437573632,254369,3246391296\\na0265,32,32,128,0,0,1,213345,1718091776,296321,4437573632,254369,3246391296\\na0266,32,32,128,0,1,0,213345,1718091776,296321,4437573632,254369,3246391296\\na0267,32,32,128,0,1,1,213345,1718091776,296321,4437573632,254369,3246391296\\na0268,32,32,128,1,0,0,213345,1718091776,296321,4437573632,254369,3246391296\\na0269,32,32,128,1,0,1,213345,1718091776,296321,4437573632,254369,3246391296\\na0270,32,32,128,1,1,0,213345,1718091776,296321,4437573632,254369,3246391296\\na0271,32,32,128,1,1,1,213345,1718091776,296321,4437573632,254369,3246391296\\na0272,32,32,256,0,0,0,692961,3681550336,812673,7004487680,737953,5226102784\\na0273,32,32,256,0,0,1,692961,3681550336,812673,7004487680,737953,5226102784\\na0274,32,32,256,0,1,0,692961,3681550336,812673,7004487680,737953,5226102784\\na0275,32,32,256,0,1,1,692961,3681550336,812673,7004487680,737953,5226102784\\na0276,32,32,256,1,0,0,692961,3681550336,812673,7004487680,737953,5226102784\\na0277,32,32,256,1,0,1,692961,3681550336,812673,7004487680,737953,5226102784\\na0278,32,32,256,1,1,0,692961,3681550336,812673,7004487680,737953,5226102784\\na0279,32,32,256,1,1,1,692961,3681550336,812673,7004487680,737953,5226102784\\na0280,32,64,16,0,0,0,77137,1613824000,197153,5390729216,124449,3280994304\\na0281,32,64,16,0,0,1,77137,1613824000,197153,5390729216,124449,3280994304\\na0282,32,64,16,0,1,0,77137,1613824000,197153,5390729216,124449,3280994304\\na0283,32,64,16,0,1,1,77137,1613824000,197153,5390729216,124449,3280994304\\na0284,32,64,16,1,0,0,77137,1613824000,197153,5390729216,124449,3280994304\\na0285,32,64,16,1,0,1,77137,1613824000,197153,5390729216,124449,3280994304\\na0286,32,64,16,1,1,0,77137,1613824000,197153,5390729216,124449,3280994304\\na0287,32,64,16,1,1,1,77137,1613824000,197153,5390729216,124449,3280994304\\na0288,32,64,32,0,0,0,93313,1679949824,222529,5607784448,141633,3351248896\\na0289,32,64,32,0,0,1,93313,1679949824,222529,5607784448,141633,3351248896\\na0290,32,64,32,0,1,0,93313,1679949824,222529,5607784448,141633,3351248896\\na0291,32,64,32,0,1,1,93313,1679949824,222529,5607784448,141633,3351248896\\na0292,32,64,32,1,0,0,93313,1679949824,222529,5607784448,141633,3351248896\\na0293,32,64,32,1,0,1,93313,1679949824,222529,5607784448,141633,3351248896\\na0294,32,64,32,1,1,0,93313,1679949824,222529,5607784448,141633,3351248896\\na0295,32,64,32,1,1,1,93313,1679949824,222529,5607784448,141633,3351248896\\na0296,32,64,64,0,0,0,139489,1868824576,287105,6098518016,189825,3548381184\\na0297,32,64,64,0,0,1,139489,1868824576,287105,6098518016,189825,3548381184\\na0298,32,64,64,0,1,0,139489,1868824576,287105,6098518016,189825,3548381184\\na0299,32,64,64,0,1,1,139489,1868824576,287105,6098518016,189825,3548381184\\na0300,32,64,64,1,0,0,139489,1868824576,287105,6098518016,189825,3548381184\\na0301,32,64,64,1,0,1,139489,1868824576,287105,6098518016,189825,3548381184\\na0302,32,64,64,1,1,0,139489,1868824576,287105,6098518016,189825,3548381184\\na0303,32,64,64,1,1,1,139489,1868824576,287105,6098518016,189825,3548381184\\na0304,32,64,128,0,0,0,287137,2473066496,471553,7306477568,341505,4169138176\\na0305,32,64,128,0,0,1,287137,2473066496,471553,7306477568,341505,4169138176\\na0306,32,64,128,0,1,0,287137,2473066496,471553,7306477568,341505,4169138176\\na0307,32,64,128,0,1,1,287137,2473066496,471553,7306477568,341505,4169138176\\na0308,32,64,128,1,0,0,287137,2473066496,471553,7306477568,341505,4169138176\\na0309,32,64,128,1,0,1,287137,2473066496,471553,7306477568,341505,4169138176\\na0310,32,64,128,1,1,0,287137,2473066496,471553,7306477568,341505,4169138176\\na0311,32,64,128,1,1,1,287137,2473066496,471553,7306477568,341505,4169138176\\na0312,32,64,256,0,0,0,803617,4587520000,1061633,10628366336,866049,6316621824\\na0313,32,64,256,0,0,1,803617,4587520000,1061633,10628366336,866049,6316621824\\na0314,32,64,256,0,1,0,803617,4587520000,1061633,10628366336,866049,6316621824\\na0315,32,64,256,0,1,1,803617,4587520000,1061633,10628366336,866049,6316621824\\na0316,32,64,256,1,0,0,803617,4587520000,1061633,10628366336,866049,6316621824\\na0317,32,64,256,1,0,1,803617,4587520000,1061633,10628366336,866049,6316621824\\na0318,32,64,256,1,1,0,803617,4587520000,1061633,10628366336,866049,6316621824\\na0319,32,64,256,1,1,1,803617,4587520000,1061633,10628366336,866049,6316621824\\na0320,32,128,16,0,0,0,215505,3765501952,584481,12525240320,282337,5738856448\\na0321,32,128,16,0,0,1,215505,3765501952,584481,12525240320,282337,5738856448\\na0322,32,128,16,0,1,0,215505,3765501952,584481,12525240320,282337,5738856448\\na0323,32,128,16,0,1,1,215505,3765501952,584481,12525240320,282337,5738856448\\na0324,32,128,16,1,0,0,215505,3765501952,584481,12525240320,282337,5738856448\\na0325,32,128,16,1,0,1,215505,3765501952,584481,12525240320,282337,5738856448\\na0326,32,128,16,1,1,0,215505,3765501952,584481,12525240320,282337,5738856448\\na0327,32,128,16,1,1,1,215505,3765501952,584481,12525240320,282337,5738856448\\na0328,32,128,32,0,0,0,240897,3869376512,628289,12931039232,309761,5851054080\\na0329,32,128,32,0,0,1,240897,3869376512,628289,12931039232,309761,5851054080\\na0330,32,128,32,0,1,0,240897,3869376512,628289,12931039232,309761,5851054080\\na0331,32,128,32,0,1,1,240897,3869376512,628289,12931039232,309761,5851054080\\na0332,32,128,32,1,0,0,240897,3869376512,628289,12931039232,309761,5851054080\\na0333,32,128,32,1,0,1,240897,3869376512,628289,12931039232,309761,5851054080\\na0334,32,128,32,1,1,0,240897,3869376512,628289,12931039232,309761,5851054080\\na0335,32,128,32,1,1,1,240897,3869376512,628289,12931039232,309761,5851054080\\na0336,32,128,64,0,0,0,305505,4133748736,729729,13799260160,378433,6132072448\\na0337,32,128,64,0,0,1,305505,4133748736,729729,13799260160,378433,6132072448\\na0338,32,128,64,0,1,0,305505,4133748736,729729,13799260160,378433,6132072448\\na0339,32,128,64,0,1,1,305505,4133748736,729729,13799260160,378433,6132072448\\na0340,32,128,64,1,0,0,305505,4133748736,729729,13799260160,378433,6132072448\\na0341,32,128,64,1,0,1,305505,4133748736,729729,13799260160,378433,6132072448\\na0342,32,128,64,1,1,0,305505,4133748736,729729,13799260160,378433,6132072448\\na0343,32,128,64,1,1,1,305505,4133748736,729729,13799260160,378433,6132072448\\na0344,32,128,128,0,0,0,490017,4888985600,987905,15762194432,571073,6920601600\\na0345,32,128,128,0,0,1,490017,4888985600,987905,15762194432,571073,6920601600\\na0346,32,128,128,0,1,0,490017,4888985600,987905,15762194432,571073,6920601600\\na0347,32,128,128,0,1,1,490017,4888985600,987905,15762194432,571073,6920601600\\na0348,32,128,128,1,0,0,490017,4888985600,987905,15762194432,571073,6920601600\\na0349,32,128,128,1,0,1,490017,4888985600,987905,15762194432,571073,6920601600\\na0350,32,128,128,1,1,0,490017,4888985600,987905,15762194432,571073,6920601600\\na0351,32,128,128,1,1,1,490017,4888985600,987905,15762194432,571073,6920601600\\na0352,32,128,256,0,0,0,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0353,32,128,256,0,0,1,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0354,32,128,256,0,1,0,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0355,32,128,256,0,1,1,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0356,32,128,256,1,0,0,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0357,32,128,256,1,0,1,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0358,32,128,256,1,1,0,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0359,32,128,256,1,1,1,1080225,7305428992,1725441,20594032640,1177537,9403629568\\na0360,32,256,16,0,0,0,713425,11692736512,2022689,37665898496,819297,14278459392\\na0361,32,256,16,0,0,1,713425,11692736512,2022689,37665898496,819297,14278459392\\na0362,32,256,16,0,1,0,713425,11692736512,2022689,37665898496,819297,14278459392\\na0363,32,256,16,0,1,1,713425,11692736512,2022689,37665898496,819297,14278459392\\na0364,32,256,16,1,0,0,713425,11692736512,2022689,37665898496,819297,14278459392\\na0365,32,256,16,1,0,1,713425,11692736512,2022689,37665898496,819297,14278459392\\na0366,32,256,16,1,1,0,713425,11692736512,2022689,37665898496,819297,14278459392\\na0367,32,256,16,1,1,1,713425,11692736512,2022689,37665898496,819297,14278459392\\na0368,32,256,32,0,0,0,757249,11872108544,2103361,38449184768,867201,14474543104\\na0369,32,256,32,0,0,1,757249,11872108544,2103361,38449184768,867201,14474543104\\na0370,32,256,32,0,1,0,757249,11872108544,2103361,38449184768,867201,14474543104\\na0371,32,256,32,0,1,1,757249,11872108544,2103361,38449184768,867201,14474543104\\na0372,32,256,32,1,0,0,757249,11872108544,2103361,38449184768,867201,14474543104\\na0373,32,256,32,1,0,1,757249,11872108544,2103361,38449184768,867201,14474543104\\na0374,32,256,32,1,1,0,757249,11872108544,2103361,38449184768,867201,14474543104\\na0375,32,256,32,1,1,1,757249,11872108544,2103361,38449184768,867201,14474543104\\na0376,32,256,64,0,0,0,858721,12287475712,2278529,40072380416,976833,14923333632\\na0377,32,256,64,0,0,1,858721,12287475712,2278529,40072380416,976833,14923333632\\na0378,32,256,64,0,1,0,858721,12287475712,2278529,40072380416,976833,14923333632\\na0379,32,256,64,0,1,1,858721,12287475712,2278529,40072380416,976833,14923333632\\na0380,32,256,64,1,0,0,858721,12287475712,2278529,40072380416,976833,14923333632\\na0381,32,256,64,1,0,1,858721,12287475712,2278529,40072380416,976833,14923333632\\na0382,32,256,64,1,1,0,858721,12287475712,2278529,40072380416,976833,14923333632\\na0383,32,256,64,1,1,1,858721,12287475712,2278529,40072380416,976833,14923333632\\na0384,32,256,128,0,0,0,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0385,32,256,128,0,0,1,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0386,32,256,128,0,1,0,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0387,32,256,128,0,1,1,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0388,32,256,128,1,0,0,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0389,32,256,128,1,0,1,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0390,32,256,128,1,1,0,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0391,32,256,128,1,1,1,1116961,13344702464,2684161,43545264128,1251393,16047407104\\na0392,32,256,256,0,0,0,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0393,32,256,256,0,0,1,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0394,32,256,256,0,1,0,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0395,32,256,256,0,1,1,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0396,32,256,256,1,0,0,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0397,32,256,256,1,0,1,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0398,32,256,256,1,1,0,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0399,32,256,256,1,1,1,1854625,16365125632,3716609,51397001216,2021697,19201523712\\na0400,64,16,16,0,0,0,54929,2736848896,144993,8290041856,175313,8328839168\\na0401,64,16,16,0,0,1,54929,2736848896,144993,8290041856,175313,8328839168\\na0402,64,16,16,0,1,0,54929,2736848896,144993,8290041856,175313,8328839168\\na0403,64,16,16,0,1,1,54929,2736848896,144993,8290041856,175313,8328839168\\na0404,64,16,16,1,0,0,54929,2736848896,144993,8290041856,175313,8328839168\\na0405,64,16,16,1,0,1,54929,2736848896,144993,8290041856,175313,8328839168\\na0406,64,16,16,1,1,0,54929,2736848896,144993,8290041856,175313,8328839168\\na0407,64,16,16,1,1,1,54929,2736848896,144993,8290041856,175313,8328839168\\na0408,64,16,32,0,0,0,64193,2774663168,156545,8365539328,184817,8367636480\\na0409,64,16,32,0,0,1,64193,2774663168,156545,8365539328,184817,8367636480\\na0410,64,16,32,0,1,0,64193,2774663168,156545,8365539328,184817,8367636480\\na0411,64,16,32,0,1,1,64193,2774663168,156545,8365539328,184817,8367636480\\na0412,64,16,32,1,0,0,64193,2774663168,156545,8365539328,184817,8367636480\\na0413,64,16,32,1,0,1,64193,2774663168,156545,8365539328,184817,8367636480\\na0414,64,16,32,1,1,0,64193,2774663168,156545,8365539328,184817,8367636480\\na0415,64,16,32,1,1,1,64193,2774663168,156545,8365539328,184817,8367636480\\na0416,64,16,64,0,0,0,96545,2906914816,193473,8573157376,217649,8501854208\\na0417,64,16,64,0,0,1,96545,2906914816,193473,8573157376,217649,8501854208\\na0418,64,16,64,0,1,0,96545,2906914816,193473,8573157376,217649,8501854208\\na0419,64,16,64,0,1,1,96545,2906914816,193473,8573157376,217649,8501854208\\na0420,64,16,64,1,0,0,96545,2906914816,193473,8573157376,217649,8501854208\\na0421,64,16,64,1,0,1,96545,2906914816,193473,8573157376,217649,8501854208\\na0422,64,16,64,1,1,0,96545,2906914816,193473,8573157376,217649,8501854208\\na0423,64,16,64,1,1,1,96545,2906914816,193473,8573157376,217649,8501854208\\na0424,64,16,128,0,0,0,216545,3397910528,322625,9214885888,338609,8996782080\\na0425,64,16,128,0,0,1,216545,3397910528,322625,9214885888,338609,8996782080\\na0426,64,16,128,0,1,0,216545,3397910528,322625,9214885888,338609,8996782080\\na0427,64,16,128,0,1,1,216545,3397910528,322625,9214885888,338609,8996782080\\na0428,64,16,128,1,0,0,216545,3397910528,322625,9214885888,338609,8996782080\\na0429,64,16,128,1,0,1,216545,3397910528,322625,9214885888,338609,8996782080\\na0430,64,16,128,1,1,0,216545,3397910528,322625,9214885888,338609,8996782080\\na0431,64,16,128,1,1,1,216545,3397910528,322625,9214885888,338609,8996782080\\na0432,64,16,256,0,0,0,677729,5285871616,802113,11404312576,801713,10892607488\\na0433,64,16,256,0,0,1,677729,5285871616,802113,11404312576,801713,10892607488\\na0434,64,16,256,0,1,0,677729,5285871616,802113,11404312576,801713,10892607488\\na0435,64,16,256,0,1,1,677729,5285871616,802113,11404312576,801713,10892607488\\na0436,64,16,256,1,0,0,677729,5285871616,802113,11404312576,801713,10892607488\\na0437,64,16,256,1,0,1,677729,5285871616,802113,11404312576,801713,10892607488\\na0438,64,16,256,1,1,0,677729,5285871616,802113,11404312576,801713,10892607488\\na0439,64,16,256,1,1,1,677729,5285871616,802113,11404312576,801713,10892607488\\na0440,64,32,16,0,0,0,73393,3010527232,188833,9431941120,203265,8754561024\\na0441,64,32,16,0,0,1,73393,3010527232,188833,9431941120,203265,8754561024\\na0442,64,32,16,0,1,0,73393,3010527232,188833,9431941120,203265,8754561024\\na0443,64,32,16,0,1,1,73393,3010527232,188833,9431941120,203265,8754561024\\na0444,64,32,16,1,0,0,73393,3010527232,188833,9431941120,203265,8754561024\\na0445,64,32,16,1,0,1,73393,3010527232,188833,9431941120,203265,8754561024\\na0446,64,32,16,1,1,0,73393,3010527232,188833,9431941120,203265,8754561024\\na0447,64,32,16,1,1,1,73393,3010527232,188833,9431941120,203265,8754561024\\na0448,64,32,32,0,0,0,84961,3057778688,204993,9554624512,215329,8803844096\\na0449,64,32,32,0,0,1,84961,3057778688,204993,9554624512,215329,8803844096\\na0450,64,32,32,0,1,0,84961,3057778688,204993,9554624512,215329,8803844096\\na0451,64,32,32,0,1,1,84961,3057778688,204993,9554624512,215329,8803844096\\na0452,64,32,32,1,0,0,84961,3057778688,204993,9554624512,215329,8803844096\\na0453,64,32,32,1,0,1,84961,3057778688,204993,9554624512,215329,8803844096\\na0454,64,32,32,1,1,0,84961,3057778688,204993,9554624512,215329,8803844096\\na0455,64,32,32,1,1,1,84961,3057778688,204993,9554624512,215329,8803844096\\na0456,64,32,64,0,0,0,121921,3208904704,251137,9856614400,253281,8959033344\\na0457,64,32,64,0,0,1,121921,3208904704,251137,9856614400,253281,8959033344\\na0458,64,32,64,0,1,0,121921,3208904704,251137,9856614400,253281,8959033344\\na0459,64,32,64,0,1,1,121921,3208904704,251137,9856614400,253281,8959033344\\na0460,64,32,64,1,0,0,121921,3208904704,251137,9856614400,253281,8959033344\\na0461,64,32,64,1,0,1,121921,3208904704,251137,9856614400,253281,8959033344\\na0462,64,32,64,1,1,0,121921,3208904704,251137,9856614400,253281,8959033344\\na0463,64,32,64,1,1,1,121921,3208904704,251137,9856614400,253281,8959033344\\na0464,64,32,128,0,0,0,251137,3737649152,398721,10687086592,384481,9495904256\\na0465,64,32,128,0,0,1,251137,3737649152,398721,10687086592,384481,9495904256\\na0466,64,32,128,0,1,0,251137,3737649152,398721,10687086592,384481,9495904256\\na0467,64,32,128,0,1,1,251137,3737649152,398721,10687086592,384481,9495904256\\na0468,64,32,128,1,0,0,251137,3737649152,398721,10687086592,384481,9495904256\\na0469,64,32,128,1,0,1,251137,3737649152,398721,10687086592,384481,9495904256\\na0470,64,32,128,1,1,0,251137,3737649152,398721,10687086592,384481,9495904256\\na0471,64,32,128,1,1,1,251137,3737649152,398721,10687086592,384481,9495904256\\na0472,64,32,256,0,0,0,730753,5701107712,915073,13254000640,868065,11475615744\\na0473,64,32,256,0,0,1,730753,5701107712,915073,13254000640,868065,11475615744\\na0474,64,32,256,0,1,0,730753,5701107712,915073,13254000640,868065,11475615744\\na0475,64,32,256,0,1,1,730753,5701107712,915073,13254000640,868065,11475615744\\na0476,64,32,256,1,0,0,730753,5701107712,915073,13254000640,868065,11475615744\\na0477,64,32,256,1,0,1,730753,5701107712,915073,13254000640,868065,11475615744\\na0478,64,32,256,1,1,0,730753,5701107712,915073,13254000640,868065,11475615744\\na0479,64,32,256,1,1,1,730753,5701107712,915073,13254000640,868065,11475615744\\na0480,64,64,16,0,0,0,124145,3784376320,317985,12395216896,272993,9832497152\\na0481,64,64,16,0,0,1,124145,3784376320,317985,12395216896,272993,9832497152\\na0482,64,64,16,0,1,0,124145,3784376320,317985,12395216896,272993,9832497152\\na0483,64,64,16,0,1,1,124145,3784376320,317985,12395216896,272993,9832497152\\na0484,64,64,16,1,0,0,124145,3784376320,317985,12395216896,272993,9832497152\\na0485,64,64,16,1,0,1,124145,3784376320,317985,12395216896,272993,9832497152\\na0486,64,64,16,1,1,0,124145,3784376320,317985,12395216896,272993,9832497152\\na0487,64,64,16,1,1,1,124145,3784376320,317985,12395216896,272993,9832497152\\na0488,64,64,32,0,0,0,140321,3850502144,343361,12612272128,290177,9902751744\\na0489,64,64,32,0,0,1,140321,3850502144,343361,12612272128,290177,9902751744\\na0490,64,64,32,0,1,0,140321,3850502144,343361,12612272128,290177,9902751744\\na0491,64,64,32,0,1,1,140321,3850502144,343361,12612272128,290177,9902751744\\na0492,64,64,32,1,0,0,140321,3850502144,343361,12612272128,290177,9902751744\\na0493,64,64,32,1,0,1,140321,3850502144,343361,12612272128,290177,9902751744\\na0494,64,64,32,1,1,0,140321,3850502144,343361,12612272128,290177,9902751744\\na0495,64,64,32,1,1,1,140321,3850502144,343361,12612272128,290177,9902751744\\na0496,64,64,64,0,0,0,186497,4039376896,407937,13103005696,338369,10099884032\\na0497,64,64,64,0,0,1,186497,4039376896,407937,13103005696,338369,10099884032\\na0498,64,64,64,0,1,0,186497,4039376896,407937,13103005696,338369,10099884032\\na0499,64,64,64,0,1,1,186497,4039376896,407937,13103005696,338369,10099884032\\na0500,64,64,64,1,0,0,186497,4039376896,407937,13103005696,338369,10099884032\\na0501,64,64,64,1,0,1,186497,4039376896,407937,13103005696,338369,10099884032\\na0502,64,64,64,1,1,0,186497,4039376896,407937,13103005696,338369,10099884032\\na0503,64,64,64,1,1,1,186497,4039376896,407937,13103005696,338369,10099884032\\na0504,64,64,128,0,0,0,334145,4643618816,592385,14310965248,490049,10720641024\\na0505,64,64,128,0,0,1,334145,4643618816,592385,14310965248,490049,10720641024\\na0506,64,64,128,0,1,0,334145,4643618816,592385,14310965248,490049,10720641024\\na0507,64,64,128,0,1,1,334145,4643618816,592385,14310965248,490049,10720641024\\na0508,64,64,128,1,0,0,334145,4643618816,592385,14310965248,490049,10720641024\\na0509,64,64,128,1,0,1,334145,4643618816,592385,14310965248,490049,10720641024\\na0510,64,64,128,1,1,0,334145,4643618816,592385,14310965248,490049,10720641024\\na0511,64,64,128,1,1,1,334145,4643618816,592385,14310965248,490049,10720641024\\na0512,64,64,256,0,0,0,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0513,64,64,256,0,0,1,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0514,64,64,256,0,1,0,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0515,64,64,256,0,1,1,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0516,64,64,256,1,0,0,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0517,64,64,256,1,0,1,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0518,64,64,256,1,1,0,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0519,64,64,256,1,1,1,850625,6758072320,1182465,17632854016,1014593,12868124672\\na0520,64,128,16,0,0,0,280945,6238044160,742177,21039677440,467745,12894339072\\na0521,64,128,16,0,0,1,280945,6238044160,742177,21039677440,467745,12894339072\\na0522,64,128,16,0,1,0,280945,6238044160,742177,21039677440,467745,12894339072\\na0523,64,128,16,0,1,1,280945,6238044160,742177,21039677440,467745,12894339072\\na0524,64,128,16,1,0,0,280945,6238044160,742177,21039677440,467745,12894339072\\na0525,64,128,16,1,0,1,280945,6238044160,742177,21039677440,467745,12894339072\\na0526,64,128,16,1,1,0,280945,6238044160,742177,21039677440,467745,12894339072\\na0527,64,128,16,1,1,1,280945,6238044160,742177,21039677440,467745,12894339072\\na0528,64,128,32,0,0,0,306337,6341918720,785985,21445476352,495169,13006536704\\na0529,64,128,32,0,0,1,306337,6341918720,785985,21445476352,495169,13006536704\\na0530,64,128,32,0,1,0,306337,6341918720,785985,21445476352,495169,13006536704\\na0531,64,128,32,0,1,1,306337,6341918720,785985,21445476352,495169,13006536704\\na0532,64,128,32,1,0,0,306337,6341918720,785985,21445476352,495169,13006536704\\na0533,64,128,32,1,0,1,306337,6341918720,785985,21445476352,495169,13006536704\\na0534,64,128,32,1,1,0,306337,6341918720,785985,21445476352,495169,13006536704\\na0535,64,128,32,1,1,1,306337,6341918720,785985,21445476352,495169,13006536704\\na0536,64,128,64,0,0,0,370945,6606290944,887425,22313697280,563841,13287555072\\na0537,64,128,64,0,0,1,370945,6606290944,887425,22313697280,563841,13287555072\\na0538,64,128,64,0,1,0,370945,6606290944,887425,22313697280,563841,13287555072\\na0539,64,128,64,0,1,1,370945,6606290944,887425,22313697280,563841,13287555072\\na0540,64,128,64,1,0,0,370945,6606290944,887425,22313697280,563841,13287555072\\na0541,64,128,64,1,0,1,370945,6606290944,887425,22313697280,563841,13287555072\\na0542,64,128,64,1,1,0,370945,6606290944,887425,22313697280,563841,13287555072\\na0543,64,128,64,1,1,1,370945,6606290944,887425,22313697280,563841,13287555072\\na0544,64,128,128,0,0,0,555457,7361527808,1145601,24276631552,756481,14076084224\\na0545,64,128,128,0,0,1,555457,7361527808,1145601,24276631552,756481,14076084224\\na0546,64,128,128,0,1,0,555457,7361527808,1145601,24276631552,756481,14076084224\\na0547,64,128,128,0,1,1,555457,7361527808,1145601,24276631552,756481,14076084224\\na0548,64,128,128,1,0,0,555457,7361527808,1145601,24276631552,756481,14076084224\\na0549,64,128,128,1,0,1,555457,7361527808,1145601,24276631552,756481,14076084224\\na0550,64,128,128,1,1,0,555457,7361527808,1145601,24276631552,756481,14076084224\\na0551,64,128,128,1,1,1,555457,7361527808,1145601,24276631552,756481,14076084224\\na0552,64,128,256,0,0,0,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0553,64,128,256,0,0,1,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0554,64,128,256,0,1,0,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0555,64,128,256,0,1,1,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0556,64,128,256,1,0,0,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0557,64,128,256,1,0,1,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0558,64,128,256,1,1,0,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0559,64,128,256,1,1,1,1145665,9777971200,1883137,29108469760,1362945,16559112192\\na0560,64,256,16,0,0,0,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0561,64,256,16,0,0,1,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0562,64,256,16,0,1,0,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0563,64,256,16,0,1,1,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0564,64,256,16,1,0,0,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0565,64,256,16,1,0,1,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0566,64,256,16,1,1,0,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0567,64,256,16,1,1,1,815729,14769258496,2254113,49200234496,1078433,22641901568\\na0568,64,256,32,0,0,0,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0569,64,256,32,0,0,1,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0570,64,256,32,0,1,0,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0571,64,256,32,0,1,1,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0572,64,256,32,1,0,0,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0573,64,256,32,1,0,1,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0574,64,256,32,1,1,0,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0575,64,256,32,1,1,1,859553,14948630528,2334785,49983520768,1126337,22837985280\\na0576,64,256,64,0,0,0,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0577,64,256,64,0,0,1,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0578,64,256,64,0,1,0,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0579,64,256,64,0,1,1,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0580,64,256,64,1,0,0,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0581,64,256,64,1,0,1,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0582,64,256,64,1,1,0,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0583,64,256,64,1,1,1,961025,15363997696,2509953,51606716416,1235969,23286775808\\na0584,64,256,128,0,0,0,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0585,64,256,128,0,0,1,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0586,64,256,128,0,1,0,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0587,64,256,128,0,1,1,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0588,64,256,128,1,0,0,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0589,64,256,128,1,0,1,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0590,64,256,128,1,1,0,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0591,64,256,128,1,1,1,1219265,16421224448,2915585,55079600128,1510529,24410849280\\na0592,64,256,256,0,0,0,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0593,64,256,256,0,0,1,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0594,64,256,256,0,1,0,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0595,64,256,256,0,1,1,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0596,64,256,256,1,0,0,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0597,64,256,256,1,0,1,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0598,64,256,256,1,1,0,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0599,64,256,256,1,1,1,1956929,19441647616,3948033,62931337216,2280833,27564965888\\na0600,128,16,16,0,0,0,176593,10248847360,497249,30905729024,638289,32303480832\\na0601,128,16,16,0,0,1,176593,10248847360,497249,30905729024,638289,32303480832\\na0602,128,16,16,0,1,0,176593,10248847360,497249,30905729024,638289,32303480832\\na0603,128,16,16,0,1,1,176593,10248847360,497249,30905729024,638289,32303480832\\na0604,128,16,16,1,0,0,176593,10248847360,497249,30905729024,638289,32303480832\\na0605,128,16,16,1,0,1,176593,10248847360,497249,30905729024,638289,32303480832\\na0606,128,16,16,1,1,0,176593,10248847360,497249,30905729024,638289,32303480832\\na0607,128,16,16,1,1,1,176593,10248847360,497249,30905729024,638289,32303480832\\na0608,128,16,32,0,0,0,185857,10286661632,508801,30981226496,647793,32342278144\\na0609,128,16,32,0,0,1,185857,10286661632,508801,30981226496,647793,32342278144\\na0610,128,16,32,0,1,0,185857,10286661632,508801,30981226496,647793,32342278144\\na0611,128,16,32,0,1,1,185857,10286661632,508801,30981226496,647793,32342278144\\na0612,128,16,32,1,0,0,185857,10286661632,508801,30981226496,647793,32342278144\\na0613,128,16,32,1,0,1,185857,10286661632,508801,30981226496,647793,32342278144\\na0614,128,16,32,1,1,0,185857,10286661632,508801,30981226496,647793,32342278144\\na0615,128,16,32,1,1,1,185857,10286661632,508801,30981226496,647793,32342278144\\na0616,128,16,64,0,0,0,218209,10418913280,545729,31188844544,680625,32476495872\\na0617,128,16,64,0,0,1,218209,10418913280,545729,31188844544,680625,32476495872\\na0618,128,16,64,0,1,0,218209,10418913280,545729,31188844544,680625,32476495872\\na0619,128,16,64,0,1,1,218209,10418913280,545729,31188844544,680625,32476495872\\na0620,128,16,64,1,0,0,218209,10418913280,545729,31188844544,680625,32476495872\\na0621,128,16,64,1,0,1,218209,10418913280,545729,31188844544,680625,32476495872\\na0622,128,16,64,1,1,0,218209,10418913280,545729,31188844544,680625,32476495872\\na0623,128,16,64,1,1,1,218209,10418913280,545729,31188844544,680625,32476495872\\na0624,128,16,128,0,0,0,338209,10909908992,674881,31830573056,801585,32971423744\\na0625,128,16,128,0,0,1,338209,10909908992,674881,31830573056,801585,32971423744\\na0626,128,16,128,0,1,0,338209,10909908992,674881,31830573056,801585,32971423744\\na0627,128,16,128,0,1,1,338209,10909908992,674881,31830573056,801585,32971423744\\na0628,128,16,128,1,0,0,338209,10909908992,674881,31830573056,801585,32971423744\\na0629,128,16,128,1,0,1,338209,10909908992,674881,31830573056,801585,32971423744\\na0630,128,16,128,1,1,0,338209,10909908992,674881,31830573056,801585,32971423744\\na0631,128,16,128,1,1,1,338209,10909908992,674881,31830573056,801585,32971423744\\na0632,128,16,256,0,0,0,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0633,128,16,256,0,0,1,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0634,128,16,256,0,1,0,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0635,128,16,256,0,1,1,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0636,128,16,256,1,0,0,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0637,128,16,256,1,0,1,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0638,128,16,256,1,1,0,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0639,128,16,256,1,1,1,799393,12797870080,1154369,34019999744,1264689,34867249152\\na0640,128,32,16,0,0,0,204273,10673520640,559521,32802603008,684673,33031192576\\na0641,128,32,16,0,0,1,204273,10673520640,559521,32802603008,684673,33031192576\\na0642,128,32,16,0,1,0,204273,10673520640,559521,32802603008,684673,33031192576\\na0643,128,32,16,0,1,1,204273,10673520640,559521,32802603008,684673,33031192576\\na0644,128,32,16,1,0,0,204273,10673520640,559521,32802603008,684673,33031192576\\na0645,128,32,16,1,0,1,204273,10673520640,559521,32802603008,684673,33031192576\\na0646,128,32,16,1,1,0,204273,10673520640,559521,32802603008,684673,33031192576\\na0647,128,32,16,1,1,1,204273,10673520640,559521,32802603008,684673,33031192576\\na0648,128,32,32,0,0,0,215841,10720772096,575681,32925286400,696737,33080475648\\na0649,128,32,32,0,0,1,215841,10720772096,575681,32925286400,696737,33080475648\\na0650,128,32,32,0,1,0,215841,10720772096,575681,32925286400,696737,33080475648\\na0651,128,32,32,0,1,1,215841,10720772096,575681,32925286400,696737,33080475648\\na0652,128,32,32,1,0,0,215841,10720772096,575681,32925286400,696737,33080475648\\na0653,128,32,32,1,0,1,215841,10720772096,575681,32925286400,696737,33080475648\\na0654,128,32,32,1,1,0,215841,10720772096,575681,32925286400,696737,33080475648\\na0655,128,32,32,1,1,1,215841,10720772096,575681,32925286400,696737,33080475648\\na0656,128,32,64,0,0,0,252801,10871898112,621825,33227276288,734689,33235664896\\na0657,128,32,64,0,0,1,252801,10871898112,621825,33227276288,734689,33235664896\\na0658,128,32,64,0,1,0,252801,10871898112,621825,33227276288,734689,33235664896\\na0659,128,32,64,0,1,1,252801,10871898112,621825,33227276288,734689,33235664896\\na0660,128,32,64,1,0,0,252801,10871898112,621825,33227276288,734689,33235664896\\na0661,128,32,64,1,0,1,252801,10871898112,621825,33227276288,734689,33235664896\\na0662,128,32,64,1,1,0,252801,10871898112,621825,33227276288,734689,33235664896\\na0663,128,32,64,1,1,1,252801,10871898112,621825,33227276288,734689,33235664896\\na0664,128,32,128,0,0,0,382017,11400642560,769409,34057748480,865889,33772535808\\na0665,128,32,128,0,0,1,382017,11400642560,769409,34057748480,865889,33772535808\\na0666,128,32,128,0,1,0,382017,11400642560,769409,34057748480,865889,33772535808\\na0667,128,32,128,0,1,1,382017,11400642560,769409,34057748480,865889,33772535808\\na0668,128,32,128,1,0,0,382017,11400642560,769409,34057748480,865889,33772535808\\na0669,128,32,128,1,0,1,382017,11400642560,769409,34057748480,865889,33772535808\\na0670,128,32,128,1,1,0,382017,11400642560,769409,34057748480,865889,33772535808\\na0671,128,32,128,1,1,1,382017,11400642560,769409,34057748480,865889,33772535808\\na0672,128,32,256,0,0,0,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0673,128,32,256,0,0,1,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0674,128,32,256,0,1,0,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0675,128,32,256,0,1,1,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0676,128,32,256,1,0,0,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0677,128,32,256,1,0,1,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0678,128,32,256,1,1,0,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0679,128,32,256,1,1,1,861633,13364101120,1285761,36624662528,1349473,35752247296\\na0680,128,64,16,0,0,0,273457,11749359616,725537,37275828224,791265,34713108480\\na0681,128,64,16,0,0,1,273457,11749359616,725537,37275828224,791265,34713108480\\na0682,128,64,16,0,1,0,273457,11749359616,725537,37275828224,791265,34713108480\\na0683,128,64,16,0,1,1,273457,11749359616,725537,37275828224,791265,34713108480\\na0684,128,64,16,1,0,0,273457,11749359616,725537,37275828224,791265,34713108480\\na0685,128,64,16,1,0,1,273457,11749359616,725537,37275828224,791265,34713108480\\na0686,128,64,16,1,1,0,273457,11749359616,725537,37275828224,791265,34713108480\\na0687,128,64,16,1,1,1,273457,11749359616,725537,37275828224,791265,34713108480\\na0688,128,64,32,0,0,0,289633,11815485440,750913,37492883456,808449,34783363072\\na0689,128,64,32,0,0,1,289633,11815485440,750913,37492883456,808449,34783363072\\na0690,128,64,32,0,1,0,289633,11815485440,750913,37492883456,808449,34783363072\\na0691,128,64,32,0,1,1,289633,11815485440,750913,37492883456,808449,34783363072\\na0692,128,64,32,1,0,0,289633,11815485440,750913,37492883456,808449,34783363072\\na0693,128,64,32,1,0,1,289633,11815485440,750913,37492883456,808449,34783363072\\na0694,128,64,32,1,1,0,289633,11815485440,750913,37492883456,808449,34783363072\\na0695,128,64,32,1,1,1,289633,11815485440,750913,37492883456,808449,34783363072\\na0696,128,64,64,0,0,0,335809,12004360192,815489,37983617024,856641,34980495360\\na0697,128,64,64,0,0,1,335809,12004360192,815489,37983617024,856641,34980495360\\na0698,128,64,64,0,1,0,335809,12004360192,815489,37983617024,856641,34980495360\\na0699,128,64,64,0,1,1,335809,12004360192,815489,37983617024,856641,34980495360\\na0700,128,64,64,1,0,0,335809,12004360192,815489,37983617024,856641,34980495360\\na0701,128,64,64,1,0,1,335809,12004360192,815489,37983617024,856641,34980495360\\na0702,128,64,64,1,1,0,335809,12004360192,815489,37983617024,856641,34980495360\\na0703,128,64,64,1,1,1,335809,12004360192,815489,37983617024,856641,34980495360\\na0704,128,64,128,0,0,0,483457,12608602112,999937,39191576576,1008321,35601252352\\na0705,128,64,128,0,0,1,483457,12608602112,999937,39191576576,1008321,35601252352\\na0706,128,64,128,0,1,0,483457,12608602112,999937,39191576576,1008321,35601252352\\na0707,128,64,128,0,1,1,483457,12608602112,999937,39191576576,1008321,35601252352\\na0708,128,64,128,1,0,0,483457,12608602112,999937,39191576576,1008321,35601252352\\na0709,128,64,128,1,0,1,483457,12608602112,999937,39191576576,1008321,35601252352\\na0710,128,64,128,1,1,0,483457,12608602112,999937,39191576576,1008321,35601252352\\na0711,128,64,128,1,1,1,483457,12608602112,999937,39191576576,1008321,35601252352\\na0712,128,64,256,0,0,0,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0713,128,64,256,0,0,1,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0714,128,64,256,0,1,0,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0715,128,64,256,0,1,1,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0716,128,64,256,1,0,0,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0717,128,64,256,1,0,1,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0718,128,64,256,1,1,0,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0719,128,64,256,1,1,1,999937,14723055616,1590017,42513465344,1532865,37748736000\\na0720,128,128,16,0,0,0,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0721,128,128,16,0,0,1,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0722,128,128,16,0,1,0,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0723,128,128,16,0,1,1,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0724,128,128,16,1,0,0,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0725,128,128,16,1,0,1,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0726,128,128,16,1,1,0,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0727,128,128,16,1,1,1,467121,14807007232,1223457,48940187648,1059745,38982909952\\na0728,128,128,32,0,0,0,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0729,128,128,32,0,0,1,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0730,128,128,32,0,1,0,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0731,128,128,32,0,1,1,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0732,128,128,32,1,0,0,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0733,128,128,32,1,0,1,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0734,128,128,32,1,1,0,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0735,128,128,32,1,1,1,492513,14910881792,1267265,49345986560,1087169,39095107584\\na0736,128,128,64,0,0,0,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0737,128,128,64,0,0,1,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0738,128,128,64,0,1,0,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0739,128,128,64,0,1,1,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0740,128,128,64,1,0,0,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0741,128,128,64,1,0,1,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0742,128,128,64,1,1,0,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0743,128,128,64,1,1,1,557121,15175254016,1368705,50214207488,1155841,39376125952\\na0744,128,128,128,0,0,0,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0745,128,128,128,0,0,1,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0746,128,128,128,0,1,0,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0747,128,128,128,0,1,1,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0748,128,128,128,1,0,0,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0749,128,128,128,1,0,1,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0750,128,128,128,1,1,0,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0751,128,128,128,1,1,1,741633,15930490880,1626881,52177141760,1348481,40164655104\\na0752,128,128,256,0,0,0,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0753,128,128,256,0,0,1,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0754,128,128,256,0,1,0,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0755,128,128,256,0,1,1,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0756,128,128,256,1,0,0,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0757,128,128,256,1,0,1,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0758,128,128,256,1,1,0,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0759,128,128,256,1,1,1,1331841,18346934272,2364417,57008979968,1954945,42647683072\\na0760,128,256,16,0,0,0,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0761,128,256,16,0,0,1,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0762,128,256,16,0,1,0,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0763,128,256,16,0,1,1,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0764,128,256,16,1,0,0,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0765,128,256,16,1,0,1,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0766,128,256,16,1,1,0,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0767,128,256,16,1,1,1,1075633,24546181120,2882849,83140542464,1817889,51146391552\\na0768,128,256,32,0,0,0,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0769,128,256,32,0,0,1,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0770,128,256,32,0,1,0,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0771,128,256,32,0,1,1,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0772,128,256,32,1,0,0,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0773,128,256,32,1,0,1,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0774,128,256,32,1,1,0,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0775,128,256,32,1,1,1,1119457,24725553152,2963521,83923828736,1865793,51342475264\\na0776,128,256,64,0,0,0,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0777,128,256,64,0,0,1,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0778,128,256,64,0,1,0,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0779,128,256,64,0,1,1,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0780,128,256,64,1,0,0,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0781,128,256,64,1,0,1,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0782,128,256,64,1,1,0,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0783,128,256,64,1,1,1,1220929,25140920320,3138689,85547024384,1975425,51791265792\\na0784,128,256,128,0,0,0,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0785,128,256,128,0,0,1,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0786,128,256,128,0,1,0,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0787,128,256,128,0,1,1,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0788,128,256,128,1,0,0,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0789,128,256,128,1,0,1,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0790,128,256,128,1,1,0,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0791,128,256,128,1,1,1,1479169,26198147072,3544321,89019908096,2249985,52915339264\\na0792,128,256,256,0,0,0,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0793,128,256,256,0,0,1,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0794,128,256,256,0,1,0,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0795,128,256,256,0,1,1,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0796,128,256,256,1,0,0,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0797,128,256,256,1,0,1,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0798,128,256,256,1,1,0,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0799,128,256,256,1,1,1,2216833,29218570240,4576769,96871645184,3020289,56069455872\\na0800,256,16,16,0,0,0,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0801,256,16,16,0,0,1,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0802,256,16,16,0,1,0,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0803,256,16,16,0,1,1,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0804,256,16,16,1,0,0,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0805,256,16,16,1,0,1,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0806,256,16,16,1,1,0,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0807,256,16,16,1,1,1,641105,39768358912,1865313,119623647232,2448977,127363186688\\na0808,256,16,32,0,0,0,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0809,256,16,32,0,0,1,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0810,256,16,32,0,1,0,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0811,256,16,32,0,1,1,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0812,256,16,32,1,0,0,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0813,256,16,32,1,0,1,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0814,256,16,32,1,1,0,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0815,256,16,32,1,1,1,650369,39806173184,1876865,119699144704,2458481,127401984000\\na0816,256,16,64,0,0,0,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0817,256,16,64,0,0,1,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0818,256,16,64,0,1,0,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0819,256,16,64,0,1,1,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0820,256,16,64,1,0,0,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0821,256,16,64,1,0,1,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0822,256,16,64,1,1,0,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0823,256,16,64,1,1,1,682721,39938424832,1913793,119906762752,2491313,127536201728\\na0824,256,16,128,0,0,0,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0825,256,16,128,0,0,1,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0826,256,16,128,0,1,0,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0827,256,16,128,0,1,1,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0828,256,16,128,1,0,0,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0829,256,16,128,1,0,1,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0830,256,16,128,1,1,0,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0831,256,16,128,1,1,1,802721,40429420544,2042945,120548491264,2612273,128031129600\\na0832,256,16,256,0,0,0,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0833,256,16,256,0,0,1,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0834,256,16,256,0,1,0,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0835,256,16,256,0,1,1,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0836,256,16,256,1,0,0,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0837,256,16,256,1,0,1,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0838,256,16,256,1,1,0,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0839,256,16,256,1,1,1,1263905,42317381632,2522433,122737917952,3075377,129926955008\\na0840,256,32,16,0,0,0,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0841,256,32,16,0,0,1,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0842,256,32,16,0,1,0,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0843,256,32,16,0,1,1,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0844,256,32,16,1,0,0,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0845,256,32,16,1,0,1,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0846,256,32,16,1,1,0,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0847,256,32,16,1,1,1,687217,40495022080,1964449,123030470656,2532225,128694878208\\na0848,256,32,32,0,0,0,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0849,256,32,32,0,0,1,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0850,256,32,32,0,1,0,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0851,256,32,32,0,1,1,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0852,256,32,32,1,0,0,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0853,256,32,32,1,0,1,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0854,256,32,32,1,1,0,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0855,256,32,32,1,1,1,698785,40542273536,1980609,123153154048,2544289,128744161280\\na0856,256,32,64,0,0,0,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0857,256,32,64,0,0,1,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0858,256,32,64,0,1,0,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0859,256,32,64,0,1,1,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0860,256,32,64,1,0,0,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0861,256,32,64,1,0,1,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0862,256,32,64,1,1,0,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0863,256,32,64,1,1,1,735745,40693399552,2026753,123455143936,2582241,128899350528\\na0864,256,32,128,0,0,0,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0865,256,32,128,0,0,1,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0866,256,32,128,0,1,0,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0867,256,32,128,0,1,1,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0868,256,32,128,1,0,0,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0869,256,32,128,1,0,1,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0870,256,32,128,1,1,0,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0871,256,32,128,1,1,1,864961,41222144000,2174337,124285616128,2713441,129436221440\\na0872,256,32,256,0,0,0,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0873,256,32,256,0,0,1,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0874,256,32,256,0,1,0,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0875,256,32,256,0,1,1,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0876,256,32,256,1,0,0,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0877,256,32,256,1,0,1,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0878,256,32,256,1,1,0,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0879,256,32,256,1,1,1,1344577,43185602560,2690689,126852530176,3197025,131415932928\\na0880,256,64,16,0,0,0,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0881,256,64,16,0,0,1,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0882,256,64,16,0,1,0,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0883,256,64,16,0,1,1,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0884,256,64,16,1,0,0,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0885,256,64,16,1,0,1,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0886,256,64,16,1,1,0,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0887,256,64,16,1,1,1,793265,42174840832,2204193,130523594752,2712545,131584753664\\na0888,256,64,32,0,0,0,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0889,256,64,32,0,0,1,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0890,256,64,32,0,1,0,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0891,256,64,32,0,1,1,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0892,256,64,32,1,0,0,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0893,256,64,32,1,0,1,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0894,256,64,32,1,1,0,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0895,256,64,32,1,1,1,809441,42240966656,2229569,130740649984,2729729,131655008256\\na0896,256,64,64,0,0,0,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0897,256,64,64,0,0,1,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0898,256,64,64,0,1,0,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0899,256,64,64,0,1,1,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0900,256,64,64,1,0,0,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0901,256,64,64,1,0,1,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0902,256,64,64,1,1,0,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0903,256,64,64,1,1,1,855617,42429841408,2294145,131231383552,2777921,131852140544\\na0904,256,64,128,0,0,0,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0905,256,64,128,0,0,1,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0906,256,64,128,0,1,0,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0907,256,64,128,0,1,1,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0908,256,64,128,1,0,0,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0909,256,64,128,1,0,1,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0910,256,64,128,1,1,0,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0911,256,64,128,1,1,1,1003265,43034083328,2478593,132439343104,2929601,132472897536\\na0912,256,64,256,0,0,0,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0913,256,64,256,0,0,1,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0914,256,64,256,0,1,0,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0915,256,64,256,0,1,1,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0916,256,64,256,1,0,0,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0917,256,64,256,1,0,1,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0918,256,64,256,1,1,0,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0919,256,64,256,1,1,1,1519745,45148536832,3068673,135761231872,3454145,134620381184\\na0920,256,128,16,0,0,0,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0921,256,128,16,0,0,1,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0922,256,128,16,0,1,0,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0923,256,128,16,0,1,1,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0924,256,128,16,1,0,0,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0925,256,128,16,1,0,1,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0926,256,128,16,1,1,0,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0927,256,128,16,1,1,1,1060657,46440448000,2849569,148227751936,3128481,138270474240\\na0928,256,128,32,0,0,0,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0929,256,128,32,0,0,1,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0930,256,128,32,0,1,0,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0931,256,128,32,0,1,1,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0932,256,128,32,1,0,0,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0933,256,128,32,1,0,1,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0934,256,128,32,1,1,0,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0935,256,128,32,1,1,1,1086049,46544322560,2893377,148633550848,3155905,138382671872\\na0936,256,128,64,0,0,0,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0937,256,128,64,0,0,1,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0938,256,128,64,0,1,0,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0939,256,128,64,0,1,1,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0940,256,128,64,1,0,0,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0941,256,128,64,1,0,1,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0942,256,128,64,1,1,0,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0943,256,128,64,1,1,1,1150657,46808694784,2994817,149501771776,3224577,138663690240\\na0944,256,128,128,0,0,0,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0945,256,128,128,0,0,1,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0946,256,128,128,0,1,0,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0947,256,128,128,0,1,1,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0948,256,128,128,1,0,0,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0949,256,128,128,1,0,1,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0950,256,128,128,1,1,0,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0951,256,128,128,1,1,1,1335169,47563931648,3252993,151464706048,3417217,139452219392\\na0952,256,128,256,0,0,0,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0953,256,128,256,0,0,1,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0954,256,128,256,0,1,0,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0955,256,128,256,0,1,1,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0956,256,128,256,1,0,0,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0957,256,128,256,1,0,1,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0958,256,128,256,1,1,0,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0959,256,128,256,1,1,1,1925377,49980375040,3990529,156296544256,4023681,141935247360\\na0960,256,256,16,0,0,0,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0961,256,256,16,0,0,1,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0962,256,256,16,0,1,0,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0963,256,256,16,0,1,1,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0964,256,256,16,1,0,0,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0965,256,256,16,1,0,1,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0966,256,256,16,1,1,0,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0967,256,256,16,1,1,1,1816625,58595540992,4803873,194507702272,4181537,155265794048\\na0968,256,256,32,0,0,0,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0969,256,256,32,0,0,1,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0970,256,256,32,0,1,0,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0971,256,256,32,0,1,1,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0972,256,256,32,1,0,0,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0973,256,256,32,1,0,1,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0974,256,256,32,1,1,0,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0975,256,256,32,1,1,1,1860449,58774913024,4884545,195290988544,4229441,155461877760\\na0976,256,256,64,0,0,0,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0977,256,256,64,0,0,1,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0978,256,256,64,0,1,0,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0979,256,256,64,0,1,1,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0980,256,256,64,1,0,0,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0981,256,256,64,1,0,1,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0982,256,256,64,1,1,0,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0983,256,256,64,1,1,1,1961921,59190280192,5059713,196914184192,4339073,155910668288\\na0984,256,256,128,0,0,0,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0985,256,256,128,0,0,1,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0986,256,256,128,0,1,0,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0987,256,256,128,0,1,1,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0988,256,256,128,1,0,0,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0989,256,256,128,1,0,1,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0990,256,256,128,1,1,0,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0991,256,256,128,1,1,1,2220161,60247506944,5465345,200387067904,4613633,157034741760\\na0992,256,256,256,0,0,0,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0993,256,256,256,0,0,1,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0994,256,256,256,0,1,0,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0995,256,256,256,0,1,1,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0996,256,256,256,1,0,0,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0997,256,256,256,1,0,1,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0998,256,256,256,1,1,0,2957825,63267930112,6497793,208238804992,5383937,160188858368\\na0999,256,256,256,1,1,1,2957825,63267930112,6497793,208238804992,5383937,160188858368\\n\'}\n', 'for rel, texto in FUENTES_E2.items():\n', '    p = PROJECT / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", "subprocess.run([PY, str(E3 / 'training/entrenar_compactas_e3.py'), '--e2', str(PROJECT),\n", "                '--output', str(OUTPUT)], check=True)\n"], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '06_Rivales_A.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# E3: semillas 6 a 10 de los rivales, sesión A\n', '\n', 'Mismo cuaderno, mismo paquete y misma receta que el benchmark del E2. Cambian la carpeta de Drive\n', '(`tesis_imo_e3_semillas_v1`) y las semillas (89, 97, 101, 103, 107).\n', '\n', '\n', 'Redes de esta sesión: transunet_2d, pidnet_s. Las otras sesiones corren las demás; pueden ir en paralelo y escriben\n', 'en subcarpetas distintas (`sesion_A`, `sesion_B`, `sesion_C`), porque cada una guarda su propio `preflight.json`.\n', '\n', '1. Entorno de ejecución → Cambiar tipo → **GPU L4**.\n', '2. Ejecutar las celdas en orden. Si se cae la sesión, volver a ejecutarlas: cada corrida se reanuda desde su\n', '   última época completa.\n', '3. No se evalúa test ni se descargan pesos.\n'], 'id': 'e2-cell-00'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/rivales/sesion_A')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "pip_install('transformers==4.46.3')\n", "pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Comprobación previa obligatoria\n', 'Comprueba hashes de datos/código, separación de grupos, identidad arquitectónica, forward, gradientes, paso de optimizador y evaluación con datos sintéticos. Guarda memoria GPU y tiempo. Si falla un modelo, corrige el entorno o reduce **explícitamente** el alcance; no hay sustituciones automáticas. El lote nominal es 2; el último lote de una imagen se une al anterior (3) para BatchNorm.\n'], 'id': 'e2-cell-05'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["MODELS=['transunet_2d', 'pidnet_s']\n", "subprocess.run([PY,str(PROJECT/'scripts/preflight_e2.py'),'--output',str(OUTPUT),'--models',*MODELS],check=True)\n", "report=json.loads((OUTPUT/'preflight.json').read_text())\n", "for name,result in report['models'].items(): print(name,result)\n"], 'id': 'e2-cell-06'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Ejecutar o reanudar\n', 'Corre las redes de esta sesión con las cinco semillas nuevas. Se recupera la última época completa al reconectar.\n'], 'id': 'e2-cell-07'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['MODELS_RUN=MODELS\n', 'SEEDS=[89, 97, 101, 103, 107]\n', "subprocess.run([PY,str(PROJECT/'scripts/experimento_e2_colab.py'),\n", "    '--output',str(OUTPUT),'--models',*MODELS_RUN,'--seeds',*[str(s) for s in SEEDS]],check=True)\n"], 'id': 'e2-cell-08'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["for p in sorted(OUTPUT.glob('*/result.json')):\n", '    r=json.loads(p.read_text())\n', "    print(r['model'],r['seed'],r['completed'],r['best_val_group_Dice'],r['best_file'])\n", "print('Los resultados permanecen en',OUTPUT)\n"], 'id': 'e2-cell-09'}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '07_Rivales_B.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# E3: semillas 6 a 10 de los rivales, sesión B\n', '\n', 'Mismo cuaderno, mismo paquete y misma receta que el benchmark del E2. Cambian la carpeta de Drive\n', '(`tesis_imo_e3_semillas_v1`) y las semillas (89, 97, 101, 103, 107).\n', '\n', '\n', 'Redes de esta sesión: vmunet, segnext_t, deeplabv3plus, unetplusplus. Las otras sesiones corren las demás; pueden ir en paralelo y escriben\n', 'en subcarpetas distintas (`sesion_A`, `sesion_B`, `sesion_C`), porque cada una guarda su propio `preflight.json`.\n', '\n', '1. Entorno de ejecución → Cambiar tipo → **GPU L4**.\n', '2. Ejecutar las celdas en orden. Si se cae la sesión, volver a ejecutarlas: cada corrida se reanuda desde su\n', '   última época completa.\n', '3. No se evalúa test ni se descargan pesos.\n'], 'id': 'e2-cell-00'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/rivales/sesion_B')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "pip_install('transformers==4.46.3')\n", "pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Comprobación previa obligatoria\n', 'Comprueba hashes de datos/código, separación de grupos, identidad arquitectónica, forward, gradientes, paso de optimizador y evaluación con datos sintéticos. Guarda memoria GPU y tiempo. Si falla un modelo, corrige el entorno o reduce **explícitamente** el alcance; no hay sustituciones automáticas. El lote nominal es 2; el último lote de una imagen se une al anterior (3) para BatchNorm.\n'], 'id': 'e2-cell-05'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["MODELS=['vmunet', 'segnext_t', 'deeplabv3plus', 'unetplusplus']\n", "subprocess.run([PY,str(PROJECT/'scripts/preflight_e2.py'),'--output',str(OUTPUT),'--models',*MODELS],check=True)\n", "report=json.loads((OUTPUT/'preflight.json').read_text())\n", "for name,result in report['models'].items(): print(name,result)\n"], 'id': 'e2-cell-06'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Ejecutar o reanudar\n', 'Corre las redes de esta sesión con las cinco semillas nuevas. Se recupera la última época completa al reconectar.\n'], 'id': 'e2-cell-07'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['MODELS_RUN=MODELS\n', 'SEEDS=[89, 97, 101, 103, 107]\n', "subprocess.run([PY,str(PROJECT/'scripts/experimento_e2_colab.py'),\n", "    '--output',str(OUTPUT),'--models',*MODELS_RUN,'--seeds',*[str(s) for s in SEEDS]],check=True)\n"], 'id': 'e2-cell-08'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["for p in sorted(OUTPUT.glob('*/result.json')):\n", '    r=json.loads(p.read_text())\n', "    print(r['model'],r['seed'],r['completed'],r['best_val_group_Dice'],r['best_file'])\n", "print('Los resultados permanecen en',OUTPUT)\n"], 'id': 'e2-cell-09'}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '08_Rivales_C.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# E3: semillas 6 a 10 de los rivales, sesión C\n', '\n', 'Mismo cuaderno, mismo paquete y misma receta que el benchmark del E2. Cambian la carpeta de Drive\n', '(`tesis_imo_e3_semillas_v1`) y las semillas (11, 23, 37, 53, 71, 89, 97, 101, 103, 107).\n', '**Sesión C:** SegFormer-B0 con la configuración ganadora de la búsqueda del cierre del E2 (lr 5,57e-4, wd 1e-4).\n', '\n', 'Redes de esta sesión: segformer_b0. Las otras sesiones corren las demás; pueden ir en paralelo y escriben\n', 'en subcarpetas distintas (`sesion_A`, `sesion_B`, `sesion_C`), porque cada una guarda su propio `preflight.json`.\n', '\n', '1. Entorno de ejecución → Cambiar tipo → **GPU L4**.\n', '2. Ejecutar las celdas en orden. Si se cae la sesión, volver a ejecutarlas: cada corrida se reanuda desde su\n', '   última época completa.\n', '3. No se evalúa test ni se descargan pesos.\n'], 'id': 'e2-cell-00'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/rivales/sesion_C')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "pip_install('transformers==4.46.3')\n", "pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Comprobación previa obligatoria\n', 'Comprueba hashes de datos/código, separación de grupos, identidad arquitectónica, forward, gradientes, paso de optimizador y evaluación con datos sintéticos. Guarda memoria GPU y tiempo. Si falla un modelo, corrige el entorno o reduce **explícitamente** el alcance; no hay sustituciones automáticas. El lote nominal es 2; el último lote de una imagen se une al anterior (3) para BatchNorm.\n'], 'id': 'e2-cell-05'}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONFIG_E3 = \'{\\n  "schema": 2,\\n  "experiment": "e3_rivales_v2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 30,\\n  "patience": 5,\\n  "seeds": [\\n    11,\\n    23,\\n    37,\\n    53,\\n    71,\\n    89,\\n    97,\\n    101,\\n    103,\\n    107\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005573638936676344,\\n      "weight_decay": 0.0001\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Igual a experimento_e2.json salvo segformer_b0: configuración ganadora (t = 2) de la búsqueda equitativa del cierre del E2. Corrección del bloque A del E3."\\n}\'\n', "ruta = PROJECT / 'config/experimento_e2.json'\n", 'original, nueva = json.loads(ruta.read_text()), json.loads(CONFIG_E3)\n', "cambios = [m for m in original['models'] if original['models'][m] != nueva['models'][m]]\n", "assert cambios == ['segformer_b0'], cambios\n", 'ruta.write_text(CONFIG_E3)\n', "print('Configuración del E3 escrita en el paquete descomprimido; cambia solo:', cambios, nueva['models']['segformer_b0'])\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["MODELS=['segformer_b0']\n", "subprocess.run([PY,str(PROJECT/'scripts/preflight_e2.py'),'--output',str(OUTPUT),'--models',*MODELS],check=True)\n", "report=json.loads((OUTPUT/'preflight.json').read_text())\n", "for name,result in report['models'].items(): print(name,result)\n"], 'id': 'e2-cell-06'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Ejecutar o reanudar\n', 'Corre las redes de esta sesión con las cinco semillas nuevas. Se recupera la última época completa al reconectar.\n'], 'id': 'e2-cell-07'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['MODELS_RUN=MODELS\n', 'SEEDS=[11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\n', "subprocess.run([PY,str(PROJECT/'scripts/experimento_e2_colab.py'),\n", "    '--output',str(OUTPUT),'--models',*MODELS_RUN,'--seeds',*[str(s) for s in SEEDS]],check=True)\n"], 'id': 'e2-cell-08'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["for p in sorted(OUTPUT.glob('*/result.json')):\n", '    r=json.loads(p.read_text())\n', "    print(r['model'],r['seed'],r['completed'],r['best_val_group_Dice'],r['best_file'])\n", "print('Los resultados permanecen en',OUTPUT)\n"], 'id': 'e2-cell-09'}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '09_Verificar_y_devolver.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 09 · Verificar y devolver resultados\n', '\n', 'Usa CPU. No entrena ni evalúa test. Verifica las 120 corridas principales esperadas, incluyendo las históricas. Los resultados precalculados o archivos ausentes se reportan; no se inventan.']}, {'cell_type': 'code', 'metadata': {}, 'source': ['from google.colab import drive, files\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import json, hashlib, zipfile, subprocess, sys\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", "BENCHMARK = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')\n", "COMPACTAS = Path('/content/compactas_e3_recibidas')\n", "ZIP = Path('/content/drive/MyDrive/compactas_a0498_e3.zip')\n", "assert BENCHMARK.is_dir(), 'Falta la carpeta histórica que confirmaste en Drive'\n", "assert ZIP.is_file(), 'Sube compactas_a0498_e3.zip a MyDrive'\n", 'COMPACTAS.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for name in z.namelist():\n', '        assert (COMPACTAS/name).resolve().is_relative_to(COMPACTAS)\n', '    z.extractall(COMPACTAS)\n', "ESPERADOS = {'fcn_a0498_s11': 'ef655608d580e34a72966f6f352e643544c3eb8c215dace5d246da98f72814c7', 'fcn_a0498_s23': 'c9f9856e9239d9c59366871525b870ac421ef6ae370bb60b29bff6960aeb2772', 'fcn_a0498_s37': 'f5c326a78e22f4d8dca84cdfd0b86a9cd9646e947ab22c59465fd82d4af00df4', 'fcn_a0498_s53': '6f220654d54344f5b6545087d3c102c0415be2d97eed8721b62ebb36789a86fa', 'fcn_a0498_s71': '9e2b5d911b7722573885eeb157cf75f43b5302d11eabcb1ad287f6c16ff1164e', 'segnet_a0498_s11': 'c2bc57be34caa5501118601b9d050be50062f5a657309850e47e2a96b5ac2d14', 'segnet_a0498_s23': 'a5d63031449a429343961f982ef26fdcf6817c139fb1c7142a3b0fd4199d2dac', 'segnet_a0498_s37': '7c358e59dd5ef539375b43d78a30b5f46de262418afac998a956bea0bc75e750', 'segnet_a0498_s53': '3c41962731ca6d1680a879f6d25608abcda7790c6e24e91ed68835463bc1376c', 'segnet_a0498_s71': 'c18f29d092e404dc42210bf284fb4b998c02eebda495322ac42e9eb2c8518ecd', 'unet_a0498_s11': '4138b06d20bd32a422a69970252187322a35fea09a4e304bcda61d904ac672a9', 'unet_a0498_s23': 'e0452e0bf9df896ffaafbbc1838c02abc070d1b37478121fc25246a33aec75aa', 'unet_a0498_s37': '59c6df9ad6c05500114d0e5e4948e8fd4cc5bd831eda4c577aab6066be25ddeb', 'unet_a0498_s53': '615d2b1a786ec37372e8ca2868783ec99ce135c556b63610b9a7bfc1e4a10f02', 'unet_a0498_s71': 'a0d80416558ccdfd1632ad34ef1bd132ac896f2c83ab7b8b7c2f5d0adb03df84'}\n", 'for run, expected in ESPERADOS.items():\n', "    assert hashlib.sha256((COMPACTAS/'fase2'/run/'best.pt').read_bytes()).hexdigest()==expected\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['SCRIPT = \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\'\n', "p = Path('/content/auditar_campana.py'); p.write_text(SCRIPT, encoding='utf-8')\n", "r = subprocess.run([sys.executable,str(p),'--campana',str(CAMPANA),\n", "    '--benchmark',str(BENCHMARK),'--compactas',str(COMPACTAS)])\n", "assert r.returncode in [0,2], 'Error del auditor'\n", "print((CAMPANA/'estado_campana.json').read_text())\n", "archive = Path('/content/resultados_e3_secuencia_v4.zip')\n", "with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:\n", "    for p in CAMPANA.rglob('*'):\n", "        if p.is_file() and p.suffix in ['.json','.csv','.txt']:\n", '            z.write(p,str(p.relative_to(CAMPANA)))\n', 'files.download(str(archive))\n', "print('Los pesos permanecen en Drive. Devuelve este ZIP para integración y revisión antes de test.')\n"], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}, '10_Evaluacion_validacion.ipynb': {'cells': [{'cell_type': 'markdown', 'metadata': {}, 'source': ['# 10 · Evaluación reproducible de validación\n', '\n', 'Solo ejecutar cuando 09 informe 120/120 verificadas. Recalcula Dice e IoU por imagen con los pesos elegidos. No usa test. Puede tardar: admite reanudación por corrida.\n', '\n', 'Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/evaluacion_validacion`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.\n']}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ['from google.colab import drive\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import subprocess, sys, os, json, hashlib, zipfile\n', "subprocess.run(['nvidia-smi'], check=True)\n", "subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)\n", "OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/evaluacion_validacion')\n", 'OUTPUT.mkdir(parents=True, exist_ok=True)\n'], 'id': 'e2-cell-01'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'\n", "ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)\n", "if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')\n", 'def sha_file(p):\n', '    h=hashlib.sha256()\n', "    with p.open('rb') as f:\n", "        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)\n", '    return h.hexdigest()\n', "assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'\n", "BASE=Path('/content/imo_e2_v2')\n", 'BASE.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for n in z.namelist():\n', "        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')\n", '    z.extractall(BASE)\n', "manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())\n", "for name, expected in manifest['files'].items():\n", '    assert sha_file(BASE/name)==expected, name\n', "PROJECT=BASE/'tesis_imo_para_codex'\n", "print('Paquete íntegro:',len(manifest['files']),'archivos')\n"], 'id': 'e2-cell-02'}, {'cell_type': 'markdown', 'metadata': {}, 'source': ['## Entorno aislado y fijado\n', 'Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.\n'], 'id': 'e2-cell-03'}, {'cell_type': 'code', 'execution_count': None, 'metadata': {}, 'outputs': [], 'source': ["subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)\n", "ENV=Path('/content/imo-e2-env')\n", "if not (ENV/'bin/python').exists():\n", "    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)\n", "PY=str(ENV/'bin/python')\n", 'def pip_install(*items):\n', "    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)\n", "pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')\n", "pip_install('-r',str(PROJECT/'requirements-e2.txt'))\n", "pip_install('transformers==4.46.3')\n", "pip_install('--no-deps','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "pip_install('--no-deps','https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl')\n", "subprocess.run([PY,'-c','import torch, torchvision, mamba_ssm; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)\n", "freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)\n", "(OUTPUT/'environment.freeze.txt').write_text(freeze)\n"], 'id': 'e2-cell-04'}, {'cell_type': 'code', 'metadata': {}, 'source': ["pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')\n", 'subprocess.run([PY, \'-c\', "import cv2,numpy; assert cv2.__version__==\'4.11.0\'; assert numpy.__version__==\'1.26.4\'; print(cv2.__version__,numpy.__version__)"],check=True)\n'], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['CONTENIDO = {\'src/__init__.py\': \'\', \'src/geometria_at.py\': \'"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\\n\\nFuentes:\\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\\\' = A\\\' r\\\'/L\\\'\\n  con r\\\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\\n\\nNotacion (la del teorema de 3.5):\\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\\n  (1/4 de 256), fijado por la arquitectura.\\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\\n  con D = rho(K).\\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\\n- C2 busca el d que cumple d = rho(M(K\\\', d)), con K\\\' el nucleo predicho. Teorema: si K es convexo, ningun lado\\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\\n  M(K\\\', D) = K.\\n\\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\\nqueda solo como prueba cruzada en tests/test_original_model.py.\\n\\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\\narea minima 9 px.\\n"""\\nimport cv2\\nimport numpy as np\\nfrom scipy import ndimage\\n\\n# ----------------------------------------------------------------------------- constantes\\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\\nR = 0.4             # razon de encogimiento de DBNet\\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\\nOCHO = np.ones((3, 3), bool)\\n\\n\\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\\ndef instancias(mascara):\\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\\n    return ndimage.label(mascara > 0, structure=OCHO)\\n\\n\\ndef area_perimetro(inst):\\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\\n    return float(inst.sum()), max(float(perim), 1.0)\\n\\n\\ndef encoger(inst, d):\\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\\n    return ndimage.distance_transform_edt(inst) > d\\n\\n\\ndef expandir(inst, d):\\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\\n    return ndimage.distance_transform_edt(~inst) <= d\\n\\n\\ndef componentes(binaria, area_min):\\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\\n    etq, n = instancias(binaria)\\n    if n == 0:\\n        return etq, []\\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\\n\\n\\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\\ndef phi(g, g0=G0):\\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\\n\\n\\ndef regla(region, r=R, g0=G0):\\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\\n    A, L = area_perimetro(region)\\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\\n    return phi(g, g0) * A * (1 - r ** 2) / L\\n\\n\\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\\n\\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\\n    etq, n = instancias(y)\\n    gt = np.zeros(y.shape, np.float32)\\n    mask = v.astype(np.float32).copy()\\n    umbral = np.zeros(y.shape, np.float32)\\n    umbral_mask = np.zeros(y.shape, np.float32)\\n    for k in range(1, n + 1):\\n        inst = etq == k\\n        D = regla(inst, r, g0)\\n        s = encoger(inst, D)\\n        if not s.any():\\n            mask[inst] = 0\\n            continue\\n        gt[s] = 1\\n        gd = expandir(inst, D)\\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\\n        umbral_mask[gd] = 1\\n    umbral = umbral * (tmax - tmin) + tmin\\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\\n\\n\\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\\ndef _orientar(p):\\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\\n    p = np.asarray(p, float)\\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\\n    return q if area2 >= 0 else q[::-1]\\n\\n\\ndef _geometria_lados(p):\\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\\n    e = np.roll(p, -1, axis=0) - p\\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\\n    return t, n\\n\\n\\ndef _esquina(v, t1, n1, t2, n2, d, limite):\\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\\n\\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \\\'squaring\\\'); entonces\\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\\n    den = 1.0 + float(n1 @ n2)\\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\\n        return [v + d * (n1 + n2) / den]\\n    u = n1 + n2\\n    nu = np.linalg.norm(u)\\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\\n    L = limite * d\\n    pts = []\\n    for t, n in ((t1, n1), (t2, n2)):\\n        ct = float(t @ u)\\n        if abs(ct) < 1e-12:\\n            pts.append(v + d * n)\\n        else:\\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\\n    return pts\\n\\n\\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\\n\\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    salida = []\\n    for i in range(len(p)):\\n        j = i - 1  # lado entrante al vertice i\\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\\n    return np.asarray(salida)\\n\\n\\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\\n    desplazamiento \\\'exacto\\\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\\n    p = _orientar(p)\\n    t, n = _geometria_lados(p)\\n    k = len(p)\\n    piezas = [p]\\n    for i in range(k):\\n        a, b = p[i], p[(i + 1) % k]\\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\\n        j = i - 1\\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\\n    return piezas\\n\\n\\ndef _pintar(salida, poligono):\\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\\n\\n\\ndef _contornos(c):\\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\\n\\n\\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\\n    salida = np.zeros(forma, np.uint8)\\n    for p in contornos:\\n        if len(p) < 3:\\n            continue\\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\\n            _pintar(salida, inglete_poligono(p, d, limite))\\n        else:\\n            for pieza in _piezas_inglete(p, d, limite):\\n                _pintar(salida, pieza)\\n    return salida\\n\\n\\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\\n\\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\\n    if desp <= 0:\\n        return c.copy()\\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\\n    return (salida > 0) | c\\n\\n\\n# ----------------------------------------------------------------------------- C2: expansion consistente\\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\\n    completo). Es la inversa de C1: M(K\\\', d) con d = rho(M(K\\\', d)) (diseno.md 3.3). La region se recorta a su caja\\n    con un margen suficiente para que los calculos sean locales.\\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\\n    ys, xs = np.nonzero(comp)\\n    g_k = float(ndimage.distance_transform_edt(comp).max())\\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\\\'; en una compacta d ~ 0,72 g\\\'\\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\\n\\n    def expandida(d):\\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\\n\\n    def h(d):  # h(d) = rho(M(K\\\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\\n        return regla(expandida(d), r, g0) - d\\n\\n    bajo = 0.0\\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\\n        d = alto\\n    else:\\n        for _ in range(iteraciones):\\n            medio = (bajo + alto) / 2\\n            if h(medio) > 0:\\n                bajo = medio\\n            else:\\n                alto = medio\\n        d = (bajo + alto) / 2\\n    salida = np.zeros(comp.shape, bool)\\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\\n    return salida\\n\\n\\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\\n    al cruce de la biseccion."""\\n    vistas, mejor = set(), (-1.0, 0.0, None)\\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\\n        reg = expandida(float(d))\\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\\n        if clave in vistas or not reg.any():\\n            continue\\n        vistas.add(clave)\\n        nuc = encoger(reg, regla(reg, r, g0))\\n        inter = int((nuc & c).sum())\\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\\n            mejor = (iou, abs(d - d_bis), reg)\\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\\n\\n\\n# ----------------------------------------------------------------------------- posproceso\\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\\n            area_min=9, ajuste=None):\\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\\\' fijo,\\n    como DBNet (D\\\' = A\\\' r\\\'/L\\\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\\n    etq, keep = componentes((prob > umbral) & v, area_min)\\n    salida = np.zeros(prob.shape, bool)\\n    for k in keep:\\n        comp = etq == k\\n        if prob[comp].mean() < puntaje_min:\\n            continue\\n        if consistente == "okr":\\n            salida |= expandir(comp, distancia_okr(comp))\\n        elif consistente:\\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\\n        else:\\n            A, L = area_perimetro(comp)\\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\\n    return salida & v\\n\\n\\ndef distancia_okr(comp):\\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\\n    el punto fijo de C2."""\\n    A, C = area_perimetro(comp)\\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\\n\\n\\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \\\'okr_sobre_at\\\' usa los nucleos del\\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\\n# \\\'dbnet\\\' es el base: regla y posproceso de DBNet (g0 = 0, r\\\' fijo, expansion euclidea).\\nVARIANTES = {\\n    "dbnet": (0.0, False, False),\\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\\\' = 1,5\\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\\\' = 1,5, expansion en inglete (C2b solo)\\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\\\' = 1,5 en inglete (sin C2a)\\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\\n}\\n\\n\\ndef objetivos_de(variante):\\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\\n    g0 = VARIANTES[variante][0]\\n    return lambda y, v: at_objetivos(y, v, g0=g0)\\n\\n\\ndef post_de(variante):\\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\\n    g0, cons, ing = VARIANTES[variante]\\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\\n\\n\\n# DBNet base, con los nombres de E2\\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\\n\\n\\ndef db_post(prob, v, **kw):\\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\\n\', \'src/original_model.py\': \'"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\\n\\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\\npreentrenados, en el mismo pipeline de E2.\\n\\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\\n    e1 = bloque(3 -> 16)             resolucion completa\\n    e2 = bloque(16 -> 32)            1/2\\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\\nU-Net de E2; las cabezas se crean al final.\\n\\nFormulacion (DBNet, seccion 3 del articulo):\\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\\n    T = sigmoid(z_T)                       mapa de umbral aprendido\\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\\nDerivada de la binarizacion, la que hace entrenable al umbral:\\n    dB/dP =  k B (1 - B)\\n    dB/dT = -k B (1 - B)\\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\\n\\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\\n    L = 5 Ls + 10 Lt + Lb\\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\\n    Lb = Dice entre B y el nucleo\\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\\n\\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\\\'dbnet_at\\\',\\nregla de DBNet si variante=\\\'dbnet\\\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\\n"""\\nfrom contextlib import nullcontext\\n\\nimport numpy as np\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nfrom . import geometria_at as G\\n\\nK_BINARIZACION = 50.0\\n\\n\\ndef bloque(a, b):\\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\n\\nclass UNetAT(nn.Module):\\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\\n\\n    Parametros:\\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\\n        cabeza: \\\'1x1\\\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\\n            de E2) o \\\'3x3\\\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\\n    forward: en entrenamiento devuelve un dict con \\\'prob\\\' (P), \\\'thresh\\\' (T), \\\'binary\\\' (B) y \\\'logit_prob\\\' (z_P, que\\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\\n    """\\n\\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\\n        super().__init__()\\n        if cabeza not in ("1x1", "3x3"):\\n            raise ValueError("cabeza debe ser \\\'1x1\\\' o \\\'3x3\\\'")\\n        self.k = float(k)\\n        self.cabeza_tipo = cabeza\\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\\n        with contexto:\\n            if semilla is not None:\\n                torch.manual_seed(int(semilla))\\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\\n            self.cabeza_prob = self._cabeza(cabeza)\\n            self.cabeza_umbral = self._cabeza(cabeza)\\n\\n    @staticmethod\\n    def _cabeza(tipo):\\n        if tipo == "1x1":\\n            return nn.Conv2d(16, 1, 1)\\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\\n\\n    # ------------------------------------------------------------------ validacion de la entrada\\n    @staticmethod\\n    def _validar(x):\\n        if not isinstance(x, torch.Tensor):\\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\\n        if x.dim() != 4:\\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\\n        if x.shape[1] != 3:\\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\\n        if not x.is_floating_point():\\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\\n        h, w = x.shape[-2:]\\n        if h % 4 or w % 4:\\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\\n\\n    # ------------------------------------------------------------------ red\\n    def _caracteristicas(self, x):\\n        a = self.e1(x)\\n        b = self.e2(F.max_pool2d(a, 2))\\n        c = self.e3(F.max_pool2d(b, 2))\\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\\n\\n    def forward(self, x):\\n        self._validar(x)\\n        h = self._caracteristicas(x)\\n        zp = self.cabeza_prob(h)\\n        if not self.training:\\n            return torch.sigmoid(zp)\\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\\n\\n    # ------------------------------------------------------------------ inferencia\\n    @torch.no_grad()\\n    def predict_proba(self, x):\\n        """P (B, 1, H, W), sin importar el modo del modulo."""\\n        self._validar(x)\\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\\n\\n    @torch.no_grad()\\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\\n        \\\'dbnet_at\\\' = C1 + C2 (expansion consistente en inglete); \\\'dbnet\\\' = el posproceso de DBNet (r\\\' = 1,5).\\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\\n        if variante not in G.VARIANTES:\\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\\n        post = G.post_de(variante)\\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\\n\\n    def perdida(self, salida, objetivos, **pesos):\\n        return perdida_db(salida, objetivos, **pesos)\\n\\n\\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\\ndef bce_ohem(logits, gt, mask, razon=3):\\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\\n    pos = (gt > 0.5) & (mask > 0.5)\\n    neg = (gt <= 0.5) & (mask > 0.5)\\n    n_pos = int(pos.sum())\\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\\n\\n\\ndef dice_perdida(p, t, m):\\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\\n    p, t = p * m, t * m\\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\\n\\n\\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\\n\\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\\n    m = objetivos["db_umbral_mask"]\\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\\n    return peso_s * ls + peso_t * lt + peso_b * lb\\n\', \'training/entrenar_original.py\': \'"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\\n\\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\\n\\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\\nDice medio por grupo en validacion; aqui, despues del posproceso.\\n\\nUso en GPU (Colab):\\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\\nHumo en CPU:\\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\\\\n        --variante dbnet_at --output C:\\\\\\\\Proyectos\\\\\\\\estudio\\\\\\\\datos\\\\\\\\deep_learning\\\\\\\\e3_humo\\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\\n(geometria_at.VARIANTES).\\n"""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nimport time\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nE2_SCRIPTS = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\repo_dl_imo\\\\e2\\\\tesis_imo_para_codex\\\\scripts")\\nVOC = Path(r"C:\\\\Proyectos\\\\estudio\\\\datos\\\\hazmat13_drive")\\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\\n\\n\\ndef cargar_arnes(e2_scripts):\\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\\n    global H, DS, G, UNetAT, perdida_db\\n    e2_scripts = Path(e2_scripts).resolve()\\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\\n    for ruta in (str(e2_scripts), str(E3)):\\n        if ruta not in sys.path:\\n            sys.path.insert(0, ruta)\\n    import experimento_e2_colab as h\\n    import dataset_ready as ds\\n    from src import geometria_at as g\\n    from src.original_model import UNetAT as red, perdida_db as perdida\\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\\n    return e2_scripts\\n\\n\\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\\ndef firma_codigo(e2_scripts):\\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\\n\\n\\ndef verificar_datos(filas, voc):\\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\\n    base = H.PROJECT / "resultados/preentrenamiento"\\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\\n        if grupos[a] & grupos[b]:\\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\\n    usados = {}\\n    for particion, rows in filas.items():\\n        if not rows:\\n            raise RuntimeError(f"Particion vacia: {particion}")\\n        for row in rows:\\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\\n                if H.sha(ruta) != esperado:\\n                    raise RuntimeError(f"Datos modificados: {ruta}")\\n            usados[row["id"]] = row["image_sha256"]\\n    norma = json.loads((base / "normalization_train.json").read_text())\\n    if norma["size"] != 256:\\n        raise RuntimeError("Normalizacion no corresponde a 256")\\n    return H.digest({"csv": firma, "filas_usadas": usados})\\n\\n\\nclass Datos:\\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\\n    los objetivos de cada variante, calculados una vez por imagen."""\\n\\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\\n        t0 = time.perf_counter()\\n        for fila in filas:\\n            x, y, v = DS.load_example(fila, data_root=voc)\\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\\n            if objetivos is not None:\\n                o = objetivos(y, v)\\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\\n\\n    def lote(self, filas, device):\\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\\n        ids = [f["id"] for f in filas]\\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\\n               for k in self.obj[ids[0]]}\\n        return x, obj\\n\\n\\n# ------------------------------------------------------------------------------------------ validacion\\n@torch.no_grad()\\ndef evaluar_validacion(modelo, datos, variante, device):\\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\\n    modelo.eval()\\n    agrupado, detalle = defaultdict(list), []\\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\\n    with torch.random.fork_rng(devices=dispositivos):\\n        torch.manual_seed(SEMILLA_VALIDACION)\\n        for fila in datos.filas:\\n            i = fila["id"]\\n            x = torch.from_numpy(datos.x[i][None]).to(device)\\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\\n\\n\\n# ------------------------------------------------------------------------------------------ entrenamiento\\ndef preflight(variante, config, identity, train, device, destino):\\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\\n    H.setup_seed(0)\\n    modelo = UNetAT().to(device).train()\\n    x, obj = train.lote(train.filas[:2], device)\\n    perdida = perdida_db(modelo(x), obj)\\n    perdida.backward()\\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\\n                                                for p in modelo.parameters())\\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\\n    if not ok:\\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\\n\\n\\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\\n    nombre = f"unetat_{variante}"\\n    H.setup_seed(semilla)\\n    dest = Path(output) / f"{nombre}_seed{semilla}"\\n    dest.mkdir(parents=True, exist_ok=True)\\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\\n    huella = H.digest(spec)\\n    cfg_file = dest / "config.json"\\n    if cfg_file.exists():\\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\\n    else:\\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\\n    last = dest / "last.pt"\\n    if (dest / "result.json").exists():\\n        res = json.loads((dest / "result.json").read_text())\\n        if res.get("fingerprint") != huella or not res.get("completed"):\\n            raise RuntimeError("Resultado incompatible")\\n        if not last.exists() or not (dest / res["best_file"]).exists():\\n            raise RuntimeError("Faltan checkpoints de corrida completada")\\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\\n        return res\\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\\n    if last.exists():\\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\\n        if guardado["fingerprint"] != huella:\\n            raise RuntimeError("Checkpoint incompatible")\\n        modelo.load_state_dict(guardado["model"])\\n        optimizador.load_state_dict(guardado["optimizer"])\\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\\n                                                              guardado["history"], guardado["best_file"])\\n        if not (dest / mejor_archivo).exists():\\n            raise RuntimeError("Falta mejor checkpoint referenciado")\\n        H.restore_rng(guardado["rng"])\\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\\n    for epoca in range(inicio, config["epochs"]):\\n        if sin_mejora >= config["patience"]:\\n            break\\n        modelo.train()\\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\\n            x, obj = train.lote(filas, device)\\n            optimizador.zero_grad(set_to_none=True)\\n            perdida = perdida_db(modelo(x), obj)\\n            if not torch.isfinite(perdida):\\n                raise RuntimeError("Perdida no finita")\\n            perdida.backward()\\n            optimizador.step()\\n            suma += perdida.item() * len(filas)\\n            vistas += len(filas)\\n        t_entreno = time.perf_counter() - t0\\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\\n        if not np.isfinite(puntaje):\\n            raise RuntimeError("Validacion no finita")\\n        if puntaje > mejor:\\n            mejor, sin_mejora = puntaje, 0\\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\\n                                                  "validation": detalle})\\n        else:\\n            sin_mejora += 1\\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\\n        H.atomic_json(dest / "history.json", historia)\\n        for viejo in dest.glob("best_epoch_*.pt"):\\n            if viejo.name != mejor_archivo:\\n                viejo.unlink()\\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\\n              f"({historia[-1][\\\'seconds\\\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\\n        if (parada is not None and epoca + 1 >= parada\\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\\n           "parameters": sum(p.numel() for p in modelo.parameters()),\\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\\n    H.atomic_json(dest / "result.json", res)\\n    return res\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser(description=__doc__.split("\\\\n")[0])\\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\\n    ap.add_argument("--output", required=True)\\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\\n    ap.add_argument("--allow-local-output", action="store_true")\\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\\n    ap.add_argument("--pause-after-epoch", type=int,\\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\\n    args = ap.parse_args(argv)\\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\\n        ap.error("--pause-after-epoch debe ser positivo")\\n\\n    e2_scripts = cargar_arnes(args.e2_scripts)\\n    output = Path(args.output).resolve()\\n    if not args.allow_local_output:\\n        import subprocess\\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\\n            raise RuntimeError("Salida debe estar en Drive")\\n    if args.cpu_smoke:\\n        if not args.max_images or args.max_images > 64:\\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\\n        device = torch.device("cpu")\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\\n        device = torch.device("cuda")\\n        if args.max_images or args.epochs:\\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\\n            or config["epochs"] < 1 or config["patience"] < 1\\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\\n        raise ValueError("Configuracion no soportada")\\n    config = copy.deepcopy(config)\\n    if args.epochs:\\n        config["epochs"] = args.epochs\\n    if args.max_images:\\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\\n\\n    H.verify_sources()\\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\\n    if args.max_images:\\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\\n    identity = {**firma_codigo(e2_scripts),\\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\\n    output.mkdir(parents=True, exist_ok=True)\\n    val = Datos(val_filas, args.voc, nombre="val")\\n    resultados = []\\n    for variante in args.variante:\\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\\n        preflight(variante, config, identity, train, device, output)\\n        for semilla in args.seeds or config["seeds"]:\\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\\n                                       parada=args.pause_after_epoch))\\n    print(json.dumps(resultados, indent=1))\\n    return resultados\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'training/entrenar_compactas_e3.py\': \'"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\\nimport argparse\\nimport copy\\nimport hashlib\\nimport json\\nimport sys\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', required=True)\\n    ap.add_argument(\\\'--output\\\', required=True)\\n    ap.add_argument(\\\'--smoke-voc\\\', help=\\\'Solo humo CPU: dos imágenes, una época, salida separada\\\')\\n    ap.add_argument(\\\'--smoke-masks\\\', help=\\\'Raíz de máscaras existentes, solo para humo local\\\')\\n    args = ap.parse_args()\\n    sys.path.insert(0, str(Path(args.e2) / \\\'scripts\\\'))\\n    import busqueda_arquitectura_e2 as B\\n    import torch\\n    e3 = Path(__file__).resolve().parents[1]\\n    train, val = B.C.partition(\\\'train\\\'), B.C.partition(\\\'val\\\')\\n    proto = json.loads((Path(args.e2) / \\\'config/experimento_e2.json\\\').read_text())\\n    if args.smoke_voc:\\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\\n        device = torch.device(\\\'cpu\\\')\\n    else:\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\\\'Se requiere GPU para la campaña\\\')\\n        device = torch.device(\\\'cuda\\\')\\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\\n    ident[\\\'extension_e3\\\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\\n    out = Path(args.output)\\n    out.mkdir(parents=True, exist_ok=True)\\n    for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n        template = json.loads((e3 / f\\\'config/plantilla_{family}_a0498.json\\\').read_text())[\\\'tarea\\\']\\n        assert template[\\\'arq\\\'] == \\\'a0498\\\' and template[\\\'w\\\'] == [64, 64, 64]\\n        assert template[\\\'b\\\'] == [0, 1, 0] and template[\\\'dropout_p\\\'] == .2\\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\\n            task = copy.deepcopy(template)\\n            task.update(semilla=seed, id=f\\\'{family}_a0498_s{seed}\\\')\\n            if args.smoke_voc:\\n                task.update(epocas=1, prueba_local=True)\\n            B.entrenar(task, out, proto, ident, device, train, val)\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    main()\\n\', \'tests/test_original_model.py\': \'"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\\n\\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\\n(ii)  la misma semilla da las mismas salidas;\\n(iii) formas de salida y entradas invalidas;\\n(iv)  gradiente no nulo y finito en todos los parametros;\\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\\n"""\\nimport io\\nimport json\\nimport sys\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pytest\\nimport torch\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\nfrom src import geometria_at as G  # noqa: E402\\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\\n\\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\\nTAM = 256\\n\\n\\n# ---------------------------------------------------------------------------------------- utilidades\\ndef _poligono(forma, puntos):\\n    m = np.zeros(forma, np.uint8)\\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return m\\n\\n\\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\\n\\n\\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\\n    w, c = alto * aspecto, forma[0] / 2 + dx\\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\\n\\n\\ndef _dice(a, b):\\n    a, b = a.astype(bool), b.astype(bool)\\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\\n\\n\\ndef _oraculo(variante, y):\\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\\n    v = np.ones(y.shape, bool)\\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\\n    return _dice(G.post_de(variante)(nucleo, v), y)\\n\\n\\ndef _lote_sintetico(n=4, tam=64):\\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\\n    rng = np.random.RandomState(7)\\n    xs, ys = [], []\\n    for i in range(n):\\n        y = np.zeros((tam, tam), np.uint8)\\n        c = tam / 2 + rng.uniform(-6, 6, 2)\\n        if i % 2 == 0:  # rombo de lado 18-24\\n            h = rng.uniform(18, 24) / np.sqrt(2)\\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\\n        else:  # rectangulo de 24 a 32 por 12 a 16\\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\\n        xs.append(x.astype(np.float32))\\n        ys.append(y)\\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\\n\\n\\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\\ndef test_i_sobreajusta_un_lote_diminuto():\\n    x, ys, obj = _lote_sintetico()\\n    modelo = UNetAT(semilla=0)\\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\\n    modelo.train()\\n    inicial = None\\n    for paso in range(250):\\n        opt.zero_grad()\\n        perdida = perdida_db(modelo(x), obj)\\n        perdida.backward()\\n        opt.step()\\n        inicial = float(perdida) if inicial is None else inicial\\n    final = float(perdida)\\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\\n    modelo.eval()\\n    mascaras = modelo.predict_mask(x, "dbnet_at")\\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\\n\\n\\n# ---------------------------------------------------------------------------------------- (ii) determinismo\\ndef test_ii_misma_semilla_mismas_salidas():\\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\\n    assert torch.equal(a(x), b(x))\\n    assert not torch.equal(a(x), c(x))\\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\\n    # crear con semilla no toca el generador global\\n    torch.manual_seed(123)\\n    antes = torch.rand(1)\\n    torch.manual_seed(123)\\n    UNetAT(semilla=99)\\n    assert torch.equal(antes, torch.rand(1))\\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\\n\\n\\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\\ndef test_iii_formas_y_entradas_invalidas():\\n    modelo = UNetAT(semilla=0)\\n    x = torch.zeros(2, 3, 64, 48)\\n    modelo.train()\\n    s = modelo(x)\\n    assert set(s) >= {"prob", "thresh", "binary"}\\n    for k in ("prob", "thresh", "binary", "logit_prob"):\\n        assert s[k].shape == (2, 1, 64, 48), k\\n    for k in ("prob", "thresh", "binary"):\\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\\n    modelo.eval()\\n    p = modelo(x)\\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\\n    assert torch.allclose(p, modelo.predict_proba(x))\\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 1, 64, 64))\\n    with pytest.raises(ValueError, match="3 canales"):\\n        modelo(torch.zeros(1, 4, 64, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo(torch.zeros(1, 3, 66, 64))\\n    with pytest.raises(ValueError, match="multiplos de 4"):\\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\\n    with pytest.raises(ValueError, match="B, 3, H, W"):\\n        modelo(torch.zeros(3, 64, 64))\\n    with pytest.raises(ValueError, match="tensor"):\\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\\n    with pytest.raises(ValueError, match="flotante"):\\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\\n    with pytest.raises(ValueError, match="variante"):\\n        modelo.predict_mask(x, "inexistente")\\n    with pytest.raises(ValueError):\\n        UNetAT(cabeza="5x5")\\n\\n\\n# ---------------------------------------------------------------------------------------- (iv) gradientes\\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\\n    x, _, obj = _lote_sintetico(n=2)\\n    for cabeza in ("1x1", "3x3"):\\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\\n        perdida = perdida_db(modelo(x), obj)\\n        assert torch.isfinite(perdida)\\n        perdida.backward()\\n        for nombre, p in modelo.named_parameters():\\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\\n    b = torch.sigmoid(50.0 * (p - t))\\n    b.sum().backward()\\n    esperado = 50.0 * b.detach() * (1 - b.detach())\\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\\n\\n\\n# ---------------------------------------------------------------------------------------- (v) la propiedad\\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\\n\\n\\ndef _medio(variante, formas):\\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rectangulos():\\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \\\'c2\\\'): la expansion consistente en inglete\\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\\n    for aspecto in (1, 2, 4):\\n        formas = [_rect(32, aspecto, d) for d in SUB]\\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\\n\\n\\ndef test_v_expansion_consistente_reconstruye_rombos():\\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\\n    for variante in dice:\\n        for lado in (10, 64):\\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\\n    assert np.mean(mejor) >= 0.96, mejor\\n\\n\\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\\n    if not JSON_GEOMETRIA.exists():\\n        pytest.skip("falta resultados/geometria_sintetica.json")\\n    previo = json.loads(JSON_GEOMETRIA.read_text())\\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\\n    for grupo, clave, formas in casos:\\n        for variante in ("dbnet", "c2", "dbnet_at"):\\n            esperado = previo[grupo][clave][variante]["dice"]\\n            obtenido = _medio(variante, formas)\\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\\n\\n\\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\\n    pyclipper = pytest.importorskip("pyclipper")\\n    salida = np.zeros(c.shape, np.uint8)\\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\\n    for k in G._contornos(c):\\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\\n    for q in off.Execute(d * escala):\\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\\n    return salida > 0\\n\\n\\ndef _iou(a, b):\\n    return (a & b).sum() / (a | b).sum()\\n\\n\\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\\n        m = np.zeros((96, 96), bool)\\n        m[20:20 + a, 10:10 + b] = True\\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\\n    m = np.zeros((96, 96), bool)\\n    m[30:50, 20:70] = True\\n    v = np.ones_like(m)\\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\\n\\n\\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\\n    forma = (160, 160)\\n    convexos = {\\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\\n    }\\n    for nombre, pts in convexos.items():\\n        c = _poligono(forma, pts) > 0\\n        for d in (2.0, 5.0, 9.0):\\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\\n            ref = _inglete_pyclipper(c, d)\\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\\n\\n\\ndef test_v_variante_dbnet_equivale_a_e2():\\n    """Con g0 = 0 y r\\\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\\n    if not (e2 / "etiquetas.py").exists():\\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\\n    sys.path.insert(0, str(e2))\\n    try:\\n        import etiquetas as E2\\n    finally:\\n        sys.path.remove(str(e2))\\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\\n    v = np.ones(y.shape, bool)\\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\\n    for k in e2o:\\n        assert np.allclose(propio[k], e2o[k]), k\\n    nucleo = propio["db_gt"][0]\\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\\n\\n\\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\\ndef test_vi_guardar_y_cargar_state_dict():\\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\\n    a = UNetAT(semilla=1).eval()\\n    buf = io.BytesIO()\\n    torch.save(a.state_dict(), buf)\\n    buf.seek(0)\\n    b = UNetAT(semilla=2).eval()\\n    assert not torch.equal(a(x), b(x))\\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\\n    assert torch.equal(a(x), b(x))\\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\\n\\n\\nclass _RedSinPosproceso(torch.nn.Module):\\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\\n\\n    def __init__(self, modelo):\\n        super().__init__()\\n        self.m = modelo\\n\\n    def forward(self, x):\\n        h = self.m._caracteristicas(x)\\n        p = torch.sigmoid(self.m.cabeza_prob(h))\\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\\n        return p, t, torch.sigmoid(self.m.k * (p - t))\\n\\n\\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\\n    ort = pytest.importorskip("onnxruntime")\\n    import tempfile\\n    modelo = UNetAT(semilla=4).eval()\\n    red = _RedSinPosproceso(modelo).eval()\\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\\n    with tempfile.TemporaryDirectory() as d:\\n        ruta = str(Path(d) / "unetat.onnx")\\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\\n                          opset_version=17, dynamo=False)\\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\\n        salidas = sesion.run(None, {"imagen": x.numpy()})\\n    with torch.no_grad():\\n        esperadas = red(x)\\n    for onnx_, torch_ in zip(salidas, esperadas):\\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\\n\', \'tests/conftest.py\': \'"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\\nimport sys\\nfrom pathlib import Path\\n\\nE3 = Path(__file__).resolve().parents[1]\\nif str(E3) not in sys.path:\\n    sys.path.insert(0, str(E3))\\n\', \'experiments/auditar_campana.py\': \'"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\\nOTHERS = [\\\'transunet_2d\\\', \\\'pidnet_s\\\', \\\'vmunet\\\', \\\'segnext_t\\\', \\\'deeplabv3plus\\\', \\\'unetplusplus\\\']\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for block in iter(lambda: f.read(1024 * 1024), b\\\'\\\'):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--benchmark\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--compactas\\\', type=Path, required=True)\\n    args = ap.parse_args()\\n    targets = []\\n    for s in SEEDS:\\n        for arm in [\\\'dbnet\\\', \\\'dbnet_at\\\']:\\n            targets.append((f\\\'unetat_{arm}\\\', s, args.campana / \\\'original\\\' / f\\\'unetat_{arm}_seed{s}\\\'))\\n        for family in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n            root = args.compactas if s in SEEDS[:5] else args.campana / \\\'compactas\\\'\\n            targets.append((family, s, root / \\\'fase2\\\' / f\\\'{family}_a0498_s{s}\\\'))\\n        for model in OTHERS + [\\\'segformer_b0\\\']:\\n            session = \\\'A\\\' if model in OTHERS[:2] else \\\'B\\\'\\n            if model == \\\'segformer_b0\\\': session = \\\'C\\\'\\n            root = args.benchmark if s in SEEDS[:5] and model != \\\'segformer_b0\\\' else args.campana / f\\\'rivales/sesion_{session}\\\'\\n            targets.append((model, s, root / f\\\'{model}_seed{s}\\\'))\\n    rows = []\\n    for model, seed, directory in targets:\\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\\\'falta_resultado\\\', dice_val=\\\'\\\')\\n        rp = directory / \\\'result.json\\\'\\n        if rp.exists():\\n            try:\\n                r = json.loads(rp.read_text())\\n                cfg = json.loads((directory / \\\'config.json\\\').read_text())\\n                assert cfg[\\\'fingerprint\\\'] == r[\\\'fingerprint\\\']\\n                assert r.get(\\\'completed\\\') is True and r.get(\\\'seed\\\', r.get(\\\'semilla\\\')) == seed\\n                assert r.get(\\\'model\\\', r.get(\\\'familia\\\')) == model\\n                if model in [\\\'fcn\\\', \\\'unet\\\', \\\'segnet\\\']:\\n                    assert r[\\\'arq\\\'] == \\\'a0498\\\'\\n                    assert cfg[\\\'tarea\\\'][\\\'w\\\'] == [64,64,64] and cfg[\\\'tarea\\\'][\\\'b\\\'] == [0,1,0]\\n                if model.startswith(\\\'unetat_\\\'):\\n                    assert cfg[\\\'identity\\\'][\\\'opencv\\\'] == \\\'4.11.0\\\'\\n                if model == \\\'segformer_b0\\\':\\n                    recipe = cfg[\\\'config\\\'][\\\'models\\\'][model]\\n                    assert recipe[\\\'optimizer\\\']==\\\'AdamW\\\' and abs(recipe[\\\'lr\\\']-.0005573638936676344)<1e-14\\n                    assert recipe[\\\'weight_decay\\\']==.0001\\n                name = r[\\\'best_file\\\']\\n                assert Path(name).name == name\\n                assert sha(directory / name) == r[\\\'best_sha256\\\']\\n                row.update(estado=\\\'verificado\\\', dice_val=r[\\\'best_val_group_Dice\\\'])\\n            except (KeyError, ValueError, AssertionError, OSError) as e:\\n                row[\\\'estado\\\'] = \\\'invalido:\\\' + type(e).__name__\\n        rows.append(row)\\n    args.campana.mkdir(parents=True, exist_ok=True)\\n    with (args.campana / \\\'inventario_campana.csv\\\').open(\\\'w\\\', newline=\\\'\\\', encoding=\\\'utf-8\\\') as f:\\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        writer.writeheader(); writer.writerows(rows)\\n    report = dict(esperadas=len(rows), verificadas=sum(r[\\\'estado\\\']==\\\'verificado\\\' for r in rows),\\n                  test_evaluado=False, pendientes=[r for r in rows if r[\\\'estado\\\']!=\\\'verificado\\\'])\\n    report[\\\'lista_para_revision_pretest\\\'] = report[\\\'verificadas\\\'] == report[\\\'esperadas\\\']\\n    (args.campana / \\\'estado_campana.json\\\').write_text(json.dumps(report, indent=2), encoding=\\\'utf-8\\\')\\n    print(json.dumps(report, indent=2))\\n    return 0 if report[\\\'lista_para_revision_pretest\\\'] else 2\\n\\n\\nif __name__ == \\\'__main__\\\':\\n    raise SystemExit(main())\\n\', \'training/compactas_e2.py\': \'"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\\n"""\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\ndef block(a, b):\\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\\n\\ndef _drop(activo, p):\\n    return nn.Dropout(p) if activo else nn.Identity()\\n\\nclass FCNBusqueda(nn.Module):\\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.head = nn.Conv2d(w3, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\\\'bilinear\\\', align_corners=False)\\n\\nclass UNetBusqueda(nn.Module):\\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\\\'nearest\\\'), m], 1)))\\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\\\'nearest\\\'), a], 1)))\\n        return self.head(d)\\n\\nclass SegNetBusqueda(nn.Module):\\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\\n\\n    def __init__(self, w, b, p=0.2):\\n        super().__init__()\\n        w1, w2, w3 = w\\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\\n        self.reduce = nn.Conv2d(w3, w2, 1)\\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\\n        self.head = nn.Conv2d(w1, 1, 1)\\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\\n\\n    def forward(self, x):\\n        a = self.r1(self.e1(x))\\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\\n        m = self.r2(self.e2(p))\\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\\n        c = self.reduce(self.r3(self.e3(p)))\\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\\n        return self.head(d)\\n\\nCLASES = {\\\'fcn\\\': FCNBusqueda, \\\'unet\\\': UNetBusqueda, \\\'segnet\\\': SegNetBusqueda}\\n\\ndef construir_red(familia, w, b, p):\\n    return CLASES[familia](tuple(w), tuple(b), p)\\n\', \'experiments/evaluar_campana.py\': \'"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\\nimport argparse\\nimport csv\\nimport hashlib\\nimport json\\nimport sys\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\n\\ndef sha(p):\\n    h = hashlib.sha256()\\n    with p.open(\\\'rb\\\') as f:\\n        for b in iter(lambda: f.read(1024*1024), b\\\'\\\'): h.update(b)\\n    return h.hexdigest()\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\'--e2\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--e3\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--voc\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--campana\\\', type=Path, required=True)\\n    ap.add_argument(\\\'--smoke\\\', action=\\\'store_true\\\', help=\\\'Solo prueba técnica, dos imágenes, salida separada\\\')\\n    args = ap.parse_args()\\n    sys.path[:0] = [str(args.e2/\\\'scripts\\\'), str(args.e3), str(args.e3/\\\'training\\\')]\\n    import numpy as np\\n    import torch\\n    import cv2\\n    from dataset_ready import load_example, partition, metrics\\n    from modelos_e2 import MODELS_E2, main_logits\\n    from compactas_e2 import construir_red\\n    from src.original_model import UNetAT\\n    assert cv2.__version__==\\\'4.11.0\\\'\\n    state = json.loads((args.campana/\\\'estado_campana.json\\\').read_text())\\n    if args.smoke:\\n        assert state.get(\\\'smoke\\\') is True\\n    else:\\n        assert state[\\\'lista_para_revision_pretest\\\'], \\\'Completar y auditar primero las 120 corridas\\\'\\n    with (args.campana/\\\'inventario_campana.csv\\\').open(encoding=\\\'utf-8\\\') as f:\\n        runs=list(csv.DictReader(f))\\n    assert (args.smoke or len(runs)==120) and all(r[\\\'estado\\\']==\\\'verificado\\\' for r in runs)\\n    device=torch.device(\\\'cuda\\\' if torch.cuda.is_available() else \\\'cpu\\\')\\n    torch.set_num_threads(2)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    rows=partition(\\\'val\\\')\\n    if args.smoke: rows=rows[:2]\\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\\n    out=args.campana/(\\\'evaluacion_validacion_humo\\\' if args.smoke else \\\'evaluacion_validacion\\\'); out.mkdir(exist_ok=True)\\n    summaries=[]\\n    for run in runs:\\n        folder=Path(run[\\\'directorio\\\']); name=run[\\\'modelo\\\']; seed=int(run[\\\'semilla\\\'])\\n        result=json.loads((folder/\\\'result.json\\\').read_text())\\n        cfg=json.loads((folder/\\\'config.json\\\').read_text())\\n        checkpoint=folder/result[\\\'best_file\\\']; assert sha(checkpoint)==result[\\\'best_sha256\\\']\\n        destination=out/f\\\'{name}_s{seed}.csv\\\'\\n        stamp=destination.with_suffix(\\\'.json\\\')\\n        identity={\\\'checkpoint_sha256\\\':result[\\\'best_sha256\\\'],\\\'script\\\':sha(Path(__file__)),\\n                  \\\'data\\\':sha(args.e2/\\\'resultados/preentrenamiento/val.csv\\\'),\\n                  \\\'normalizacion\\\':sha(args.e2/\\\'resultados/preentrenamiento/normalization_train.json\\\'),\\n                  \\\'opencv\\\':cv2.__version__,\\\'torch\\\':torch.__version__, \\\'smoke\\\':args.smoke,\\n                  \\\'dispositivo\\\':torch.cuda.get_device_name(0) if device.type==\\\'cuda\\\' else \\\'cpu\\\'}\\n        if stamp.exists() and destination.exists():\\n            previous=json.loads(stamp.read_text())\\n            assert previous[\\\'identity\\\']==identity and previous[\\\'csv_sha256\\\']==sha(destination)\\n            summaries.append(previous); continue\\n        torch.manual_seed(20261118)\\n        if name.startswith(\\\'unetat_\\\'):\\n            model=UNetAT()\\n        elif name in [\\\'fcn\\\',\\\'unet\\\',\\\'segnet\\\']:\\n            t=cfg[\\\'tarea\\\']; model=construir_red(name,t[\\\'w\\\'],t[\\\'b\\\'],t[\\\'dropout_p\\\'])\\n        else:\\n            model=MODELS_E2[name]()\\n        model.load_state_dict(torch.load(checkpoint,map_location=\\\'cpu\\\',weights_only=True)[\\\'model\\\'],strict=True)\\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\\\'cuda\\\' else []):\\n            torch.manual_seed(20261118)\\n            for row,x,y,v in data:\\n                tensor=torch.from_numpy(x[None]).to(device)\\n                if name.startswith(\\\'unetat_\\\'):\\n                    prob=model.predict_mask(tensor,name.removeprefix(\\\'unetat_\\\'),valida=v[None])[0].astype(np.float32)\\n                else:\\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\\n                scores=metrics(prob,y,v)\\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\\\'id\\\'],group_id=row[\\\'group_id\\\'],**scores))\\n                groups[row[\\\'group_id\\\']].append(scores[\\\'Dice\\\'])\\n        with destination.open(\\\'w\\\',newline=\\\'\\\',encoding=\\\'utf-8\\\') as f:\\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\\n        record={\\\'modelo\\\':name,\\\'semilla\\\':seed,\\\'identity\\\':identity,\\\'csv_sha256\\\':sha(destination),\\n                \\\'imagenes\\\':len(predictions),\\\'grupos\\\':len(groups),\\\'dice_val\\\':value,\\n                \\\'dice_guardado\\\':result[\\\'best_val_group_Dice\\\'],\\n                \\\'diferencia_dice\\\':value-result[\\\'best_val_group_Dice\\\'],\\\'test_evaluado\\\':False}\\n        stamp.write_text(json.dumps(record,indent=2),encoding=\\\'utf-8\\\'); summaries.append(record)\\n        print(name,seed,\\\'Dice\\\',value,\\\'delta\\\',record[\\\'diferencia_dice\\\'],flush=True)\\n        del model\\n        if device.type==\\\'cuda\\\':torch.cuda.empty_cache()\\n    (out/\\\'resumen.json\\\').write_text(json.dumps(summaries,indent=2),encoding=\\\'utf-8\\\')\\n\\n\\nif __name__==\\\'__main__\\\':main()\\n\', \'config/hpo_original_t0.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t0",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t1.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t1",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.00010193470686927321,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t2.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t2",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.009289398227793906,\\n      "weight_decay": 0.0001\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/hpo_original_t3.json\': \'{\\n  "schema": 2,\\n  "experiment": "e3_hpo_original_t3",\\n  "pretrained": false,\\n  "size": 256,\\n  "batch_size": 2,\\n  "epochs": 10,\\n  "patience": 3,\\n  "seeds": [\\n    11\\n  ],\\n  "threshold": 0.5,\\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\\n  "pidnet_aux_weight": 0.4,\\n  "pidnet_boundary_weight": 1.0,\\n  "models": {\\n    "fcn": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "segnet": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "unet": {\\n      "optimizer": "Adam",\\n      "lr": 0.0011899757998003779,\\n      "weight_decay": 0.0\\n    },\\n    "deeplabv3plus": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0005,\\n      "weight_decay": 0.01\\n    },\\n    "unetplusplus": {\\n      "optimizer": "Adam",\\n      "lr": 0.0005,\\n      "weight_decay": 0.0\\n    },\\n    "pidnet_s": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    },\\n    "segnext_t": {\\n      "optimizer": "AdamW",\\n      "lr": 0.0001,\\n      "weight_decay": 0.01\\n    },\\n    "segformer_b0": {\\n      "optimizer": "AdamW",\\n      "lr": 6e-05,\\n      "weight_decay": 0.01\\n    },\\n    "transunet_2d": {\\n      "optimizer": "SGD",\\n      "lr": 0.001,\\n      "momentum": 0.9,\\n      "weight_decay": 0.0001\\n    },\\n    "vmunet": {\\n      "optimizer": "AdamW",\\n      "lr": 0.001,\\n      "weight_decay": 0.01\\n    }\\n  },\\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\\n}\', \'config/plantilla_fcn_a0498.json\': \'{\\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\\n  "tarea": {\\n    "id": "fcn_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "fcn",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.001,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 4039376896,\\n    "macs_efectivos": 4039376896,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_unet_a0498.json\': \'{\\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\\n  "tarea": {\\n    "id": "unet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "unet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 13103005696,\\n    "macs_efectivos": 13103005696,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\', \'config/plantilla_segnet_a0498.json\': \'{\\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\\n  "tarea": {\\n    "id": "segnet_a0498_s11",\\n    "bloque": "fase2",\\n    "familia": "segnet",\\n    "arq": "a0498",\\n    "w": [\\n      64,\\n      64,\\n      64\\n    ],\\n    "b": [\\n      0,\\n      1,\\n      0\\n    ],\\n    "dropout_p": 0.2,\\n    "semilla": 11,\\n    "optim": {\\n      "optimizer": "Adam",\\n      "lr": 0.0003,\\n      "weight_decay": 0.0\\n    },\\n    "epocas": 30,\\n    "paciencia": 5,\\n    "conservar": "best",\\n    "macs": 10099884032,\\n    "macs_efectivos": 10099884032,\\n    "px": 256,\\n    "fraccion_train": 1.0,\\n    "semilla_subconjunto": 20261123\\n  },\\n  "protocolo": {\\n    "batch_size": 2,\\n    "size": 256,\\n    "threshold": 0.5\\n  },\\n  "identidad": {\\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\\n  },\\n  "entorno": {\\n    "packages": {\\n      "torch": "2.5.1+cu121",\\n      "torchvision": "0.20.1+cu121",\\n      "numpy": "1.26.4",\\n      "Pillow": "11.1.0",\\n      "segmentation-models-pytorch": "0.5.0",\\n      "timm": "1.0.15",\\n      "scipy": "1.15.2",\\n      "ml-collections": "1.0.0",\\n      "einops": "0.8.1",\\n      "mamba-ssm": null,\\n      "causal-conv1d": null,\\n      "triton": "3.1.0",\\n      "transformers": null\\n    },\\n    "python": "3.11.11",\\n    "cuda": "12.1",\\n    "device": "NVIDIA L4"\\n  },\\n  "filas": {\\n    "train": 1015,\\n    "val": 218\\n  }\\n}\'}\n', "FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}\n", 'HISTORICO = \'{\\n "rombos_por_lado_px": {\\n  "6": {\\n   "area_px": 41,\\n   "dbnet": {\\n    "dice": 0.9526,\\n    "nucleo_px": 15\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.5348,\\n    "nucleo_px": 44\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8332,\\n    "nucleo_px": 15\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 15\\n   },\\n   "at_redondo": {\\n    "dice": 0.931,\\n    "nucleo_px": 44\\n   },\\n   "dbnet_at": {\\n    "dice": 0.6421,\\n    "nucleo_px": 44\\n   }\\n  },\\n  "8": {\\n   "area_px": 85,\\n   "dbnet": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.6863,\\n    "nucleo_px": 54\\n   },\\n   "c2_redondo": {\\n    "dice": 0.8683,\\n    "nucleo_px": 35\\n   },\\n   "c2": {\\n    "dice": 0.947,\\n    "nucleo_px": 35\\n   },\\n   "at_redondo": {\\n    "dice": 0.9536,\\n    "nucleo_px": 54\\n   },\\n   "dbnet_at": {\\n    "dice": 0.947,\\n    "nucleo_px": 54\\n   }\\n  },\\n  "10": {\\n   "area_px": 113,\\n   "dbnet": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7801,\\n    "nucleo_px": 62\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 42\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 42\\n   },\\n   "at_redondo": {\\n    "dice": 0.9823,\\n    "nucleo_px": 62\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 62\\n   }\\n  },\\n  "12": {\\n   "area_px": 145,\\n   "dbnet": {\\n    "dice": 0.9027,\\n    "nucleo_px": 70\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.7478,\\n    "nucleo_px": 95\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9253,\\n    "nucleo_px": 70\\n   },\\n   "c2": {\\n    "dice": 0.9352,\\n    "nucleo_px": 70\\n   },\\n   "at_redondo": {\\n    "dice": 0.9196,\\n    "nucleo_px": 95\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9352,\\n    "nucleo_px": 95\\n   }\\n  },\\n  "16": {\\n   "area_px": 265,\\n   "dbnet": {\\n    "dice": 0.9352,\\n    "nucleo_px": 103\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8737,\\n    "nucleo_px": 122\\n   },\\n   "c2_redondo": {\\n    "dice": 0.911,\\n    "nucleo_px": 103\\n   },\\n   "c2": {\\n    "dice": 0.9689,\\n    "nucleo_px": 103\\n   },\\n   "at_redondo": {\\n    "dice": 0.9839,\\n    "nucleo_px": 122\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 122\\n   }\\n  },\\n  "24": {\\n   "area_px": 613,\\n   "dbnet": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9039,\\n    "nucleo_px": 262\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9509,\\n    "nucleo_px": 218\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 218\\n   },\\n   "at_redondo": {\\n    "dice": 0.9546,\\n    "nucleo_px": 262\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9515,\\n    "nucleo_px": 262\\n   }\\n  },\\n  "32": {\\n   "area_px": 1105,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9483,\\n    "nucleo_px": 402\\n   },\\n   "c2_redondo": {\\n    "dice": 0.969,\\n    "nucleo_px": 380\\n   },\\n   "c2": {\\n    "dice": 0.9842,\\n    "nucleo_px": 380\\n   },\\n   "at_redondo": {\\n    "dice": 0.9747,\\n    "nucleo_px": 402\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 402\\n   }\\n  },\\n  "48": {\\n   "area_px": 2381,\\n   "dbnet": {\\n    "dice": 0.969,\\n    "nucleo_px": 835\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9687,\\n    "nucleo_px": 835\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9689,\\n    "nucleo_px": 835\\n   },\\n   "c2": {\\n    "dice": 0.975,\\n    "nucleo_px": 835\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 835\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 835\\n   }\\n  },\\n  "64": {\\n   "area_px": 4141,\\n   "dbnet": {\\n    "dice": 0.9716,\\n    "nucleo_px": 1436\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9718,\\n    "nucleo_px": 1436\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1436\\n   },\\n   "at_redondo": {\\n    "dice": 0.9714,\\n    "nucleo_px": 1438\\n   },\\n   "dbnet_at": {\\n    "dice": 1.0,\\n    "nucleo_px": 1438\\n   }\\n  },\\n  "96": {\\n   "area_px": 9385,\\n   "dbnet": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9684,\\n    "nucleo_px": 3270\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9607,\\n    "nucleo_px": 3270\\n   },\\n   "c2": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   },\\n   "at_redondo": {\\n    "dice": 0.9683,\\n    "nucleo_px": 3270\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9873,\\n    "nucleo_px": 3270\\n   }\\n  }\\n },\\n "rectangulos_alto32_por_aspecto": {\\n  "1": {\\n   "dbnet": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9079,\\n    "nucleo_px": 441\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9775,\\n    "nucleo_px": 361\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 361\\n   },\\n   "at_redondo": {\\n    "dice": 0.9506,\\n    "nucleo_px": 441\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9412,\\n    "nucleo_px": 441\\n   }\\n  },\\n  "2": {\\n   "dbnet": {\\n    "dice": 0.9407,\\n    "nucleo_px": 705\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9588,\\n    "nucleo_px": 705\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 705\\n   },\\n   "at_redondo": {\\n    "dice": 0.9593,\\n    "nucleo_px": 833\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9555,\\n    "nucleo_px": 833\\n   }\\n  },\\n  "4": {\\n   "dbnet": {\\n    "dice": 0.8231,\\n    "nucleo_px": 1177\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.9132,\\n    "nucleo_px": 1417\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9852,\\n    "nucleo_px": 1177\\n   },\\n   "c2": {\\n    "dice": 1.0,\\n    "nucleo_px": 1177\\n   },\\n   "at_redondo": {\\n    "dice": 0.991,\\n    "nucleo_px": 1417\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9629,\\n    "nucleo_px": 1417\\n   }\\n  },\\n  "8": {\\n   "dbnet": {\\n    "dice": 0.7714,\\n    "nucleo_px": 2245\\n   },\\n   "c1_rfijo": {\\n    "dice": 0.8576,\\n    "nucleo_px": 2750\\n   },\\n   "c2_redondo": {\\n    "dice": 0.9663,\\n    "nucleo_px": 2245\\n   },\\n   "c2": {\\n    "dice": 0.9696,\\n    "nucleo_px": 2245\\n   },\\n   "at_redondo": {\\n    "dice": 0.9701,\\n    "nucleo_px": 2750\\n   },\\n   "dbnet_at": {\\n    "dice": 0.9419,\\n    "nucleo_px": 2750\\n   }\\n  }\\n }\\n}\'\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", '\n', "E3 = Path('/content/secuencia_e3_v4/e3')\n", "sello = OUTPUT / 'firmas_secuencia.json'\n", 'if sello.exists() and json.loads(sello.read_text()) != FIRMAS:\n', "    raise RuntimeError('La carpeta contiene otra versión; no mezclar')\n", 'for rel, texto in CONTENIDO.items():\n', '    p = E3 / rel\n', '    p.parent.mkdir(parents=True, exist_ok=True)\n', "    p.write_text(texto, encoding='utf-8')\n", '    assert sha_file(p) == FIRMAS[rel]\n', 'sello.write_text(json.dumps(FIRMAS, indent=2))\n', "ref = E3.parent / 'resultados/geometria_sintetica.json'\n", 'ref.parent.mkdir(exist_ok=True)\n', 'ref.write_text(HISTORICO)\n', "(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(\n", "    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))\n", 'def verificar_original(semillas):\n', "    for arm in ['dbnet', 'dbnet_at']:\n", '        for seed in semillas:\n', "            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'\n", "            r = json.loads((folder / 'result.json').read_text())\n", "            assert r['completed'] and r['seed']==seed and r['variante']==arm\n", "            assert sha_file(folder / r['best_file']) == r['best_sha256']\n", 'def entrenar(semillas):\n', "    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),\n", "        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),\n", "        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ['from google.colab import drive, files\n', "drive.mount('/content/drive')\n", 'from pathlib import Path\n', 'import json, hashlib, zipfile, subprocess, sys\n', "CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')\n", "BENCHMARK = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')\n", "COMPACTAS = Path('/content/compactas_e3_recibidas')\n", "ZIP = Path('/content/drive/MyDrive/compactas_a0498_e3.zip')\n", "assert BENCHMARK.is_dir(), 'Falta la carpeta histórica que confirmaste en Drive'\n", "assert ZIP.is_file(), 'Sube compactas_a0498_e3.zip a MyDrive'\n", 'COMPACTAS.mkdir(exist_ok=True)\n', 'with zipfile.ZipFile(ZIP) as z:\n', '    for name in z.namelist():\n', '        assert (COMPACTAS/name).resolve().is_relative_to(COMPACTAS)\n', '    z.extractall(COMPACTAS)\n', "ESPERADOS = {'fcn_a0498_s11': 'ef655608d580e34a72966f6f352e643544c3eb8c215dace5d246da98f72814c7', 'fcn_a0498_s23': 'c9f9856e9239d9c59366871525b870ac421ef6ae370bb60b29bff6960aeb2772', 'fcn_a0498_s37': 'f5c326a78e22f4d8dca84cdfd0b86a9cd9646e947ab22c59465fd82d4af00df4', 'fcn_a0498_s53': '6f220654d54344f5b6545087d3c102c0415be2d97eed8721b62ebb36789a86fa', 'fcn_a0498_s71': '9e2b5d911b7722573885eeb157cf75f43b5302d11eabcb1ad287f6c16ff1164e', 'segnet_a0498_s11': 'c2bc57be34caa5501118601b9d050be50062f5a657309850e47e2a96b5ac2d14', 'segnet_a0498_s23': 'a5d63031449a429343961f982ef26fdcf6817c139fb1c7142a3b0fd4199d2dac', 'segnet_a0498_s37': '7c358e59dd5ef539375b43d78a30b5f46de262418afac998a956bea0bc75e750', 'segnet_a0498_s53': '3c41962731ca6d1680a879f6d25608abcda7790c6e24e91ed68835463bc1376c', 'segnet_a0498_s71': 'c18f29d092e404dc42210bf284fb4b998c02eebda495322ac42e9eb2c8518ecd', 'unet_a0498_s11': '4138b06d20bd32a422a69970252187322a35fea09a4e304bcda61d904ac672a9', 'unet_a0498_s23': 'e0452e0bf9df896ffaafbbc1838c02abc070d1b37478121fc25246a33aec75aa', 'unet_a0498_s37': '59c6df9ad6c05500114d0e5e4948e8fd4cc5bd831eda4c577aab6066be25ddeb', 'unet_a0498_s53': '615d2b1a786ec37372e8ca2868783ec99ce135c556b63610b9a7bfc1e4a10f02', 'unet_a0498_s71': 'a0d80416558ccdfd1632ad34ef1bd132ac896f2c83ab7b8b7c2f5d0adb03df84'}\n", 'for run, expected in ESPERADOS.items():\n', "    assert hashlib.sha256((COMPACTAS/'fase2'/run/'best.pt').read_bytes()).hexdigest()==expected\n"], 'execution_count': None, 'outputs': []}, {'cell_type': 'code', 'metadata': {}, 'source': ["subprocess.run([PY,str(E3/'experiments/auditar_campana.py'),\n", "    '--campana',str(CAMPANA),'--benchmark',str(BENCHMARK),'--compactas',str(COMPACTAS)],check=True)\n", "subprocess.run([PY,str(E3/'experiments/evaluar_campana.py'),'--e2',str(PROJECT),'--e3',str(E3),\n", "    '--voc',str(BASE),'--campana',str(CAMPANA)],check=True)\n", "archive=Path('/content/evaluacion_validacion_e3_v4.zip')\n", "with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:\n", "    for p in (CAMPANA/'evaluacion_validacion').glob('*'):\n", '        if p.is_file():z.write(p,p.name)\n', 'files.download(str(archive))\n', "print('Devuelve este ZIP para contraste con E2 y revisión estadística antes de test.')\n"], 'execution_count': None, 'outputs': []}], 'metadata': {'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'}, 'language_info': {'name': 'python'}, 'colab': {'name': 'Benchmark_E2_Colab_v2.ipynb'}}, 'nbformat': 4, 'nbformat_minor': 5}}
if __name__ == '__main__':
    run(NOTEBOOKS, os.environ.get('E3_CARRIL', 'todo'))
